In [49]:
print("ram ram")

ram ram


In [28]:
import json
import re
from collections import defaultdict

def parse_sql_to_compressed_string(sql_file_path, output_json_path="schema.json"):
    """
    Parses SQL schema and compresses table schemas into single-line ultra-compact string signatures.
    """
    with open(sql_file_path, "r", encoding="utf-8", errors="ignore") as f:
        sql_content = f.read()

    table_pattern = re.compile(
        r"CREATE\s+TABLE\s+\[dbo\]\.\[(?P<table_name>\w+)\]\s*\((?P<columns_block>.*?)\)\s*ON",
        re.DOTALL | re.IGNORECASE
    )

    column_pattern = re.compile(
        r"^\s*\[(?P<col_name>\w+)\]\s+(?:AS\s+.*|\[(?P<data_type>\w+)\](?:\((?P<type_arg>[\d,\s]+|max)\))?\s*(?P<identity>IDENTITY\(\d+,\d+\))?\s*(?P<nullability>NOT\s+NULL|NULL)?)",
        re.MULTILINE | re.IGNORECASE
    )

    pk_pattern = re.compile(
        r"PRIMARY\s+KEY.*?\((?P<pk_cols>.*?)\)",
        re.DOTALL | re.IGNORECASE
    )

    schema_json = {}

    for table_match in table_pattern.finditer(sql_content):
        table_name = table_match.group("table_name")
        columns_block = table_match.group("columns_block")

        pk_cols = set()
        pk_match = pk_pattern.search(columns_block)
        if pk_match:
            pk_cols = set(re.findall(r"\[(\w+)\]", pk_match.group("pk_cols")))

        columns = []

        for line in columns_block.splitlines():
            line_str = line.strip()
            
            # Computed Columns
            computed_match = re.search(r"\[(\w+)\]\s+AS\s+\((.*?)\)", line_str, re.IGNORECASE)
            if computed_match:
                columns.append((computed_match.group(1), "COMPUTED", ""))
                continue

            # Standard Columns
            col_match = column_pattern.match(line_str)
            if col_match:
                col_name = col_match.group("col_name")
                data_type = col_match.group("data_type").lower()
                type_arg = col_match.group("type_arg")
                identity = bool(col_match.group("identity"))
                nullability = col_match.group("nullability") or ""

                flags = []
                if col_name in pk_cols:
                    flags.append("PK")
                if identity:
                    flags.append("ID")
                if "NOT NULL" in nullability.upper():
                    flags.append("NN")

                flag_str = f"({','.join(flags)})" if flags else ""
                
                # Normalize types
                if data_type in ["nvarchar", "varchar", "char"]:
                    dtype = f"str{type_arg}" if type_arg else "str"
                elif data_type in ["decimal", "numeric"]:
                    dtype = f"dec({type_arg})" if type_arg else "dec"
                else:
                    dtype = data_type

                columns.append((col_name, dtype, flag_str))

        # Group columns dynamically into the single compressed line format
        schema_json[table_name] = compress_columns(columns)

    with open(output_json_path, "w", encoding="utf-8") as out_file:
        json.dump(schema_json, out_file, indent=2)

    print(f"Successfully generated compressed schema JSON at '{output_json_path}'.")


def compress_columns(columns):
    """
    Helper function to aggregate column definitions into a single line format.
    """
    specials = []
    groups = defaultdict(list)
    str50_cols = []

    # Sequence collapse for machine numbers (Machine1Name, Machine2Name -> Machine1-3Name)
    machine_cols = defaultdict(list)

    for col_name, dtype, flags in columns:
        if flags or dtype == "COMPUTED":
            specials.append(f"{col_name}:{dtype}{flags}")
        elif dtype == "str50":
            str50_cols.append(col_name)
        elif re.match(r"^Machine[1-9]", col_name):
            base_key = re.sub(r"\d+", "", col_name)
            machine_cols[(base_key, dtype)].append(col_name)
        else:
            groups[dtype].append(col_name)

    parts = list(specials)

    # Process grouped non-special columns
    for dtype, cols in groups.items():
        # Wildcard grouping for Date, Qty, Bags
        dates = [c for c in cols if c.endswith("Date") or c.endswith("On")]
        qtys = [c for c in cols if "Qty" in c or "Quantity" in c or "KGIn" in c]
        bags = [c for c in cols if "Bag" in c or "Bags" in c]
        
        rest = [c for c in cols if c not in dates and c not in qtys and c not in bags]

        if dates:
            parts.append(f"*Date/*On:{dtype}")
        if qtys:
            parts.append(f"*Qty/*Quantity:{dtype}")
        if bags:
            parts.append(f"*Bags/*Bag:{dtype}")
        
        if rest:
            parts.append(f"{'/'.join(rest)}:{dtype}")

    # Process machine series
    for (base_key, dtype), cols in machine_cols.items():
        num_start = re.search(r"\d+", cols[0]).group()
        num_end = re.search(r"\d+", cols[-1]).group()
        series_name = f"Machine{num_start}-{num_end}{base_key.replace('Machine', '')}"
        parts.append(f"{series_name}:{dtype}")

    # Add 50-length strings in array format
    if str50_cols:
        parts.append(f"[{', '.join(str50_cols)}]:str50")

    return ", ".join(parts)


# Execute
parse_sql_to_compressed_string(
    r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.sql", 
    r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.json"
)

Successfully generated compressed schema JSON at 'C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.json'.


In [29]:

import json
import re
from pathlib import Path


# ============================================================
# 1. CONFIGURATION
# ============================================================

sql_file = Path(
    r"C:\Users\soroj\OneDrive\Desktop"
    r"\IIIOT Activate Projects"
    r"\MANUFACTURING-AGENTIC-AI"
    r"\docs\video_analytics_db_info"
    r"\construction_ai_schema.sql"
)

output_file = Path(
    r"C:\Users\soroj\OneDrive\Desktop"
    r"\IIIOT Activate Projects"
    r"\MANUFACTURING-AGENTIC-AI"
    r"\docs\video_analytics_db_info"
    r"\construction_ai_schema.json"
)


# ============================================================
# 2. TYPE CONVERSION
# ============================================================

def normalize_type(type_str: str) -> str:
    """
    Convert PostgreSQL types into compact schema types.

    Examples:
        integer              -> int
        character varying(100) -> str100
        text                 -> text
        boolean              -> bool
        timestamp without time zone -> timestamp
    """

    t = type_str.strip().lower()

    # Remove PostgreSQL schema qualification
    t = t.replace("pg_catalog.", "")

    # Remove array notation (preserve as array type)
    is_array = t.endswith("[]")
    if is_array:
        t = t[:-2].strip()

    # Remove double quotes around type names
    t = t.replace('"', '')

    # Integer types
    if t in ("smallint", "integer", "int", "int4", "bigint", "int8"):
        result = "int"

    # Boolean
    elif t in ("boolean", "bool"):
        result = "bool"

    # Character varying
    elif t.startswith("character varying"):
        match = re.search(r"\((\d+)\)", t)
        result = f"str{match.group(1)}" if match else "str"

    # Character
    elif t.startswith("character"):
        match = re.search(r"\((\d+)\)", t)
        result = f"str{match.group(1)}" if match else "str"

    # VARCHAR shorthand
    elif t.startswith("varchar"):
        match = re.search(r"\((\d+)\)", t)
        result = f"str{match.group(1)}" if match else "str"

    # Text
    elif t == "text":
        result = "text"

    # JSON
    elif t in ("json", "jsonb"):
        result = t

    # Numeric
    elif t.startswith(("numeric", "decimal")):
        result = "decimal"

    # Floating point
    elif t in ("real", "float4", "double precision", "float8"):
        result = "float"

    # Timestamp
    elif t.startswith("timestamp"):
        result = "timestamp"

    # Date / time
    elif t == "date":
        result = "date"

    elif t.startswith("time"):
        result = "time"

    # UUID
    elif t == "uuid":
        result = "uuid"

    # Bytea
    elif t == "bytea":
        result = "bytes"

    # Default: preserve the PostgreSQL type
    else:
        result = t

    return f"{result}[]" if is_array else result


# ============================================================
# 3. SPLIT SQL COLUMNS SAFELY
# ============================================================

def split_sql_items(body: str):
    """
    Split SQL definitions on commas, ignoring commas inside
    parentheses or quoted strings.
    """

    items = []
    current = []
    depth = 0
    quote = None

    i = 0

    while i < len(body):
        char = body[i]

        # Handle SQL single-quoted strings
        if quote == "'":
            current.append(char)

            if char == "'":
                # SQL escaped quote: ''
                if i + 1 < len(body) and body[i + 1] == "'":
                    current.append(body[i + 1])
                    i += 1
                else:
                    quote = None

            i += 1
            continue

        if char == "'":
            quote = "'"
            current.append(char)

        elif char == "(":
            depth += 1
            current.append(char)

        elif char == ")":
            depth -= 1
            current.append(char)

        elif char == "," and depth == 0:
            item = "".join(current).strip()

            if item:
                items.append(item)

            current = []

        else:
            current.append(char)

        i += 1

    final_item = "".join(current).strip()

    if final_item:
        items.append(final_item)

    return items


# ============================================================
# 4. EXTRACT CREATE TABLE BLOCKS
# ============================================================

def extract_create_tables(sql: str):
    """
    Extract CREATE TABLE definitions from PostgreSQL SQL dump.
    """

    pattern = re.compile(
        r"CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?"
        r"(?:(?:[\w]+\.)?)(?P<table>[^\s(]+)\s*\(",
        re.IGNORECASE
    )

    tables = []

    for match in pattern.finditer(sql):

        table_name = match.group("table").strip('"')
        start = match.end()

        depth = 1
        quote = None
        i = start

        while i < len(sql) and depth > 0:

            char = sql[i]

            if quote == "'":

                if char == "'":
                    if i + 1 < len(sql) and sql[i + 1] == "'":
                        i += 1
                    else:
                        quote = None

            elif char == "'":
                quote = "'"

            elif char == "(":
                depth += 1

            elif char == ")":
                depth -= 1

            i += 1

        body = sql[start:i - 1]

        tables.append({
            "name": table_name,
            "body": body
        })

    return tables


# ============================================================
# 5. PARSE COLUMN DEFINITIONS
# ============================================================

def parse_columns(body: str):

    columns = []
    table_primary_keys = set()

    items = split_sql_items(body)

    for item in items:

        item_clean = item.strip()

        # Ignore table-level constraints
        if re.match(
            r"^(CONSTRAINT|PRIMARY\s+KEY|UNIQUE|CHECK|FOREIGN\s+KEY)",
            item_clean,
            re.IGNORECASE
        ):
            # Extract primary keys defined at table level
            pk_match = re.search(
                r"PRIMARY\s+KEY\s*\(([^)]+)\)",
                item_clean,
                re.IGNORECASE
            )

            if pk_match:
                pk_columns = pk_match.group(1).split(",")

                for pk_col in pk_columns:
                    table_primary_keys.add(
                        pk_col.strip().strip('"')
                    )

            continue

        # Extract column name and SQL type
        column_match = re.match(
            r'^"?(?P<name>[\w]+)"?\s+'
            r'(?P<type>[a-zA-Z_][\w\s]*(?:\([^)]*\))?(?:\[\])?)'
            r'(?P<rest>.*)$',
            item_clean,
            re.IGNORECASE | re.DOTALL
        )

        if not column_match:
            continue

        name = column_match.group("name")
        raw_type = column_match.group("type").strip()
        rest = column_match.group("rest").strip()

        # Fix types that may have been captured with extra keywords
        type_match = re.match(
            r'^(smallint|integer|int|int4|bigint|int8|'
            r'boolean|bool|text|jsonb?|'
            r'character\s+varying(?:\(\d+\))?|'
            r'character(?:\(\d+\))?|'
            r'varchar(?:\(\d+\))?|'
            r'numeric(?:\([^)]*\))?|'
            r'decimal(?:\([^)]*\))?|'
            r'real|float4|double\s+precision|float8|'
            r'timestamp(?:\s+without\s+time\s+zone|'
            r'\s+with\s+time\s+zone)?|'
            r'date|time(?:\s+without\s+time\s+zone|'
            r'\s+with\s+time\s+zone)?|'
            r'uuid|bytea)'
            r'(\[\])?',
            raw_type,
            re.IGNORECASE
        )

        if type_match:
            raw_type = type_match.group(0)

        normalized_type = normalize_type(raw_type)

        # Constraints
        attributes = []

        if re.search(r"\bNOT\s+NULL\b", rest, re.IGNORECASE):
            attributes.append("NN")

        if re.search(r"\bPRIMARY\s+KEY\b", rest, re.IGNORECASE):
            attributes.append("PK")

        # Identity / serial / auto-increment
        if re.search(
            r"\bGENERATED\s+(?:ALWAYS|BY\s+DEFAULT)\s+AS\s+IDENTITY\b",
            rest,
            re.IGNORECASE
        ):
            attributes.append("ID")

        elif re.search(
            r"\bDEFAULT\s+nextval\s*\(",
            rest,
            re.IGNORECASE
        ):
            attributes.append("ID")

        elif raw_type.lower() in (
            "serial",
            "bigserial",
            "smallserial"
        ):
            attributes.append("ID")

        # Avoid duplicate attribute markers
        attributes = list(dict.fromkeys(attributes))

        column_data = {
            "name": name,
            "type": normalized_type,
            "attributes": attributes
        }

        columns.append(column_data)

    # Apply table-level primary keys
    for column in columns:

        if column["name"] in table_primary_keys:

            if "PK" not in column["attributes"]:
                column["attributes"].append("PK")

    return columns


# ============================================================
# 6. FORMAT TABLE INTO REQUIRED STRING
# ============================================================

def format_table(columns):

    formatted_columns = []

    for column in columns:

        name = column["name"]
        column_type = column["type"]
        attributes = column["attributes"]

        if attributes:
            formatted = (
                f"{name}:{column_type}"
                f"({','.join(attributes)})"
            )
        else:
            formatted = f"{name}:{column_type}"

        formatted_columns.append(formatted)

    return ", ".join(formatted_columns)


# ============================================================
# 7. MAIN CONVERSION FUNCTION
# ============================================================

def convert_sql_to_json():

    if not sql_file.exists():
        raise FileNotFoundError(
            f"SQL file not found: {sql_file}"
        )

    sql = sql_file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    tables = extract_create_tables(sql)

    schema = {}

    for table in tables:

        table_name = table["name"]

        columns = parse_columns(table["body"])

        schema[table_name] = format_table(columns)

    output_file.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    output_file.write_text(
        json.dumps(
            schema,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print("=" * 60)
    print("SQL TO JSON CONVERSION COMPLETED")
    print("=" * 60)
    print(f"Tables extracted: {len(schema)}")
    print(f"Output file: {output_file}")
    print("=" * 60)


# ============================================================
# 8. RUN
# ============================================================

if __name__ == "__main__":
    convert_sql_to_json()

SQL TO JSON CONVERSION COMPLETED
Tables extracted: 52
Output file: C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\video_analytics_db_info\construction_ai_schema.json


In [30]:
import json
import os
import re
from typing import Any, Dict, List, Set

# -------------------------------------------------------------------
# 1. Schema Paths Configuration
# -------------------------------------------------------------------
SCHEMA_PATHS = {
    "video_analytics": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\video_analytics_db_info\construction_ai_schema.json",
    "mes": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.json",
}

# -------------------------------------------------------------------
# 2. JEV Registry with Fine-Grained Table & Column Mapping
# -------------------------------------------------------------------
def parse_schema_file(file_path: str) -> Dict[str, List[str]]:
    """
    Parses table definitions and extracts pure column names from shorthand notation.
    Example: 'alerts': 'id:i(PK,NN), track_id:i(NN), class_name:c(NN)...' -> ['id', 'track_id', 'class_name', ...]
    """
    table_map = {}
    if not os.path.exists(file_path):
        return table_map

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for table_name, schema_str in data.items():
        columns = []
        if isinstance(schema_str, str):
            # Split comma-separated column representations
            raw_cols = schema_str.split(",")
            for col_def in raw_cols:
                # Clean up definition (e.g., 'class_name:c(NN)' -> 'class_name')
                col_name = col_def.strip().split(":")[0].strip()
                # Handle grouped columns like 'camera_id/zone_id/assignment_id'
                if "/" in col_name:
                    for sub_col in col_name.split("/"):
                        clean_col = sub_col.replace("*", "").strip()
                        if clean_col:
                            columns.append(clean_col)
                else:
                    clean_col = col_name.replace("*", "").strip()
                    if clean_col:
                        columns.append(clean_col)
        elif isinstance(schema_str, list):
            columns = schema_str

        table_map[table_name] = columns

    return table_map


def build_jev_detailed_registry(paths_dict: Dict[str, str]) -> Dict[str, Dict[str, List[str]]]:
    """Builds a structured dictionary of DB -> Table -> [Columns]."""
    detailed_registry = {}
    for db_name, path in paths_dict.items():
        detailed_registry[db_name] = parse_schema_file(path)
    return detailed_registry


JEV_DETAILED_REGISTRY = build_jev_detailed_registry(SCHEMA_PATHS)


# -------------------------------------------------------------------
# 3. Pure JEV Table & Column Matching Engine
# -------------------------------------------------------------------
def identify_tables_and_columns(user_query: str) -> Dict[str, Any]:
    """
    Scans the user query and identifies target databases, matched tables, 
    and specific responsible columns for data extraction.
    """
    tokens = set(re.findall(r"\w+", user_query.lower()))

    results = {
        "user_query": user_query,
        "selected_databases": [],
        "target_mappings": []
    }

    for db_name, tables in JEV_DETAILED_REGISTRY.items():
        db_has_match = False

        for table_name, columns in tables.items():
            matched_cols = []
            
            # Check table name match
            table_match = table_name.lower() in tokens or any(t in table_name.lower() for t in tokens if len(t) > 3)
            
            # Check column matches
            for col in columns:
                if col.lower() in tokens:
                    matched_cols.append(col)

            # If table name or columns match the query tokens
            if table_match or matched_cols:
                db_has_match = True
                
                # Responsible primary columns
                responsible_columns = list(set(matched_cols)) if matched_cols else columns[:5]
                
                results["target_mappings"].append({
                    "database": db_name,
                    "responsible_table": table_name,
                    "matched_query_columns": responsible_columns,
                    "all_table_columns": columns
                })

        if db_has_match and db_name not in results["selected_databases"]:
            results["selected_databases"].append(db_name)

    return results


# -------------------------------------------------------------------
# 4. Execution Example
# -------------------------------------------------------------------
if __name__ == "__main__":
    test_query = "live camera feed of the construction site and check for any safety violations or alerts in worker behavior."

    print(f"User Query: {test_query}\n")
    output = identify_tables_and_columns(test_query)

    print("--- DB SELECTION & RESPONSIBLE TABLES/COLUMNS ---")
    print(json.dumps(output, indent=2))

User Query: live camera feed of the construction site and check for any safety violations or alerts in worker behavior.

--- DB SELECTION & RESPONSIBLE TABLES/COLUMNS ---
{
  "user_query": "live camera feed of the construction site and check for any safety violations or alerts in worker behavior.",
  "selected_databases": [
    "video_analytics"
  ],
  "target_mappings": [
    {
      "database": "video_analytics",
      "responsible_table": "alerts",
      "matched_query_columns": [
        "id",
        "camera_id",
        "camera_name",
        "zone_id",
        "assignment_id"
      ],
      "all_table_columns": [
        "id",
        "camera_id",
        "camera_name",
        "zone_id",
        "assignment_id",
        "camera_rule_id",
        "track_id",
        "class_name",
        "confidence",
        "snapshot_path",
        "is_acknowledged",
        "created_at"
      ]
    },
    {
      "database": "video_analytics",
      "responsible_table": "camera_status_logs",


In [31]:
import json
import os
import re
from typing import Any, Dict, List, Set

# -------------------------------------------------------------------
# 1. Schema Paths Configuration
# -------------------------------------------------------------------
SCHEMA_PATHS = {
    "video_analytics": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\video_analytics_db_info\construction_ai_schema.json",
    "mes": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.json",
}

# -------------------------------------------------------------------
# 2. Vocabulary Maps for Precise Sub-Intent Keyword Match
# -------------------------------------------------------------------
DOMAIN_VOCABULARY = {
    "video_analytics": {
        "camera": ["cameras", "camera_status_logs", "basler_devices"],
        "configuration": ["cameras", "hse_camera_rules", "basler_model_assignments", "ai_models"],
        "config": ["cameras", "hse_camera_rules", "basler_model_assignments"],
        "feed": ["cameras"],
        "stream": ["cameras"],
        "violation": ["alerts", "anomaly_flags", "hse_camera_rules"],
        "safety": ["alerts", "anomaly_flags"],
        "worker": ["attendances", "access_rules"],
        "alert": ["alerts", "anomaly_flags"],
    },
    "mes": {
        "machine": ["machine_master", "machine_telemetry", "machine_status_logs"],
        "utilization": ["machine_utilization_logs", "production_line_efficiency", "machine_master"],
        "work_order": ["work_orders", "job_cards"],
        "production": ["production_reports", "work_orders"],
        "shift": ["shift_logs", "operator_schedules"],
    }
}

# -------------------------------------------------------------------
# 3. Clean Schema Parser (Noise Reduction for Token Savings)
# -------------------------------------------------------------------
def parse_and_clean_schema(file_path: str) -> Dict[str, List[str]]:
    """
    Parses table definitions and cleans up parsing noise like 'NN)', 'PK', 'i(PK'
    to minimize prompt token size.
    """
    table_map = {}
    if not os.path.exists(file_path):
        return table_map

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for table_name, schema_str in data.items():
        clean_cols = []
        if isinstance(schema_str, str):
            raw_cols = schema_str.split(",")
            for col_def in raw_cols:
                # Extract clean column name
                col_name = col_def.strip().split(":")[0].strip()
                col_name = re.sub(r'[^a-zA-Z0-9_/]', '', col_name)  # Strips noise like NN), PK, etc.
                
                if "/" in col_name:
                    for sub_col in col_name.split("/"):
                        if sub_col and sub_col not in ["NN", "PK"]:
                            clean_cols.append(sub_col)
                else:
                    if col_name and col_name not in ["NN", "PK"]:
                        clean_cols.append(col_name)
        elif isinstance(schema_str, list):
            clean_cols = schema_str

        # Remove duplicate clean column entries
        table_map[table_name] = list(dict.fromkeys(clean_cols))

    return table_map


def build_clean_registry(paths_dict: Dict[str, str]) -> Dict[str, Dict[str, List[str]]]:
    return {db_name: parse_and_clean_schema(path) for db_name, path in paths_dict.items()}

JEV_CLEAN_REGISTRY = build_clean_registry(SCHEMA_PATHS)


# -------------------------------------------------------------------
# 4. Query Customizer & Multi-DB Splitter Engine
# -------------------------------------------------------------------
def customize_and_split_query(user_query: str) -> Dict[str, Any]:
    """
    Splits user query into per-DB customized queries and isolates minimum required table/column contexts.
    """
    tokens = set(re.findall(r"\w+", user_query.lower()))

    pipeline_output = {
        "raw_user_query": user_query,
        "selected_databases": [],
        "customized_db_requests": []
    }

    for db_name, tables in JEV_CLEAN_REGISTRY.items():
        db_vocab = DOMAIN_VOCABULARY.get(db_name, {})
        matched_tokens = tokens.intersection(set(db_vocab.keys()))

        if matched_tokens:
            pipeline_output["selected_databases"].append(db_name)
            
            # Map specific tables responsible for matched domain tokens
            suggested_tables = set()
            for token in matched_tokens:
                suggested_tables.update(db_vocab[token])

            db_target_mappings = []
            for table_name in suggested_tables:
                if table_name in tables:
                    cols = tables[table_name]
                    # Filter query-relevant columns
                    matched_cols = [c for c in cols if c.lower() in tokens or any(t in c.lower() for t in matched_tokens)]
                    
                    db_target_mappings.append({
                        "responsible_table": table_name,
                        "primary_columns": matched_cols if matched_cols else cols[:5],
                        "all_table_columns": cols
                    })

            # Create customized query sub-intent
            if db_name == "video_analytics":
                custom_intent = f"Fetch camera configuration and status details matching keywords: {', '.join(matched_tokens)}"
            elif db_name == "mes":
                custom_intent = f"Fetch machine utilization metrics and operational logs matching keywords: {', '.join(matched_tokens)}"
            else:
                custom_intent = f"Fetch data for {', '.join(matched_tokens)}"

            pipeline_output["customized_db_requests"].append({
                "database": db_name,
                "customized_sub_query": custom_intent,
                "token_keywords": list(matched_tokens),
                "target_mappings": db_target_mappings
            })

    return pipeline_output


# -------------------------------------------------------------------
# 5. Execution Test
# -------------------------------------------------------------------
if __name__ == "__main__":
    # Test Query: Multi-DB requirement (MES Machine Utilization + Video Analytics Camera Config)
    multi_db_query = "I want the data from the mes application for machine utilization and from video analytics I want camera configuration information."

    print(f"User Query: {multi_db_query}\n")
    processed_output = customize_and_split_query(multi_db_query)

    print("--- CUSTOMIZED MULTI-DB SPLIT & SCHEMA CONTEXT ---")
    print(json.dumps(processed_output, indent=2))

User Query: I want the data from the mes application for machine utilization and from video analytics I want camera configuration information.

--- CUSTOMIZED MULTI-DB SPLIT & SCHEMA CONTEXT ---
{
  "raw_user_query": "I want the data from the mes application for machine utilization and from video analytics I want camera configuration information.",
  "selected_databases": [
    "video_analytics",
    "mes"
  ],
  "customized_db_requests": [
    {
      "database": "video_analytics",
      "customized_sub_query": "Fetch camera configuration and status details matching keywords: camera, configuration",
      "token_keywords": [
        "camera",
        "configuration"
      ],
      "target_mappings": [
        {
          "responsible_table": "basler_model_assignments",
          "primary_columns": [
            "basler_camera_id"
          ],
          "all_table_columns": [
            "id",
            "basler_camera_id",
            "model_id",
            "confidence_threshold",
 

In [32]:
import json
import os
import re
from typing import Any, Dict, List, Set, Tuple
from collections import defaultdict

# -------------------------------------------------------------------
# 1. Schema Paths (update these to your actual paths)
# -------------------------------------------------------------------
SCHEMA_PATHS = {
    "video_analytics": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\video_analytics_db_info\construction_ai_schema.json",
    "mes": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.json",
}

# -------------------------------------------------------------------
# 2. Improved Domain Vocabulary (based on real tables you shared)
# -------------------------------------------------------------------
DOMAIN_VOCABULARY = {
    "video_analytics": {
        # Camera related
        "camera": ["cameras", "camera_status_logs", "basler_devices", "basler_model_assignments", "hse_camera_rules"],
        "cameras": ["cameras", "camera_status_logs", "basler_devices"],
        "feed": ["cameras"],
        "stream": ["cameras"],
        "rtsp": ["cameras"],
        "basler": ["basler_devices", "basler_model_assignments", "defect_detections"],

        # Configuration
        "config": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments", "ai_models"],
        "configuration": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments", "ai_models"],
        "rule": ["hse_camera_rules", "hse_rule_definitions", "detection_assignments"],
        "assignment": ["detection_assignments", "basler_model_assignments"],

        # Safety / Alerts
        "alert": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "violation": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "safety": ["alerts", "anomaly_flags", "hse_rule_events", "incidents", "hse_camera_rules"],
        "anomaly": ["anomaly_flags"],
        "incident": ["incidents"],

        # Workers / Attendance
        "worker": ["attendances", "employees", "employee_movements", "access_rules"],
        "employee": ["employees", "attendances", "employee_movements"],
        "attendance": ["attendances"],
        "face": ["cameras", "employees", "attendances"],

        # Models / AI
        "model": ["ai_models", "ai_model_classes", "basler_model_assignments", "detection_assignments"],
        "ai": ["ai_models", "ai_model_classes"],

        # Zones
        "zone": ["zones", "zone_risk_scores", "detection_assignments"],
    },

    "mes": {
        # Machine related
        "machine": ["Machine", "MachineMaster", "MachineFGMapping", "MachineMaterialMapping", "CapacityAnalysis", "MaintenanceWindow"],
        "machines": ["Machine", "MachineMaster"],
        "utilization": ["CapacityAnalysis", "MachineMaster", "WorkOrder", "GanttSchedule"],
        "util": ["CapacityAnalysis", "MachineMaster"],
        "capacity": ["CapacityAnalysis", "MachineMaster", "ContainerCapacity"],
        "oee": ["CapacityAnalysis", "MachineMaster"],

        # Work Order / Production
        "workorder": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog", "WorkOrderResource", "WorkOrderChangeLog"],
        "work_order": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog"],
        "wo": ["WorkOrder", "WorkOrderStep"],
        "production": ["WorkOrder", "ProductionPlanning", "FinishedGood", "SemiFinishedGood", "WIPStorage"],
        "schedule": ["GanttSchedule", "WorkOrder", "ShiftCalendar", "MPSHeader", "MpsMaster"],
        "gantt": ["GanttSchedule"],

        # Shift / Operator
        "shift": ["ShiftMaster", "ShiftCalendar", "OperatorMaster", "CapacityAnalysis"],
        "operator": ["OperatorMaster", "WorkOrderResource", "WorkOrderStep"],

        # Inventory / Material
        "inventory": ["Inventory", "InventoryByLot", "Materials", "RawMaterial", "FinishedGood", "SemiFinishedGood"],
        "material": ["Materials", "RawMaterial", "BOMLine", "BOMMaster", "WIPRMRequest"],
        "bom": ["BOMMaster", "BOMLine", "WorkOrderBOM"],
        "stock": ["Inventory", "InventoryByLot", "FinishedGood", "SemiFinishedGood"],

        # Planning
        "mps": ["MPSHeader", "MpsMaster", "MPSRawMaterialRequirement"],
        "mrp": ["MRP_Run", "MRP_Demand", "MRP_Result"],
        "planning": ["MPSHeader", "MpsMaster", "ProductionPlanning", "WeeklyPlanning"],

        # Maintenance
        "maintenance": ["MaintenanceWindow", "Machine", "MachineMaster"],
        "downtime": ["MaintenanceWindow", "CapacityAnalysis"],
    }
}

# -------------------------------------------------------------------
# 3. Robust Schema Cleaner
# -------------------------------------------------------------------
def clean_column_name(raw: str) -> List[str]:
    """Extract clean column name(s) from noisy schema strings."""
    # Remove type info and constraints
    name = raw.strip().split(":")[0].strip()
    
    # Handle patterns like *Date/*On/*_at or employee_ids/allowed_plant_ids
    if "/" in name:
        parts = re.split(r"[/]", name)
        cleaned = []
        for p in parts:
            p = re.sub(r"[^a-zA-Z0-9_]", "", p)
            if p and p.lower() not in {"nn", "pk", "id", "date", "on", "at"}:
                cleaned.append(p)
        return cleaned if cleaned else [re.sub(r"[^a-zA-Z0-9_]", "", name)]
    
    # Normal case
    name = re.sub(r"[^a-zA-Z0-9_]", "", name)
    if name and name.lower() not in {"nn", "pk"}:
        return [name]
    return []


def parse_and_clean_schema(file_path: str) -> Dict[str, List[str]]:
    table_map = {}
    if not os.path.exists(file_path):
        print(f"[WARN] Schema file not found: {file_path}")
        return table_map

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for table_name, schema_str in data.items():
        clean_cols = []
        if isinstance(schema_str, str):
            for col_def in schema_str.split(","):
                clean_cols.extend(clean_column_name(col_def))
        elif isinstance(schema_str, list):
            clean_cols = [str(c) for c in schema_str]

        # Deduplicate while preserving order
        table_map[table_name] = list(dict.fromkeys(c for c in clean_cols if c))

    return table_map


def build_clean_registry(paths_dict: Dict[str, str]) -> Dict[str, Dict[str, List[str]]]:
    registry = {}
    for db_name, path in paths_dict.items():
        registry[db_name] = parse_and_clean_schema(path)
        print(f"[INFO] Loaded {db_name}: {len(registry[db_name])} tables")
    return registry


# Build once
JEV_CLEAN_REGISTRY = build_clean_registry(SCHEMA_PATHS)


# -------------------------------------------------------------------
# 4. Smarter Query Splitter
# -------------------------------------------------------------------
def normalize_token(token: str) -> str:
    token = token.lower().strip()
    # Simple stemming / aliases
    aliases = {
        "cameras": "camera",
        "machines": "machine",
        "utilizations": "utilization",
        "configs": "config",
        "configurations": "configuration",
        "workorders": "workorder",
        "work-order": "workorder",
        "work_orders": "workorder",
    }
    return aliases.get(token, token)


def get_relevant_columns(cols: List[str], tokens: Set[str], matched_domain_tokens: Set[str]) -> List[str]:
    """Rank columns by relevance."""
    scored = []
    for col in cols:
        col_l = col.lower()
        score = 0
        if col_l in tokens:
            score += 10
        for t in matched_domain_tokens:
            if t in col_l or col_l in t:
                score += 5
        # Prefer useful columns
        if any(x in col_l for x in ["id", "name", "code", "status", "date", "qty", "quantity", "count"]):
            score += 1
        scored.append((score, col))
    
    scored.sort(key=lambda x: (-x[0], x[1]))
    # Return top relevant ones (or first 8 if none scored high)
    relevant = [c for s, c in scored if s > 0]
    return relevant[:8] if relevant else cols[:8]


def customize_and_split_query(user_query: str) -> Dict[str, Any]:
    tokens = set(re.findall(r"[a-zA-Z0-9_]+", user_query.lower()))
    normalized_tokens = {normalize_token(t) for t in tokens}

    pipeline_output = {
        "raw_user_query": user_query,
        "selected_databases": [],
        "customized_db_requests": []
    }

    for db_name, tables in JEV_CLEAN_REGISTRY.items():
        db_vocab = DOMAIN_VOCABULARY.get(db_name, {})
        
        # 1. Exact / normalized vocab hits
        matched_tokens = set()
        for t in normalized_tokens:
            if t in db_vocab:
                matched_tokens.add(t)
            # Also check partial matches inside vocab keys
            for vocab_key in db_vocab:
                if t in vocab_key or vocab_key in t:
                    matched_tokens.add(vocab_key)

        # 2. Fallback: match against actual table names
        if not matched_tokens:
            for table_name in tables:
                table_l = table_name.lower()
                for t in normalized_tokens:
                    if t in table_l or table_l in t:
                        matched_tokens.add(t)

        if not matched_tokens:
            continue  # this DB is not relevant

        pipeline_output["selected_databases"].append(db_name)

        # Collect candidate tables
        suggested_tables: Set[str] = set()
        for token in matched_tokens:
            if token in db_vocab:
                suggested_tables.update(db_vocab[token])

        # Also add any table whose name contains a matched token
        for table_name in tables:
            table_l = table_name.lower()
            for t in matched_tokens:
                if t in table_l:
                    suggested_tables.add(table_name)

        # Build target mappings
        db_target_mappings = []
        for table_name in sorted(suggested_tables):
            if table_name not in tables:
                # Try case-insensitive lookup (MES has PascalCase)
                found = None
                for real_name in tables:
                    if real_name.lower() == table_name.lower():
                        found = real_name
                        break
                if not found:
                    continue
                table_name = found

            cols = tables[table_name]
            primary_cols = get_relevant_columns(cols, tokens, matched_tokens)

            db_target_mappings.append({
                "responsible_table": table_name,
                "primary_columns": primary_cols,
                "all_table_columns": cols
            })

        # Generate a clean customized sub-query
        keywords_str = ", ".join(sorted(matched_tokens))
        if db_name == "video_analytics":
            custom_intent = f"Retrieve camera configuration, status, rules and related video-analytics data for: {keywords_str}"
        elif db_name == "mes":
            custom_intent = f"Retrieve machine utilization, capacity, work-order and production data for: {keywords_str}"
        else:
            custom_intent = f"Retrieve data related to: {keywords_str}"

        pipeline_output["customized_db_requests"].append({
            "database": db_name,
            "customized_sub_query": custom_intent,
            "token_keywords": sorted(list(matched_tokens)),
            "target_mappings": db_target_mappings
        })

    return pipeline_output


# -------------------------------------------------------------------
# 5. Test Suite – helps you find and fix issues quickly
# -------------------------------------------------------------------
def run_tests():
    print("=" * 70)
    print("RUNNING DIAGNOSTIC TESTS")
    print("=" * 70)

    test_cases = [
        {
            "name": "Original multi-DB query",
            "query": "I want the data from the mes application for machine utilization and from video analytics I want camera configuration information."
        },
        {
            "name": "Only MES machine utilization",
            "query": "Show me machine utilization and capacity for last week"
        },
        {
            "name": "Only Video camera config",
            "query": "Give me camera configuration and status of all cameras"
        },
        {
            "name": "Work order + alerts",
            "query": "I need work order status and any safety alerts or violations"
        },
        {
            "name": "Shift + operator + machine",
            "query": "What is the shift wise machine utilization and operator assignment?"
        },
        {
            "name": "Inventory + production",
            "query": "Show finished goods inventory and production planning data"
        },
        {
            "name": "HSE rules + zones",
            "query": "List all active HSE camera rules and zone risk scores"
        },
        {
            "name": "No matching domain (should return empty)",
            "query": "What is the weather today?"
        },
    ]

    for i, tc in enumerate(test_cases, 1):
        print(f"\n--- Test {i}: {tc['name']} ---")
        print(f"Query: {tc['query']}")
        result = customize_and_split_query(tc["query"])

        print(f"Selected DBs : {result['selected_databases']}")
        for req in result["customized_db_requests"]:
            print(f"  → {req['database'].upper()}")
            print(f"     Sub-query : {req['customized_sub_query']}")
            print(f"     Keywords  : {req['token_keywords']}")
            print(f"     Tables    : {[m['responsible_table'] for m in req['target_mappings']]}")
            if not req["target_mappings"]:
                print("     ⚠ WARNING: No target tables found!")
            else:
                for m in req["target_mappings"][:3]:  # show first 3
                    print(f"        • {m['responsible_table']} → {m['primary_columns'][:5]}")

        # Quick health checks
        if "mes" in result["selected_databases"]:
            mes_req = next(r for r in result["customized_db_requests"] if r["database"] == "mes")
            if not mes_req["target_mappings"]:
                print("     ❌ MES has empty target_mappings – check vocabulary or table name casing")

        if "video_analytics" in result["selected_databases"]:
            va_req = next(r for r in result["customized_db_requests"] if r["database"] == "video_analytics")
            if not va_req["target_mappings"]:
                print("     ❌ Video Analytics has empty target_mappings")

    print("\n" + "=" * 70)
    print("SCHEMA HEALTH CHECK")
    print("=" * 70)
    for db, tables in JEV_CLEAN_REGISTRY.items():
        print(f"\n{db.upper()} – {len(tables)} tables")
        # Show a few important ones
        important = []
        if db == "mes":
            important = ["Machine", "MachineMaster", "CapacityAnalysis", "WorkOrder", "ShiftMaster"]
        else:
            important = ["cameras", "alerts", "ai_models", "hse_camera_rules", "zones"]

        for t in important:
            found = None
            for real in tables:
                if real.lower() == t.lower():
                    found = real
                    break
            if found:
                print(f"  ✓ {found}  ({len(tables[found])} columns)")
            else:
                print(f"  ✗ {t}  NOT FOUND")


# -------------------------------------------------------------------
# 6. Main
# -------------------------------------------------------------------
if __name__ == "__main__":
    # First run the diagnostic tests
    run_tests()

    print("\n\n" + "=" * 70)
    print("EXAMPLE: Original query result (pretty printed)")
    print("=" * 70)

    multi_db_query = "I want the data from the mes application for machine utilization and from video analytics I want camera configuration information."
    result = customize_and_split_query(multi_db_query)
    print(json.dumps(result, indent=2))

[INFO] Loaded video_analytics: 52 tables
[INFO] Loaded mes: 108 tables
RUNNING DIAGNOSTIC TESTS

--- Test 1: Original multi-DB query ---
Query: I want the data from the mes application for machine utilization and from video analytics I want camera configuration information.
Selected DBs : ['video_analytics', 'mes']
  → VIDEO_ANALYTICS
     Sub-query : Retrieve camera configuration, status, rules and related video-analytics data for: ai, assignment, camera, cameras, config, configuration, incident, violation
     Keywords  : ['ai', 'assignment', 'camera', 'cameras', 'config', 'configuration', 'incident', 'violation']
     Tables    : ['ai_model_classes', 'ai_models', 'alerts', 'anomaly_flags', 'basler_devices', 'basler_model_assignments', 'camera_status_logs', 'cameras', 'counting_configs', 'detection_assignments', 'hse_camera_rules', 'hse_rule_events', 'incidents']
        • ai_model_classes → ['id', 'class_name', 'model_id']
        • ai_models → ['id', 'config_path', 'group_id', 'nam

In [33]:
import json
import os
import re
from typing import Any, Dict, List, Set, Tuple
from collections import defaultdict

# -------------------------------------------------------------------
# 1. Schema Paths
# -------------------------------------------------------------------
SCHEMA_PATHS = {
    "video_analytics": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\video_analytics_db_info\construction_ai_schema.json",
    "mes": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.json",
}

# -------------------------------------------------------------------
# 2. Stop words – never treat these as domain signals
# -------------------------------------------------------------------
STOP_WORDS = {
    "a", "an", "the", "and", "or", "of", "in", "on", "for", "to", "from", "with",
    "is", "are", "was", "were", "be", "been", "being", "have", "has", "had",
    "do", "does", "did", "will", "would", "can", "could", "should", "may", "might",
    "i", "me", "my", "we", "you", "your", "he", "she", "it", "they", "them",
    "this", "that", "these", "those", "what", "which", "who", "whom", "whose",
    "all", "any", "some", "no", "not", "only", "just", "also", "very", "too",
    "want", "need", "show", "give", "get", "list", "data", "information", "details",
    "application", "system", "last", "week", "today", "yesterday", "status", "active"
}

# -------------------------------------------------------------------
# 3. Domain Vocabulary (tight + high-signal)
# -------------------------------------------------------------------
DOMAIN_VOCABULARY = {
    "video_analytics": {
        "camera": ["cameras", "camera_status_logs", "basler_devices", "basler_model_assignments", "hse_camera_rules"],
        "cameras": ["cameras", "camera_status_logs", "basler_devices"],
        "config": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments"],
        "configuration": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments"],
        "rule": ["hse_camera_rules", "hse_rule_definitions", "hse_rule_events"],
        "hse": ["hse_camera_rules", "hse_rule_definitions", "hse_rule_events"],
        "alert": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "violation": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "safety": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "anomaly": ["anomaly_flags"],
        "incident": ["incidents"],
        "zone": ["zones", "zone_risk_scores"],
        "worker": ["attendances", "employees", "employee_movements"],
        "employee": ["employees", "attendances", "employee_movements"],
        "attendance": ["attendances"],
        "model": ["ai_models", "ai_model_classes", "basler_model_assignments"],
        "basler": ["basler_devices", "basler_model_assignments"],
    },
    "mes": {
        "machine": ["Machine", "MachineMaster", "MachineFGMapping", "MachineMaterialMapping", "CapacityAnalysis"],
        "machines": ["Machine", "MachineMaster"],
        "utilization": ["CapacityAnalysis", "MachineMaster"],
        "util": ["CapacityAnalysis", "MachineMaster"],
        "capacity": ["CapacityAnalysis", "MachineMaster", "ContainerCapacity"],
        "workorder": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog", "WorkOrderResource"],
        "work_order": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog"],
        "wo": ["WorkOrder", "WorkOrderStep"],
        "production": ["WorkOrder", "ProductionPlanning", "FinishedGood", "SemiFinishedGood"],
        "shift": ["ShiftMaster", "ShiftCalendar", "OperatorMaster"],
        "operator": ["OperatorMaster", "WorkOrderResource"],
        "inventory": ["Inventory", "InventoryByLot", "FinishedGood", "SemiFinishedGood"],
        "material": ["Materials", "RawMaterial", "BOMLine", "BOMMaster"],
        "bom": ["BOMMaster", "BOMLine"],
        "mps": ["MPSHeader", "MpsMaster"],
        "mrp": ["MRP_Run", "MRP_Demand", "MRP_Result"],
        "planning": ["MPSHeader", "MpsMaster", "ProductionPlanning", "WeeklyPlanning"],
        "maintenance": ["MaintenanceWindow"],
        "schedule": ["GanttSchedule", "WorkOrder"],
    }
}

# -------------------------------------------------------------------
# 4. Robust Column Cleaner
# -------------------------------------------------------------------
def clean_column_name(raw: str) -> List[str]:
    name = raw.strip().split(":")[0].strip()

    # Handle *Date/*On/*_at style and slash-separated fields
    if "/" in name:
        parts = re.split(r"[/]", name)
        cleaned = []
        for p in parts:
            p = re.sub(r"[^a-zA-Z0-9_]", "", p)
            if p and len(p) > 1 and not p.isdigit() and p.lower() not in {"nn", "pk", "id", "date", "on", "at"}:
                cleaned.append(p)
        return cleaned

    name = re.sub(r"[^a-zA-Z0-9_]", "", name)
    if (name and len(name) > 1 and not name.isdigit()
            and name.lower() not in {"nn", "pk", "id", "date", "on", "at"}
            and not re.match(r"^\d+nn$", name.lower())):
        return [name]
    return []


def parse_and_clean_schema(file_path: str) -> Dict[str, List[str]]:
    table_map = {}
    if not os.path.exists(file_path):
        print(f"[WARN] Schema file not found: {file_path}")
        return table_map

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for table_name, schema_str in data.items():
        clean_cols = []
        if isinstance(schema_str, str):
            for col_def in schema_str.split(","):
                clean_cols.extend(clean_column_name(col_def))
        elif isinstance(schema_str, list):
            clean_cols = [str(c) for c in schema_str if c]

        # Final filter + dedup
        table_map[table_name] = list(dict.fromkeys(
            c for c in clean_cols
            if c and len(c) > 1 and not c.isdigit() and not re.match(r"^\d", c)
        ))
    return table_map


def build_clean_registry(paths: Dict[str, str]) -> Dict[str, Dict[str, List[str]]]:
    registry = {}
    for db, path in paths.items():
        registry[db] = parse_and_clean_schema(path)
        print(f"[INFO] Loaded {db}: {len(registry[db])} tables")
    return registry


JEV_CLEAN_REGISTRY = build_clean_registry(SCHEMA_PATHS)


# -------------------------------------------------------------------
# 5. Smarter Matching Logic
# -------------------------------------------------------------------
def extract_tokens(query: str) -> Set[str]:
    tokens = set(re.findall(r"[a-zA-Z0-9_]+", query.lower()))
    return {t for t in tokens if t not in STOP_WORDS and len(t) > 1}


def find_matched_keywords(tokens: Set[str], vocab: Dict[str, List[str]]) -> Set[str]:
    """Only keep strong matches."""
    matched = set()
    for t in tokens:
        # Exact match
        if t in vocab:
            matched.add(t)
            continue
        # High-quality partial (token is contained in a vocab key or vice-versa, length check)
        for key in vocab:
            if (t in key or key in t) and min(len(t), len(key)) >= 4:
                matched.add(key)
    return matched


def score_table(table_name: str, matched_keywords: Set[str], vocab: Dict[str, List[str]]) -> int:
    score = 0
    table_l = table_name.lower()
    for kw in matched_keywords:
        if kw in vocab and table_name in vocab[kw]:
            score += 10          # direct mapping
        if kw in table_l:
            score += 6
        if table_l in kw:
            score += 4
    return score


def get_relevant_columns(cols: List[str], tokens: Set[str], matched: Set[str]) -> List[str]:
    scored = []
    for col in cols:
        col_l = col.lower()
        score = 0
        if col_l in tokens:
            score += 12
        for m in matched:
            if m in col_l or col_l in m:
                score += 5
        # Prefer useful columns
        if any(x in col_l for x in ["id", "name", "code", "status", "qty", "quantity", "util", "capacity", "date"]):
            score += 2
        scored.append((score, col))

    scored.sort(key=lambda x: (-x[0], x[1]))
    relevant = [c for s, c in scored if s >= 2]
    return relevant[:8] if relevant else cols[:6]


def customize_and_split_query(user_query: str) -> Dict[str, Any]:
    tokens = extract_tokens(user_query)
    result = {
        "raw_user_query": user_query,
        "selected_databases": [],
        "customized_db_requests": []
    }

    for db_name, tables in JEV_CLEAN_REGISTRY.items():
        vocab = DOMAIN_VOCABULARY.get(db_name, {})
        matched = find_matched_keywords(tokens, vocab)

        if not matched:
            continue

        # Collect candidate tables with score
        candidates = []
        for table_name in tables:
            sc = score_table(table_name, matched, vocab)
            if sc >= 6:                     # threshold – only keep strong tables
                candidates.append((sc, table_name))

        # Also force-include tables that are explicitly listed in vocab for matched keywords
        for kw in matched:
            for t in vocab.get(kw, []):
                # case-insensitive find
                real = next((r for r in tables if r.lower() == t.lower()), None)
                if real and not any(real == c[1] for c in candidates):
                    candidates.append((10, real))

        if not candidates:
            continue

        candidates.sort(key=lambda x: -x[0])
        # Keep top N tables (prevent explosion)
        top_tables = [t for _, t in candidates[:12]]

        mappings = []
        for table_name in top_tables:
            cols = tables[table_name]
            primary = get_relevant_columns(cols, tokens, matched)
            mappings.append({
                "responsible_table": table_name,
                "primary_columns": primary,
                "all_table_columns": cols
            })

        # Clean sub-query text
        keywords_str = ", ".join(sorted(matched))
        if db_name == "video_analytics":
            intent = f"Retrieve camera configuration / status / rules / alerts related to: {keywords_str}"
        else:
            intent = f"Retrieve machine / production / work-order / capacity data related to: {keywords_str}"

        result["selected_databases"].append(db_name)
        result["customized_db_requests"].append({
            "database": db_name,
            "customized_sub_query": intent,
            "token_keywords": sorted(list(matched)),
            "target_mappings": mappings
        })

    return result


# -------------------------------------------------------------------
# 6. Diagnostic Tests
# -------------------------------------------------------------------
def run_tests():
    print("=" * 70)
    print("RUNNING IMPROVED DIAGNOSTIC TESTS")
    print("=" * 70)

    tests = [
        ("Original multi-DB", "I want the data from the mes application for machine utilization and from video analytics I want camera configuration information."),
        ("Only MES util", "Show me machine utilization and capacity for last week"),
        ("Only Video camera", "Give me camera configuration and status of all cameras"),
        ("Work order + alerts", "I need work order status and any safety alerts or violations"),
        ("Shift + operator", "What is the shift wise machine utilization and operator assignment?"),
        ("Inventory + production", "Show finished goods inventory and production planning data"),
        ("HSE + zones", "List all active HSE camera rules and zone risk scores"),
        ("No match", "What is the weather today?"),
        ("Pure machine", "machine utilization report"),
        ("Pure camera", "camera configuration details"),
    ]

    for name, query in tests:
        print(f"\n--- {name} ---")
        print(f"Query: {query}")
        out = customize_and_split_query(query)
        print(f"Selected DBs: {out['selected_databases']}")

        for req in out["customized_db_requests"]:
            print(f"  → {req['database'].upper()}")
            print(f"     Keywords : {req['token_keywords']}")
            tables = [m['responsible_table'] for m in req['target_mappings']]
            print(f"     Tables   : {tables}")
            if tables:
                for m in req['target_mappings'][:3]:
                    print(f"        • {m['responsible_table']} → {m['primary_columns'][:5]}")
            else:
                print("     ⚠ No tables selected")

        if not out["selected_databases"]:
            print("  ✓ Correctly returned no databases")

    print("\n" + "=" * 70)
    print("SCHEMA HEALTH")
    print("=" * 70)
    for db, tables in JEV_CLEAN_REGISTRY.items():
        print(f"{db}: {len(tables)} tables")
        sample = list(tables.keys())[:5]
        print(f"  Sample: {sample}")


if __name__ == "__main__":
    run_tests()

    print("\n\n" + "=" * 70)
    print("FINAL RESULT FOR ORIGINAL QUERY")
    print("=" * 70)
    q = "I want the data from the mes application for machine utilization and from video analytics I want camera configuration information."
    print(json.dumps(customize_and_split_query(q), indent=2))

[INFO] Loaded video_analytics: 52 tables
[INFO] Loaded mes: 108 tables
RUNNING IMPROVED DIAGNOSTIC TESTS

--- Original multi-DB ---
Query: I want the data from the mes application for machine utilization and from video analytics I want camera configuration information.
Selected DBs: ['video_analytics', 'mes']
  → VIDEO_ANALYTICS
     Keywords : ['camera', 'configuration']
     Tables   : ['cameras', 'hse_camera_rules', 'basler_model_assignments', 'camera_status_logs', 'basler_devices', 'detection_assignments']
        • cameras → ['camera_number', 'department_id', 'location_id', 'name', 'plant_id']
        • hse_camera_rules → ['camera_id', 'config_override', 'rule_id', 'zone_id']
        • basler_model_assignments → ['basler_camera_id', 'confidence_threshold', 'model_id']
  → MES
     Keywords : ['machine', 'utilization']
     Tables   : ['MachineMaster', 'CapacityAnalysis', 'Machine', 'MachineFGMapping', 'MachineMaterialMapping', 'WeighingMachine']
        • MachineMaster → ['Utiliza

In [34]:
import json
import os
import re
from typing import Any, Dict, List, Set, Tuple
from collections import defaultdict

# -------------------------------------------------------------------
# 1. Schema Paths
# -------------------------------------------------------------------
SCHEMA_PATHS = {
    "video_analytics": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\video_analytics_db_info\construction_ai_schema.json",
    "mes": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.json",
}

# -------------------------------------------------------------------
# 2. Stop words
# -------------------------------------------------------------------
STOP_WORDS = {
    "a", "an", "the", "and", "or", "of", "in", "on", "for", "to", "from", "with",
    "is", "are", "was", "were", "be", "been", "being", "have", "has", "had",
    "do", "does", "did", "will", "would", "can", "could", "should", "may", "might",
    "i", "me", "my", "we", "you", "your", "he", "she", "it", "they", "them",
    "this", "that", "these", "those", "what", "which", "who", "whom", "whose",
    "all", "any", "some", "no", "not", "only", "just", "also", "very", "too",
    "want", "need", "show", "give", "get", "list", "data", "information", "details",
    "application", "system", "last", "week", "today", "yesterday", "status", "active",
    "report"
}

# -------------------------------------------------------------------
# 3. Domain Vocabulary
# -------------------------------------------------------------------
DOMAIN_VOCABULARY = {
    "video_analytics": {
        "camera": ["cameras", "camera_status_logs", "basler_devices", "basler_model_assignments", "hse_camera_rules"],
        "cameras": ["cameras", "camera_status_logs", "basler_devices"],
        "config": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments"],
        "configuration": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments"],
        "rule": ["hse_camera_rules", "hse_rule_definitions", "hse_rule_events"],
        "hse": ["hse_camera_rules", "hse_rule_definitions", "hse_rule_events"],
        "alert": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "violation": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "safety": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "anomaly": ["anomaly_flags"],
        "incident": ["incidents"],
        "zone": ["zones", "zone_risk_scores"],
        "worker": ["attendances", "employees", "employee_movements"],
        "employee": ["employees", "attendances", "employee_movements"],
        "attendance": ["attendances"],
        "model": ["ai_models", "ai_model_classes", "basler_model_assignments"],
        "basler": ["basler_devices", "basler_model_assignments"],
    },
    "mes": {
        "machine": ["Machine", "MachineMaster", "MachineFGMapping", "MachineMaterialMapping", "CapacityAnalysis"],
        "machines": ["Machine", "MachineMaster"],
        "utilization": ["CapacityAnalysis", "MachineMaster"],
        "util": ["CapacityAnalysis", "MachineMaster"],
        "capacity": ["CapacityAnalysis", "MachineMaster", "ContainerCapacity"],
        "workorder": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog", "WorkOrderResource"],
        "work_order": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog"],
        "wo": ["WorkOrder", "WorkOrderStep"],
        "production": ["WorkOrder", "ProductionPlanning", "FinishedGood", "SemiFinishedGood"],
        "shift": ["ShiftMaster", "ShiftCalendar", "OperatorMaster"],
        "operator": ["OperatorMaster", "WorkOrderResource"],
        "inventory": ["Inventory", "InventoryByLot", "FinishedGood", "SemiFinishedGood"],
        "material": ["Materials", "RawMaterial", "BOMLine", "BOMMaster"],
        "bom": ["BOMMaster", "BOMLine"],
        "mps": ["MPSHeader", "MpsMaster"],
        "mrp": ["MRP_Run", "MRP_Demand", "MRP_Result"],
        "planning": ["MPSHeader", "MpsMaster", "ProductionPlanning", "WeeklyPlanning"],
        "maintenance": ["MaintenanceWindow"],
        "schedule": ["GanttSchedule", "WorkOrder"],
    }
}

# -------------------------------------------------------------------
# 4. Schema Parser (cleaned)
# -------------------------------------------------------------------
def clean_column_name(raw: str) -> List[str]:
    name = raw.strip().split(":")[0].strip()
    if "/" in name:
        parts = re.split(r"[/]", name)
        cleaned = []
        for p in parts:
            p = re.sub(r"[^a-zA-Z0-9_]", "", p)
            if p and len(p) > 1 and not p.isdigit() and p.lower() not in {"nn", "pk", "id", "date", "on", "at"}:
                cleaned.append(p)
        return cleaned

    name = re.sub(r"[^a-zA-Z0-9_]", "", name)
    if (name and len(name) > 1 and not name.isdigit()
            and name.lower() not in {"nn", "pk", "id", "date", "on", "at"}
            and not re.match(r"^\d+nn$", name.lower())):
        return [name]
    return []


def parse_and_clean_schema(file_path: str) -> Dict[str, List[str]]:
    table_map = {}
    if not os.path.exists(file_path):
        print(f"[WARN] Schema file not found: {file_path}")
        return table_map

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for table_name, schema_str in data.items():
        clean_cols = []
        if isinstance(schema_str, str):
            for col_def in schema_str.split(","):
                clean_cols.extend(clean_column_name(col_def))
        elif isinstance(schema_str, list):
            clean_cols = [str(c) for c in schema_str if c]

        table_map[table_name] = list(dict.fromkeys(
            c for c in clean_cols
            if c and len(c) > 1 and not c.isdigit() and not re.match(r"^\d", c)
        ))
    return table_map


def build_clean_registry(paths: Dict[str, str]) -> Dict[str, Dict[str, List[str]]]:
    registry = {}
    for db, path in paths.items():
        registry[db] = parse_and_clean_schema(path)
        print(f"[INFO] Loaded {db}: {len(registry[db])} tables")
    return registry


JEV_CLEAN_REGISTRY = build_clean_registry(SCHEMA_PATHS)


# -------------------------------------------------------------------
# 5. Query Splitter (same as before – already good)
# -------------------------------------------------------------------
def extract_tokens(query: str) -> Set[str]:
    tokens = set(re.findall(r"[a-zA-Z0-9_]+", query.lower()))
    return {t for t in tokens if t not in STOP_WORDS and len(t) > 1}


def find_matched_keywords(tokens: Set[str], vocab: Dict[str, List[str]]) -> Set[str]:
    matched = set()
    for t in tokens:
        if t in vocab:
            matched.add(t)
            continue
        for key in vocab:
            if (t in key or key in t) and min(len(t), len(key)) >= 4:
                matched.add(key)
    return matched


def score_table(table_name: str, matched_keywords: Set[str], vocab: Dict[str, List[str]]) -> int:
    score = 0
    table_l = table_name.lower()
    for kw in matched_keywords:
        if kw in vocab and table_name in vocab[kw]:
            score += 10
        if kw in table_l:
            score += 6
        if table_l in kw:
            score += 4
    return score


def get_relevant_columns(cols: List[str], tokens: Set[str], matched: Set[str]) -> List[str]:
    scored = []
    for col in cols:
        col_l = col.lower()
        score = 0
        if col_l in tokens:
            score += 12
        for m in matched:
            if m in col_l or col_l in m:
                score += 5
        if any(x in col_l for x in ["id", "name", "code", "status", "qty", "quantity", "util", "capacity", "date"]):
            score += 2
        scored.append((score, col))

    scored.sort(key=lambda x: (-x[0], x[1]))
    relevant = [c for s, c in scored if s >= 2]
    return relevant[:8] if relevant else cols[:6]


def customize_and_split_query(user_query: str) -> Dict[str, Any]:
    tokens = extract_tokens(user_query)
    result = {
        "raw_user_query": user_query,
        "selected_databases": [],
        "customized_db_requests": []
    }

    for db_name, tables in JEV_CLEAN_REGISTRY.items():
        vocab = DOMAIN_VOCABULARY.get(db_name, {})
        matched = find_matched_keywords(tokens, vocab)

        if not matched:
            continue

        candidates = []
        for table_name in tables:
            sc = score_table(table_name, matched, vocab)
            if sc >= 6:
                candidates.append((sc, table_name))

        for kw in matched:
            for t in vocab.get(kw, []):
                real = next((r for r in tables if r.lower() == t.lower()), None)
                if real and not any(real == c[1] for c in candidates):
                    candidates.append((10, real))

        if not candidates:
            continue

        candidates.sort(key=lambda x: -x[0])
        top_tables = [t for _, t in candidates[:10]]

        mappings = []
        for table_name in top_tables:
            cols = tables[table_name]
            primary = get_relevant_columns(cols, tokens, matched)
            mappings.append({
                "responsible_table": table_name,
                "primary_columns": primary,
                "all_table_columns": cols
            })

        keywords_str = ", ".join(sorted(matched))
        if db_name == "video_analytics":
            intent = f"Retrieve camera configuration / status / rules / alerts related to: {keywords_str}"
        else:
            intent = f"Retrieve machine / production / work-order / capacity data related to: {keywords_str}"

        result["selected_databases"].append(db_name)
        result["customized_db_requests"].append({
            "database": db_name,
            "customized_sub_query": intent,
            "token_keywords": sorted(list(matched)),
            "target_mappings": mappings
        })

    return result


# -------------------------------------------------------------------
# 6. NEW LAYER → SQL Query Generator (Correct & Safe)
# -------------------------------------------------------------------
def detect_date_column(columns: List[str]) -> str | None:
    """Try to find a good date/timestamp column for ORDER BY."""
    priority = ["created_at", "updated_at", "checked_at", "timestamp", "date", "createdon", "updatedon", "startdate", "enddate"]
    cols_lower = {c.lower(): c for c in columns}
    for p in priority:
        if p in cols_lower:
            return cols_lower[p]
    # fallback: any column containing date/time
    for c in columns:
        if any(x in c.lower() for x in ["date", "time", "at", "on"]):
            return c
    return None


def generate_sql_for_table(table_name: str, primary_columns: List[str], all_columns: List[str], limit: int = 100) -> str:
    """Generate a clean, safe SELECT statement."""
    # Prefer primary_columns, fall back to all if empty
    cols = primary_columns if primary_columns else all_columns[:8]
    if not cols:
        cols = ["*"]

    # Quote column names safely (works for both Postgres & SQL Server)
    col_list = ", ".join(f'"{c}"' for c in cols)

    sql = f'SELECT {col_list}\nFROM "{table_name}"'

    date_col = detect_date_column(all_columns)
    if date_col:
        sql += f'\nORDER BY "{date_col}" DESC'

    sql += f"\nLIMIT {limit};"
    return sql


def generate_sql_queries(pipeline_output: Dict[str, Any], limit_per_table: int = 50) -> Dict[str, Any]:
    """
    Main SQL generation layer.
    Takes the output of customize_and_split_query() and produces executable SQL.
    """
    sql_result = {
        "raw_user_query": pipeline_output.get("raw_user_query"),
        "databases": []
    }

    for req in pipeline_output.get("customized_db_requests", []):
        db_entry = {
            "database": req["database"],
            "intent": req["customized_sub_query"],
            "keywords": req["token_keywords"],
            "queries": []
        }

        for mapping in req["target_mappings"]:
            table = mapping["responsible_table"]
            primary = mapping["primary_columns"]
            all_cols = mapping["all_table_columns"]

            sql = generate_sql_for_table(table, primary, all_cols, limit=limit_per_table)

            db_entry["queries"].append({
                "table": table,
                "sql": sql,
                "columns_used": primary if primary else all_cols[:8],
                "explanation": f"Fetch relevant columns from {table} ordered by most recent records"
            })

        sql_result["databases"].append(db_entry)

    return sql_result


# -------------------------------------------------------------------
# 7. Pretty printer + Test harness
# -------------------------------------------------------------------
def print_sql_result(sql_result: Dict[str, Any]):
    print("\n" + "=" * 70)
    print("GENERATED SQL QUERIES")
    print("=" * 70)
    print(f"Original Query: {sql_result['raw_user_query']}\n")

    for db in sql_result["databases"]:
        print(f"▶ DATABASE: {db['database'].upper()}")
        print(f"  Intent   : {db['intent']}")
        print(f"  Keywords : {db['keywords']}\n")

        for i, q in enumerate(db["queries"], 1):
            print(f"  [{i}] Table: {q['table']}")
            print(f"      Columns: {q['columns_used']}")
            print(f"      SQL:")
            for line in q["sql"].splitlines():
                print(f"        {line}")
            print()


def run_full_pipeline(user_query: str):
    print(f"\n{'='*70}")
    print(f"USER QUERY: {user_query}")
    print(f"{'='*70}")

    # Step 1: Split + schema context
    split_result = customize_and_split_query(user_query)

    # Step 2: Generate SQL
    sql_result = generate_sql_queries(split_result, limit_per_table=50)

    # Pretty print
    print_sql_result(sql_result)

    return sql_result


# -------------------------------------------------------------------
# 8. Main – Test the full pipeline
# -------------------------------------------------------------------
if __name__ == "__main__":
    test_queries = [
        "I want the data from the mes application for machine utilization and from video analytics I want camera configuration information.",
        "Show me machine utilization and capacity for last week",
        "Give me camera configuration and status of all cameras",
        "I need work order status and any safety alerts or violations",
        "List all active HSE camera rules and zone risk scores",
    ]

    for q in test_queries:
        run_full_pipeline(q)
        print("\n" + "-" * 70 + "\n")

[INFO] Loaded video_analytics: 52 tables
[INFO] Loaded mes: 108 tables

USER QUERY: I want the data from the mes application for machine utilization and from video analytics I want camera configuration information.

GENERATED SQL QUERIES
Original Query: I want the data from the mes application for machine utilization and from video analytics I want camera configuration information.

▶ DATABASE: VIDEO_ANALYTICS
  Intent   : Retrieve camera configuration / status / rules / alerts related to: camera, configuration
  Keywords : ['camera', 'configuration']

  [1] Table: cameras
      Columns: ['camera_number', 'department_id', 'location_id', 'name', 'plant_id', 'status', 'user_id']
      SQL:
        SELECT "camera_number", "department_id", "location_id", "name", "plant_id", "status", "user_id"
        FROM "cameras"
        ORDER BY "rtsp_template" DESC
        LIMIT 50;

  [2] Table: hse_camera_rules
      Columns: ['camera_id', 'config_override', 'rule_id', 'zone_id']
      SQL:
        

In [44]:
import json
import os
import re
from typing import Any, Dict, List, Set, Optional, Tuple
from datetime import datetime, timedelta
from pathlib import Path

# -------------------------------------------------------------------
# Third-party libs
# -------------------------------------------------------------------
try:
    from dotenv import load_dotenv
except ImportError:
    raise ImportError("pip install python-dotenv")

try:
    import pyodbc
except ImportError:
    raise ImportError("pip install pyodbc")

try:
    import psycopg2
    import psycopg2.extras
except ImportError:
    raise ImportError("pip install psycopg2-binary")

try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4, landscape
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import mm
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
        PageBreak, HRFlowable
    )
    from reportlab.lib.enums import TA_CENTER
except ImportError:
    raise ImportError("pip install reportlab")

load_dotenv()

# -------------------------------------------------------------------
# 1. Schema Paths
# -------------------------------------------------------------------
SCHEMA_PATHS = {
    "video_analytics": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\video_analytics_db_info\construction_ai_schema.json",
    "mes": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.json",
}

# -------------------------------------------------------------------
# 2. Stop words
# -------------------------------------------------------------------
STOP_WORDS = {
    "a", "an", "the", "and", "or", "of", "in", "on", "for", "to", "from", "with",
    "is", "are", "was", "were", "be", "been", "being", "have", "has", "had",
    "do", "does", "did", "will", "would", "can", "could", "should", "may", "might",
    "i", "me", "my", "we", "you", "your", "he", "she", "it", "they", "them",
    "this", "that", "these", "those", "what", "which", "who", "whom", "whose",
    "all", "any", "some", "no", "not", "only", "just", "also", "very", "too",
    "want", "need", "show", "give", "get", "list", "data", "information", "details",
    "application", "system", "report", "me", "please"
}

# -------------------------------------------------------------------
# 3. Domain Vocabulary
# -------------------------------------------------------------------
DOMAIN_VOCABULARY = {
    "video_analytics": {
        "camera": ["cameras", "camera_status_logs", "basler_devices", "basler_model_assignments", "hse_camera_rules"],
        "cameras": ["cameras", "camera_status_logs", "basler_devices"],
        "config": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments"],
        "configuration": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments"],
        "rule": ["hse_camera_rules", "hse_rule_definitions", "hse_rule_events"],
        "hse": ["hse_camera_rules", "hse_rule_definitions", "hse_rule_events"],
        "alert": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "violation": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "safety": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "anomaly": ["anomaly_flags"],
        "incident": ["incidents"],
        "zone": ["zones", "zone_risk_scores"],
        "worker": ["attendances", "employees", "employee_movements"],
        "employee": ["employees", "attendances", "employee_movements"],
        "attendance": ["attendances"],
        "model": ["ai_models", "ai_model_classes", "basler_model_assignments"],
        "basler": ["basler_devices", "basler_model_assignments"],
    },
    "mes": {
        "machine": ["Machine", "MachineMaster", "MachineFGMapping", "MachineMaterialMapping", "CapacityAnalysis"],
        "machines": ["Machine", "MachineMaster"],
        "utilization": ["CapacityAnalysis", "MachineMaster"],
        "util": ["CapacityAnalysis", "MachineMaster"],
        "capacity": ["CapacityAnalysis", "MachineMaster", "ContainerCapacity"],
        "workorder": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog", "WorkOrderResource"],
        "work_order": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog"],
        "wo": ["WorkOrder", "WorkOrderStep"],
        "production": ["WorkOrder", "ProductionPlanning", "FinishedGood", "SemiFinishedGood"],
        "shift": ["ShiftMaster", "ShiftCalendar", "OperatorMaster"],
        "operator": ["OperatorMaster", "WorkOrderResource"],
        "inventory": ["Inventory", "InventoryByLot", "FinishedGood", "SemiFinishedGood"],
        "material": ["Materials", "RawMaterial", "BOMLine", "BOMMaster"],
        "bom": ["BOMMaster", "BOMLine"],
        "mps": ["MPSHeader", "MpsMaster"],
        "mrp": ["MRP_Run", "MRP_Demand", "MRP_Result"],
        "planning": ["MPSHeader", "MpsMaster", "ProductionPlanning", "WeeklyPlanning"],
        "maintenance": ["MaintenanceWindow"],
        "schedule": ["GanttSchedule", "WorkOrder"],
        "downtime": ["MaintenanceWindow", "CapacityAnalysis"],
    }
}

# -------------------------------------------------------------------
# 4. Schema Cleaner
# -------------------------------------------------------------------
NOISE_COLUMNS = {
    "nn", "pk", "id", "date", "on", "at", "_at", "created", "updated",
    "qty", "quantity", "count", "bags", "bag"
}

def clean_column_name(raw: str) -> List[str]:
    name = raw.strip().split(":")[0].strip()
    if "/" in name:
        parts = re.split(r"[/]", name)
        cleaned = []
        for p in parts:
            p = re.sub(r"[^a-zA-Z0-9_]", "", p)
            if (p and len(p) > 2 and not p.isdigit()
                    and p.lower() not in NOISE_COLUMNS and not p.startswith("_")):
                cleaned.append(p)
        return cleaned
    name = re.sub(r"[^a-zA-Z0-9_]", "", name)
    if (name and len(name) > 2 and not name.isdigit()
            and name.lower() not in NOISE_COLUMNS
            and not name.startswith("_") and not re.match(r"^\d", name)):
        return [name]
    return []


def parse_and_clean_schema(file_path: str) -> Dict[str, List[str]]:
    table_map = {}
    if not os.path.exists(file_path):
        print(f"[WARN] Schema file not found: {file_path}")
        return table_map
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for table_name, schema_str in data.items():
        clean_cols = []
        if isinstance(schema_str, str):
            for col_def in schema_str.split(","):
                clean_cols.extend(clean_column_name(col_def))
        elif isinstance(schema_str, list):
            clean_cols = [str(c) for c in schema_str if c]
        table_map[table_name] = list(dict.fromkeys(
            c for c in clean_cols
            if c and len(c) > 2 and not c.startswith("_")
            and c.lower() not in NOISE_COLUMNS and not re.match(r"^\d", c)
        ))
    return table_map


def build_clean_registry(paths: Dict[str, str]) -> Dict[str, Dict[str, List[str]]]:
    registry = {}
    for db, path in paths.items():
        registry[db] = parse_and_clean_schema(path)
        print(f"[INFO] Loaded {db}: {len(registry[db])} tables")
    return registry


JEV_CLEAN_REGISTRY = build_clean_registry(SCHEMA_PATHS)

# -------------------------------------------------------------------
# 5. Query Splitter
# -------------------------------------------------------------------
def extract_tokens(query: str) -> Set[str]:
    tokens = set(re.findall(r"[a-zA-Z0-9_]+", query.lower()))
    return {t for t in tokens if t not in STOP_WORDS and len(t) > 1}


def find_matched_keywords(tokens: Set[str], vocab: Dict[str, List[str]]) -> Set[str]:
    matched = set()
    for t in tokens:
        if t in vocab:
            matched.add(t)
            continue
        for key in vocab:
            if (t in key or key in t) and min(len(t), len(key)) >= 4:
                matched.add(key)
    return matched


def score_table(table_name: str, matched_keywords: Set[str], vocab: Dict[str, List[str]]) -> int:
    score = 0
    table_l = table_name.lower()
    for kw in matched_keywords:
        if kw in vocab and table_name in vocab[kw]:
            score += 10
        if kw in table_l:
            score += 6
    return score


def get_relevant_columns(cols: List[str], tokens: Set[str], matched: Set[str]) -> List[str]:
    scored = []
    for col in cols:
        col_l = col.lower()
        score = 0
        if col_l in tokens:
            score += 12
        for m in matched:
            if m in col_l or col_l in m:
                score += 5
        if any(x in col_l for x in ["id", "name", "code", "status", "qty", "quantity", "util", "capacity", "is_active", "active"]):
            score += 2
        scored.append((score, col))
    scored.sort(key=lambda x: (-x[0], x[1]))
    relevant = [c for s, c in scored if s >= 2]
    if len(relevant) < 3:
        relevant = cols[:8]
    return relevant[:8]


def customize_and_split_query(user_query: str) -> Dict[str, Any]:
    tokens = extract_tokens(user_query)
    result = {
        "raw_user_query": user_query,
        "selected_databases": [],
        "customized_db_requests": []
    }
    for db_name, tables in JEV_CLEAN_REGISTRY.items():
        vocab = DOMAIN_VOCABULARY.get(db_name, {})
        matched = find_matched_keywords(tokens, vocab)
        if not matched:
            continue
        candidates = []
        for table_name in tables:
            sc = score_table(table_name, matched, vocab)
            if sc >= 6:
                candidates.append((sc, table_name))
        for kw in matched:
            for t in vocab.get(kw, []):
                real = next((r for r in tables if r.lower() == t.lower()), None)
                if real and not any(real == c[1] for c in candidates):
                    candidates.append((10, real))
        if not candidates:
            continue
        candidates.sort(key=lambda x: -x[0])
        top_tables = [t for _, t in candidates[:10]]
        mappings = []
        for table_name in top_tables:
            cols = tables[table_name]
            primary = get_relevant_columns(cols, tokens, matched)
            mappings.append({
                "responsible_table": table_name,
                "primary_columns": primary,
                "all_table_columns": cols
            })
        keywords_str = ", ".join(sorted(matched))
        intent = (f"Retrieve camera / rules / alerts related to: {keywords_str}"
                  if db_name == "video_analytics"
                  else f"Retrieve machine / production / work-order data related to: {keywords_str}")
        result["selected_databases"].append(db_name)
        result["customized_db_requests"].append({
            "database": db_name,
            "customized_sub_query": intent,
            "token_keywords": sorted(list(matched)),
            "target_mappings": mappings
        })
    return result

# -------------------------------------------------------------------
# 6. Date Detection
# -------------------------------------------------------------------
REAL_DATE_HINTS = [
    "created_at", "updated_at", "checked_at", "triggered_at", "timestamp",
    "createdon", "updatedon", "created_date", "updated_date",
    "startdate", "enddate", "plannedstart", "actualstart", "date",
    "changedat", "lastupdated"
]
BAD_DATE_COLUMNS = {
    "updatedby", "createdby", "status", "type", "name", "code", "location",
    "rtsp_template", "utilizationpercent", "utilization"
}

def detect_date_column(columns: List[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in columns}
    for p in REAL_DATE_HINTS:
        if p in cols_lower:
            return cols_lower[p]
    for c in columns:
        cl = c.lower()
        if cl in BAD_DATE_COLUMNS:
            continue
        if any(h in cl for h in ["_at", "_on", "date", "time", "timestamp"]) and len(cl) > 4:
            return c
    return None


def extract_time_filter(user_query: str) -> Optional[str]:
    q = user_query.lower()
    today = datetime.now().date()
    if "last week" in q or "past week" in q:
        start = today - timedelta(days=7)
        return f"{{date_col}} >= '{start}'"
    if "today" in q:
        return f"{{date_col}} >= '{today}'"
    if "yesterday" in q:
        y = today - timedelta(days=1)
        return f"{{date_col}} >= '{y}' AND {{date_col}} < '{today}'"
    if "last month" in q or "past month" in q:
        start = today - timedelta(days=30)
        return f"{{date_col}} >= '{start}'"
    return None

# -------------------------------------------------------------------
# 7. SQL Generator
# -------------------------------------------------------------------
def generate_sql_for_table(
    table_name: str,
    primary_columns: List[str],
    all_columns: List[str],
    time_filter: Optional[str] = None,
    limit: int = 50
) -> str:
    cols = primary_columns if primary_columns else all_columns[:8]
    if not cols:
        cols = ["*"]
    for possible_id in ["id", "Id", "ID", f"{table_name}Id"]:
        if possible_id in all_columns and possible_id not in cols:
            cols.insert(0, possible_id)
            break
    col_list = ", ".join(f'"{c}"' for c in cols)
    sql = f'SELECT {col_list}\nFROM "{table_name}"'
    date_col = detect_date_column(all_columns)
    if time_filter and date_col:
        real_filter = time_filter.replace("{date_col}", f'"{date_col}"')
        sql += f"\nWHERE {real_filter}"
    if date_col:
        sql += f'\nORDER BY "{date_col}" DESC'
    sql += f"\nLIMIT {limit};"
    return sql


def generate_sql_queries(pipeline_output: Dict[str, Any], limit_per_table: int = 50) -> Dict[str, Any]:
    time_filter = extract_time_filter(pipeline_output.get("raw_user_query", ""))
    sql_result = {
        "raw_user_query": pipeline_output.get("raw_user_query"),
        "time_filter_applied": time_filter,
        "databases": []
    }
    for req in pipeline_output.get("customized_db_requests", []):
        db_entry = {
            "database": req["database"],
            "intent": req["customized_sub_query"],
            "keywords": req["token_keywords"],
            "queries": []
        }
        for mapping in req["target_mappings"]:
            table = mapping["responsible_table"]
            primary = mapping["primary_columns"]
            all_cols = mapping["all_table_columns"]
            sql = generate_sql_for_table(table, primary, all_cols, time_filter, limit=limit_per_table)
            db_entry["queries"].append({
                "table": table,
                "sql": sql,
                "columns_used": primary if primary else all_cols[:8],
                "date_column_used": detect_date_column(all_cols),
                "explanation": f"Fetch relevant columns from {table}"
            })
        sql_result["databases"].append(db_entry)
    return sql_result

# -------------------------------------------------------------------
# 8. Validator
# -------------------------------------------------------------------
def validate_sql(sql: str, table: str, columns_used: List[str], date_col: Optional[str]) -> List[str]:
    issues = []
    if date_col and "ORDER BY" not in sql:
        issues.append(f"Has date column '{date_col}' but no ORDER BY")
    if len(columns_used) < 2:
        issues.append("Too few columns selected")
    if "*" in sql:
        issues.append("Using SELECT *")
    for bad in ["_at", "4NN", "2NN", "DateOn"]:
        if f'"{bad}"' in sql or f".{bad}" in sql:
            issues.append(f"Noise column still present: {bad}")
    return issues


def analyze_pipeline_result(sql_result: Dict[str, Any]) -> Dict[str, Any]:
    report = {
        "query": sql_result["raw_user_query"],
        "databases_selected": [d["database"] for d in sql_result["databases"]],
        "total_queries": 0,
        "issues_found": [],
        "status": "OK"
    }
    for db in sql_result["databases"]:
        for q in db["queries"]:
            report["total_queries"] += 1
            issues = validate_sql(q["sql"], q["table"], q["columns_used"], q.get("date_column_used"))
            for iss in issues:
                report["issues_found"].append(f"[{db['database']}.{q['table']}] {iss}")
    if report["issues_found"]:
        report["status"] = "ISSUES_DETECTED"
    return report

# -------------------------------------------------------------------
# 9. Database connections & execution
# -------------------------------------------------------------------
def get_mes_connection():
    driver = os.getenv("DB_DRIVER", "ODBC Driver 18 for SQL Server")
    server = os.getenv("DB_SERVER", "localhost,1433")
    database = os.getenv("DB_NAME", "mes_new")
    trusted = os.getenv("DB_TRUSTED_CONNECTION", "yes").lower() == "yes"
    encrypt = os.getenv("DB_ENCRYPT", "no")
    trust_cert = os.getenv("DB_TRUST_SERVER_CERTIFICATE", "yes")
    conn_str = (
        f"DRIVER={{{driver}}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Trusted_Connection={'yes' if trusted else 'no'};"
        f"Encrypt={encrypt};"
        f"TrustServerCertificate={trust_cert};"
    )
    return pyodbc.connect(conn_str, timeout=15)


def get_video_analytics_connection():
    url = os.getenv("CONSTRUCTION_DB_URL")
    if url and url.startswith("postgresql"):
        url = url.replace("postgresql+psycopg2://", "postgresql://")
        return psycopg2.connect(url)
    return psycopg2.connect(
        host=os.getenv("PG_HOST", "localhost"),
        port=os.getenv("PG_PORT", "5432"),
        dbname=os.getenv("PG_DB", "construction_ai"),
        user=os.getenv("PG_USER", "postgres"),
        password=os.getenv("PG_PASSWORD", "0987654321")
    )


def execute_sql(db_name: str, sql: str) -> Tuple[List[str], List[Tuple]]:
    if db_name == "mes":
        sql = re.sub(r'"([^"]+)"', r'[\1]', sql)
        m = re.search(r"LIMIT\s+(\d+)", sql, re.IGNORECASE)
        if m:
            limit = m.group(1)
            sql = re.sub(r"LIMIT\s+\d+", "", sql, flags=re.IGNORECASE)
            sql = re.sub(r"(SELECT\s+)", rf"\1TOP {limit} ", sql, count=1, flags=re.IGNORECASE)
    try:
        if db_name == "mes":
            conn = get_mes_connection()
            cursor = conn.cursor()
            cursor.execute(sql)
            columns = [col[0] for col in cursor.description] if cursor.description else []
            rows = cursor.fetchall()
            cursor.close()
            conn.close()
            return columns, [tuple(r) for r in rows]
        else:
            conn = get_video_analytics_connection()
            cursor = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)
            cursor.execute(sql)
            columns = [desc[0] for desc in cursor.description] if cursor.description else []
            rows = cursor.fetchall()
            cursor.close()
            conn.close()
            return columns, [tuple(r) for r in rows]
    except Exception as e:
        print(f"[ERROR] Failed to execute on {db_name}: {e}")
        print(f"SQL was:\n{sql}")
        return [], []


def run_all_queries(sql_result: Dict[str, Any]) -> Dict[str, Any]:
    execution_result = {
        "raw_user_query": sql_result["raw_user_query"],
        "time_filter_applied": sql_result.get("time_filter_applied"),
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "databases": []
    }
    for db_entry in sql_result.get("databases", []):
        db_name = db_entry["database"]
        new_db = {
            "database": db_name,
            "intent": db_entry["intent"],
            "keywords": db_entry["keywords"],
            "queries": []
        }
        for q in db_entry["queries"]:
            cols, rows = execute_sql(db_name, q["sql"])
            new_db["queries"].append({
                "table": q["table"],
                "sql": q["sql"],
                "columns": cols,
                "rows": rows,
                "row_count": len(rows),
                "explanation": q["explanation"]
            })
        execution_result["databases"].append(new_db)
    return execution_result

# -------------------------------------------------------------------
# 10. Rich Metadata Builder
# -------------------------------------------------------------------
def _detect_type(values: list) -> str:
    non_null = [v for v in values if v is not None]
    if not non_null:
        return "unknown"
    sample = non_null[0]
    if isinstance(sample, bool):
        return "boolean"
    if isinstance(sample, int) and not isinstance(sample, bool):
        return "integer"
    if isinstance(sample, float):
        return "numeric"
    try:
        float(sample)
        return "numeric"
    except (TypeError, ValueError):
        pass
    return "string"


def _build_rich_metadata(
    table_name: str,
    columns: List[str],
    rows: List[Tuple],
    max_sample: int = 5
) -> Dict[str, Any]:
    if not rows or not columns:
        return {
            "table_name": table_name,
            "total_records": 0,
            "total_columns": len(columns),
            "columns": columns,
            "column_details": {},
            "data_quality": {}
        }

    total_records = len(rows)
    col_details = {}
    zero_values = {}
    potential_outliers = []

    for col_idx, col_name in enumerate(columns):
        col_values = [row[col_idx] for row in rows]
        non_null = [v for v in col_values if v is not None]
        null_count = total_records - len(non_null)
        unique_count = len(set(str(v) for v in non_null))

        dtype = _detect_type(col_values)
        detail = {
            "data_type": dtype,
            "null_count": null_count,
            "unique_count": unique_count,
            "sample_values": []
        }

        samples = []
        seen = set()
        for v in non_null:
            s = str(v)
            if s not in seen:
                samples.append(v)
                seen.add(s)
            if len(samples) >= max_sample:
                break
        detail["sample_values"] = samples

        if dtype in ("numeric", "integer"):
            nums = []
            for v in non_null:
                try:
                    nums.append(float(v))
                except (TypeError, ValueError):
                    continue
            if nums:
                detail["min"] = round(min(nums), 2)
                detail["max"] = round(max(nums), 2)
                detail["avg"] = round(sum(nums) / len(nums), 2)

                zero_cnt = sum(1 for n in nums if n == 0)
                if zero_cnt:
                    zero_values[col_name] = zero_cnt

                if len(nums) > 5:
                    avg = detail["avg"]
                    for n in nums:
                        if avg > 0 and n > avg * 10:
                            potential_outliers.append({
                                "column": col_name,
                                "value": n,
                                "issue": f"Unusually high value (>{avg * 10:.1f})"
                            })
                            break

        col_details[col_name] = detail

    return {
        "table_name": table_name,
        "total_records": total_records,
        "total_columns": len(columns),
        "columns": columns,
        "column_details": col_details,
        "data_quality": {
            "potential_outliers": potential_outliers,
            "zero_values": zero_values
        }
    }


def _truncate(val: Any, max_len: int = 40) -> str:
    s = str(val) if val is not None else ""
    return s if len(s) <= max_len else s[:max_len - 1] + "…"


def _render_metadata_block(meta: Dict[str, Any], styles) -> list:
    elements = []

    header = (
        f"<b>Table:</b> {meta['table_name']} &nbsp;&nbsp;|&nbsp;&nbsp; "
        f"<b>Records:</b> {meta['total_records']} &nbsp;&nbsp;|&nbsp;&nbsp; "
        f"<b>Columns:</b> {meta['total_columns']}"
    )
    elements.append(Paragraph(header, styles["MetadataBox"]))

    # Column details table
    col_data = [["Column", "Type", "Min", "Max", "Avg", "Nulls", "Unique", "Sample"]]
    for col, d in meta["column_details"].items():
        col_data.append([
            Paragraph(f"<b>{col}</b>", styles["Meta"]),
            d.get("data_type", ""),
            str(d.get("min", "–")),
            str(d.get("max", "–")),
            str(d.get("avg", "–")),
            str(d.get("null_count", 0)),
            str(d.get("unique_count", 0)),
            ", ".join(str(s) for s in d.get("sample_values", [])[:3])
        ])

    t = Table(col_data, colWidths=[90, 55, 50, 50, 50, 40, 45, 150])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2874a6")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 7),
        ("ALIGN", (2, 1), (6, -1), "CENTER"),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#aab7b8")),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#eaf2f8")]),
        ("LEFTPADDING", (0, 0), (-1, -1), 3),
        ("RIGHTPADDING", (0, 0), (-1, -1), 3),
        ("TOPPADDING", (0, 0), (-1, -1), 2),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]))
    elements.append(t)
    elements.append(Spacer(1, 3 * mm))

    # Data quality notes
    dq = meta.get("data_quality", {})
    notes = []
    if dq.get("potential_outliers"):
        for o in dq["potential_outliers"][:3]:
            notes.append(f"• {o['column']}: {o['value']} – {o['issue']}")
    if dq.get("zero_values"):
        zeros = ", ".join(f"{k} ({v})" for k, v in dq["zero_values"].items())
        notes.append(f"• Zero values found in: {zeros}")

    if notes:
        elements.append(Paragraph("<b>Data Quality Notes:</b>", styles["Meta"]))
        for n in notes:
            elements.append(Paragraph(n, styles["Meta"]))
        elements.append(Spacer(1, 2 * mm))

    return elements

# -------------------------------------------------------------------
# 11. PDF Generator  (FINAL RULE)
# -------------------------------------------------------------------
def generate_pdf_report(
    execution_result: Dict[str, Any],
    output_path: str = None,
    metadata_threshold: int = 5
) -> str:
    """
    Final rule:
      - rows > 5  → show ONLY rich Metadata (no data table)
      - rows ≤ 5  → show the small data table (no metadata)
    """
    if output_path is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"query_report_{ts}.pdf"

    output_path = str(Path(output_path).resolve())

    doc = SimpleDocTemplate(
        output_path,
        pagesize=landscape(A4),
        leftMargin=12 * mm,
        rightMargin=12 * mm,
        topMargin=12 * mm,
        bottomMargin=12 * mm
    )

    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="ReportTitle", parent=styles["Heading1"],
                               fontSize=16, alignment=TA_CENTER, spaceAfter=6))
    styles.add(ParagraphStyle(name="SectionHeader", parent=styles["Heading2"],
                               fontSize=12, textColor=colors.HexColor("#1a5276"),
                               spaceBefore=10, spaceAfter=4))
    styles.add(ParagraphStyle(name="SubHeader", parent=styles["Heading3"],
                               fontSize=10, textColor=colors.HexColor("#2874a6"),
                               spaceBefore=6, spaceAfter=3))
    styles.add(ParagraphStyle(name="Meta", parent=styles["Normal"],
                               fontSize=8, textColor=colors.grey))
    styles.add(ParagraphStyle(name="MetadataBox", parent=styles["Normal"],
                               fontSize=8, leading=11,
                               backColor=colors.HexColor("#eaf2f8"),
                               borderPadding=6, spaceBefore=2, spaceAfter=4))

    story = []

    # Title
    story.append(Paragraph("Manufacturing & Video-Analytics Query Report", styles["ReportTitle"]))
    story.append(Spacer(1, 4 * mm))
    story.append(Paragraph(f"<b>User Query:</b> {execution_result['raw_user_query']}", styles["Normal"]))
    story.append(Paragraph(f"<b>Generated at:</b> {execution_result['generated_at']}", styles["Meta"]))
    if execution_result.get("time_filter_applied"):
        story.append(Paragraph(f"<b>Time filter:</b> {execution_result['time_filter_applied']}", styles["Meta"]))
    story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor("#1a5276")))
    story.append(Spacer(1, 6 * mm))

    total_tables = 0
    total_rows = 0

    for db in execution_result["databases"]:
        story.append(Paragraph(f"Database: {db['database'].upper()}", styles["SectionHeader"]))
        story.append(Paragraph(f"Intent: {db['intent']}", styles["Meta"]))
        story.append(Paragraph(f"Keywords: {', '.join(db['keywords'])}", styles["Meta"]))
        story.append(Spacer(1, 3 * mm))

        for q in db["queries"]:
            total_tables += 1
            row_count = q["row_count"]
            total_rows += row_count

            header = f"Table: {q['table']}  ({row_count} rows)"
            story.append(Paragraph(header, styles["SubHeader"]))

            # -------------------------------------------------------
            # FINAL RULE
            # -------------------------------------------------------
            if row_count > metadata_threshold:
                # Only Metadata – NO data table
                meta = _build_rich_metadata(q["table"], q["columns"], q["rows"])
                for flowable in _render_metadata_block(meta, styles):
                    story.append(flowable)
                story.append(Spacer(1, 4 * mm))
            else:
                # Small result set → show the actual rows
                if not q["rows"]:
                    story.append(Paragraph("<i>No data returned.</i>", styles["Meta"]))
                    story.append(Spacer(1, 4 * mm))
                    continue

                col_names = q["columns"][:10]
                data = [[Paragraph(f"<b>{c}</b>", styles["Meta"]) for c in col_names]]
                for row in q["rows"]:
                    data.append([Paragraph(_truncate(cell), styles["Meta"]) for cell in row[:10]])

                col_widths = [doc.width / len(col_names)] * len(col_names)
                t = Table(data, colWidths=col_widths, repeatRows=1)
                t.setStyle(TableStyle([
                    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1a5276")),
                    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
                    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                    ("FONTSIZE", (0, 0), (-1, -1), 7),
                    ("ALIGN", (0, 0), (-1, -1), "LEFT"),
                    ("VALIGN", (0, 0), (-1, -1), "TOP"),
                    ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#aab7b8")),
                    ("ROWBACKGROUNDS", (0, 1), (-1, -1),
                     [colors.white, colors.HexColor("#eaf2f8")]),
                    ("LEFTPADDING", (0, 0), (-1, -1), 3),
                    ("RIGHTPADDING", (0, 0), (-1, -1), 3),
                    ("TOPPADDING", (0, 0), (-1, -1), 2),
                    ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
                ]))
                story.append(t)
                story.append(Spacer(1, 5 * mm))

        story.append(PageBreak())

    # Overall summary
    story.append(Paragraph("Overall Summary", styles["SectionHeader"]))
    story.append(Paragraph(
        f"Databases queried: {len(execution_result['databases'])} &nbsp;&nbsp;|&nbsp;&nbsp; "
        f"Tables: {total_tables} &nbsp;&nbsp;|&nbsp;&nbsp; "
        f"Total rows returned: {total_rows}",
        styles["Normal"]
    ))
    story.append(Paragraph(
        f"<i>Tables with more than {metadata_threshold} records show only Metadata. "
        f"Tables with ≤ {metadata_threshold} records show the actual data.</i>",
        styles["Meta"]
    ))

    doc.build(story)
    print(f"[INFO] PDF report written to: {output_path}")
    return output_path

# -------------------------------------------------------------------
# 12. High-level entry point
# -------------------------------------------------------------------
def process_query_and_generate_pdf(user_query: str, pdf_path: str = None) -> str:
    print(f"\n>>> Processing: {user_query}")
    split = customize_and_split_query(user_query)
    if not split["selected_databases"]:
        print("[WARN] No databases matched the query.")
    sql_result = generate_sql_queries(split, limit_per_table=50)
    report = analyze_pipeline_result(sql_result)
    print(f"DBs selected : {report['databases_selected']}")
    print(f"Queries gen  : {report['total_queries']}")
    print(f"Status       : {report['status']}")

    execution = run_all_queries(sql_result)
    pdf_file = generate_pdf_report(execution, output_path=pdf_path)
    return pdf_file

# -------------------------------------------------------------------
# 13. Main
# -------------------------------------------------------------------
if __name__ == "__main__":
    user_q = (
        "I want the data from the mes application for machine utilization "
        "and from video analytics I want camera configuration information."
    )
    pdf_file = process_query_and_generate_pdf(user_q, pdf_path="mes_and_camera_report.pdf")
    print(f"\n✅ Done. Open the PDF: {pdf_file}")

[INFO] Loaded video_analytics: 52 tables
[INFO] Loaded mes: 108 tables

>>> Processing: I want the data from the mes application for machine utilization and from video analytics I want camera configuration information.
DBs selected : ['video_analytics', 'mes']
Queries gen  : 12
Status       : OK
[INFO] PDF report written to: C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\jupyter notebooks\mes_and_camera_report.pdf

✅ Done. Open the PDF: C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\jupyter notebooks\mes_and_camera_report.pdf


In [55]:
import json
import os
import re
import io
from typing import Any, Dict, List, Set, Optional, Tuple
from datetime import datetime, timedelta
from pathlib import Path

# -------------------------------------------------------------------
# Third-party libs
# -------------------------------------------------------------------
try:
    from dotenv import load_dotenv
except ImportError:
    raise ImportError("pip install python-dotenv")

try:
    import pyodbc
except ImportError:
    raise ImportError("pip install pyodbc")

try:
    import psycopg2
    import psycopg2.extras
except ImportError:
    raise ImportError("pip install psycopg2-binary")

try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4, landscape
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import mm
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
        PageBreak, HRFlowable, Image, KeepTogether
    )
    from reportlab.lib.enums import TA_CENTER
except ImportError:
    raise ImportError("pip install reportlab")

try:
    import matplotlib
    matplotlib.use("Agg")          # non-interactive backend
    import matplotlib.pyplot as plt
    import numpy as np
except ImportError:
    raise ImportError("pip install matplotlib numpy")

load_dotenv()

# -------------------------------------------------------------------
# 1. Schema Paths
# -------------------------------------------------------------------
SCHEMA_PATHS = {
    "video_analytics": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\video_analytics_db_info\construction_ai_schema.json",
    "mes": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.json",
}

# -------------------------------------------------------------------
# 2. Stop words
# -------------------------------------------------------------------
STOP_WORDS = {
    "a", "an", "the", "and", "or", "of", "in", "on", "for", "to", "from", "with",
    "is", "are", "was", "were", "be", "been", "being", "have", "has", "had",
    "do", "does", "did", "will", "would", "can", "could", "should", "may", "might",
    "i", "me", "my", "we", "you", "your", "he", "she", "it", "they", "them",
    "this", "that", "these", "those", "what", "which", "who", "whom", "whose",
    "all", "any", "some", "no", "not", "only", "just", "also", "very", "too",
    "want", "need", "show", "give", "get", "list", "data", "information", "details",
    "application", "system", "report", "me", "please"
}

# -------------------------------------------------------------------
# 3. Domain Vocabulary
# -------------------------------------------------------------------
DOMAIN_VOCABULARY = {
    "video_analytics": {
        "camera": ["cameras", "camera_status_logs", "basler_devices", "basler_model_assignments", "hse_camera_rules"],
        "cameras": ["cameras", "camera_status_logs", "basler_devices"],
        "config": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments"],
        "configuration": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments"],
        "rule": ["hse_camera_rules", "hse_rule_definitions", "hse_rule_events"],
        "hse": ["hse_camera_rules", "hse_rule_definitions", "hse_rule_events"],
        "alert": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "violation": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "safety": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "anomaly": ["anomaly_flags"],
        "incident": ["incidents"],
        "zone": ["zones", "zone_risk_scores"],
        "worker": ["attendances", "employees", "employee_movements"],
        "employee": ["employees", "attendances", "employee_movements"],
        "attendance": ["attendances"],
        "model": ["ai_models", "ai_model_classes", "basler_model_assignments"],
        "basler": ["basler_devices", "basler_model_assignments"],
    },
    "mes": {
        "machine": ["Machine", "MachineMaster", "MachineFGMapping", "MachineMaterialMapping", "CapacityAnalysis"],
        "machines": ["Machine", "MachineMaster"],
        "utilization": ["CapacityAnalysis", "MachineMaster"],
        "util": ["CapacityAnalysis", "MachineMaster"],
        "capacity": ["CapacityAnalysis", "MachineMaster", "ContainerCapacity"],
        "workorder": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog", "WorkOrderResource"],
        "work_order": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog"],
        "wo": ["WorkOrder", "WorkOrderStep"],
        "production": ["WorkOrder", "ProductionPlanning", "FinishedGood", "SemiFinishedGood"],
        "shift": ["ShiftMaster", "ShiftCalendar", "OperatorMaster"],
        "operator": ["OperatorMaster", "WorkOrderResource"],
        "inventory": ["Inventory", "InventoryByLot", "FinishedGood", "SemiFinishedGood"],
        "material": ["Materials", "RawMaterial", "BOMLine", "BOMMaster"],
        "bom": ["BOMMaster", "BOMLine"],
        "mps": ["MPSHeader", "MpsMaster"],
        "mrp": ["MRP_Run", "MRP_Demand", "MRP_Result"],
        "planning": ["MPSHeader", "MpsMaster", "ProductionPlanning", "WeeklyPlanning"],
        "maintenance": ["MaintenanceWindow"],
        "schedule": ["GanttSchedule", "WorkOrder"],
        "downtime": ["MaintenanceWindow", "CapacityAnalysis"],
    }
}

# -------------------------------------------------------------------
# 4. Schema Cleaner
# -------------------------------------------------------------------
NOISE_COLUMNS = {
    "nn", "pk", "id", "date", "on", "at", "_at", "created", "updated",
    "qty", "quantity", "count", "bags", "bag"
}

def clean_column_name(raw: str) -> List[str]:
    name = raw.strip().split(":")[0].strip()
    if "/" in name:
        parts = re.split(r"[/]", name)
        cleaned = []
        for p in parts:
            p = re.sub(r"[^a-zA-Z0-9_]", "", p)
            if (p and len(p) > 2 and not p.isdigit()
                    and p.lower() not in NOISE_COLUMNS and not p.startswith("_")):
                cleaned.append(p)
        return cleaned
    name = re.sub(r"[^a-zA-Z0-9_]", "", name)
    if (name and len(name) > 2 and not name.isdigit()
            and name.lower() not in NOISE_COLUMNS
            and not name.startswith("_") and not re.match(r"^\d", name)):
        return [name]
    return []


def parse_and_clean_schema(file_path: str) -> Dict[str, List[str]]:
    table_map = {}
    if not os.path.exists(file_path):
        print(f"[WARN] Schema file not found: {file_path}")
        return table_map
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for table_name, schema_str in data.items():
        clean_cols = []
        if isinstance(schema_str, str):
            for col_def in schema_str.split(","):
                clean_cols.extend(clean_column_name(col_def))
        elif isinstance(schema_str, list):
            clean_cols = [str(c) for c in schema_str if c]
        table_map[table_name] = list(dict.fromkeys(
            c for c in clean_cols
            if c and len(c) > 2 and not c.startswith("_")
            and c.lower() not in NOISE_COLUMNS and not re.match(r"^\d", c)
        ))
    return table_map


def build_clean_registry(paths: Dict[str, str]) -> Dict[str, Dict[str, List[str]]]:
    registry = {}
    for db, path in paths.items():
        registry[db] = parse_and_clean_schema(path)
        print(f"[INFO] Loaded {db}: {len(registry[db])} tables")
    return registry


JEV_CLEAN_REGISTRY = build_clean_registry(SCHEMA_PATHS)

# -------------------------------------------------------------------
# 5. Query Splitter
# -------------------------------------------------------------------
def extract_tokens(query: str) -> Set[str]:
    tokens = set(re.findall(r"[a-zA-Z0-9_]+", query.lower()))
    return {t for t in tokens if t not in STOP_WORDS and len(t) > 1}


def find_matched_keywords(tokens: Set[str], vocab: Dict[str, List[str]]) -> Set[str]:
    matched = set()
    for t in tokens:
        if t in vocab:
            matched.add(t)
            continue
        for key in vocab:
            if (t in key or key in t) and min(len(t), len(key)) >= 4:
                matched.add(key)
    return matched


def score_table(table_name: str, matched_keywords: Set[str], vocab: Dict[str, List[str]]) -> int:
    score = 0
    table_l = table_name.lower()
    for kw in matched_keywords:
        if kw in vocab and table_name in vocab[kw]:
            score += 10
        if kw in table_l:
            score += 6
    return score


def get_relevant_columns(cols: List[str], tokens: Set[str], matched: Set[str]) -> List[str]:
    scored = []
    for col in cols:
        col_l = col.lower()
        score = 0
        if col_l in tokens:
            score += 12
        for m in matched:
            if m in col_l or col_l in m:
                score += 5
        if any(x in col_l for x in ["id", "name", "code", "status", "qty", "quantity", "util", "capacity", "is_active", "active"]):
            score += 2
        scored.append((score, col))
    scored.sort(key=lambda x: (-x[0], x[1]))
    relevant = [c for s, c in scored if s >= 2]
    if len(relevant) < 3:
        relevant = cols[:8]
    return relevant[:8]


def customize_and_split_query(user_query: str) -> Dict[str, Any]:
    tokens = extract_tokens(user_query)
    result = {
        "raw_user_query": user_query,
        "selected_databases": [],
        "customized_db_requests": []
    }
    for db_name, tables in JEV_CLEAN_REGISTRY.items():
        vocab = DOMAIN_VOCABULARY.get(db_name, {})
        matched = find_matched_keywords(tokens, vocab)
        if not matched:
            continue
        candidates = []
        for table_name in tables:
            sc = score_table(table_name, matched, vocab)
            if sc >= 6:
                candidates.append((sc, table_name))
        for kw in matched:
            for t in vocab.get(kw, []):
                real = next((r for r in tables if r.lower() == t.lower()), None)
                if real and not any(real == c[1] for c in candidates):
                    candidates.append((10, real))
        if not candidates:
            continue
        candidates.sort(key=lambda x: -x[0])
        top_tables = [t for _, t in candidates[:10]]
        mappings = []
        for table_name in top_tables:
            cols = tables[table_name]
            primary = get_relevant_columns(cols, tokens, matched)
            mappings.append({
                "responsible_table": table_name,
                "primary_columns": primary,
                "all_table_columns": cols
            })
        keywords_str = ", ".join(sorted(matched))
        intent = (f"Retrieve camera / rules / alerts related to: {keywords_str}"
                  if db_name == "video_analytics"
                  else f"Retrieve machine / production / work-order data related to: {keywords_str}")
        result["selected_databases"].append(db_name)
        result["customized_db_requests"].append({
            "database": db_name,
            "customized_sub_query": intent,
            "token_keywords": sorted(list(matched)),
            "target_mappings": mappings
        })
    return result

# -------------------------------------------------------------------
# 6. Date Detection
# -------------------------------------------------------------------
REAL_DATE_HINTS = [
    "created_at", "updated_at", "checked_at", "triggered_at", "timestamp",
    "createdon", "updatedon", "created_date", "updated_date",
    "startdate", "enddate", "plannedstart", "actualstart", "date",
    "changedat", "lastupdated"
]
BAD_DATE_COLUMNS = {
    "updatedby", "createdby", "status", "type", "name", "code", "location",
    "rtsp_template", "utilizationpercent", "utilization"
}

def detect_date_column(columns: List[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in columns}
    for p in REAL_DATE_HINTS:
        if p in cols_lower:
            return cols_lower[p]
    for c in columns:
        cl = c.lower()
        if cl in BAD_DATE_COLUMNS:
            continue
        if any(h in cl for h in ["_at", "_on", "date", "time", "timestamp"]) and len(cl) > 4:
            return c
    return None


def extract_time_filter(user_query: str) -> Optional[str]:
    q = user_query.lower()
    today = datetime.now().date()
    if "last week" in q or "past week" in q:
        start = today - timedelta(days=7)
        return f"{{date_col}} >= '{start}'"
    if "today" in q:
        return f"{{date_col}} >= '{today}'"
    if "yesterday" in q:
        y = today - timedelta(days=1)
        return f"{{date_col}} >= '{y}' AND {{date_col}} < '{today}'"
    if "last month" in q or "past month" in q:
        start = today - timedelta(days=30)
        return f"{{date_col}} >= '{start}'"
    return None

# -------------------------------------------------------------------
# 7. SQL Generator
# -------------------------------------------------------------------
def generate_sql_for_table(
    table_name: str,
    primary_columns: List[str],
    all_columns: List[str],
    time_filter: Optional[str] = None,
    limit: int = 50
) -> str:
    cols = primary_columns if primary_columns else all_columns[:8]
    if not cols:
        cols = ["*"]
    for possible_id in ["id", "Id", "ID", f"{table_name}Id"]:
        if possible_id in all_columns and possible_id not in cols:
            cols.insert(0, possible_id)
            break
    col_list = ", ".join(f'"{c}"' for c in cols)
    sql = f'SELECT {col_list}\nFROM "{table_name}"'
    date_col = detect_date_column(all_columns)
    if time_filter and date_col:
        real_filter = time_filter.replace("{date_col}", f'"{date_col}"')
        sql += f"\nWHERE {real_filter}"
    if date_col:
        sql += f'\nORDER BY "{date_col}" DESC'
    sql += f"\nLIMIT {limit};"
    return sql


def generate_sql_queries(pipeline_output: Dict[str, Any], limit_per_table: int = 50) -> Dict[str, Any]:
    time_filter = extract_time_filter(pipeline_output.get("raw_user_query", ""))
    sql_result = {
        "raw_user_query": pipeline_output.get("raw_user_query"),
        "time_filter_applied": time_filter,
        "databases": []
    }
    for req in pipeline_output.get("customized_db_requests", []):
        db_entry = {
            "database": req["database"],
            "intent": req["customized_sub_query"],
            "keywords": req["token_keywords"],
            "queries": []
        }
        for mapping in req["target_mappings"]:
            table = mapping["responsible_table"]
            primary = mapping["primary_columns"]
            all_cols = mapping["all_table_columns"]
            sql = generate_sql_for_table(table, primary, all_cols, time_filter, limit=limit_per_table)
            db_entry["queries"].append({
                "table": table,
                "sql": sql,
                "columns_used": primary if primary else all_cols[:8],
                "date_column_used": detect_date_column(all_cols),
                "explanation": f"Fetch relevant columns from {table}"
            })
        sql_result["databases"].append(db_entry)
    return sql_result

# -------------------------------------------------------------------
# 8. Validator
# -------------------------------------------------------------------
def validate_sql(sql: str, table: str, columns_used: List[str], date_col: Optional[str]) -> List[str]:
    issues = []
    if date_col and "ORDER BY" not in sql:
        issues.append(f"Has date column '{date_col}' but no ORDER BY")
    if len(columns_used) < 2:
        issues.append("Too few columns selected")
    if "*" in sql:
        issues.append("Using SELECT *")
    for bad in ["_at", "4NN", "2NN", "DateOn"]:
        if f'"{bad}"' in sql or f".{bad}" in sql:
            issues.append(f"Noise column still present: {bad}")
    return issues


def analyze_pipeline_result(sql_result: Dict[str, Any]) -> Dict[str, Any]:
    report = {
        "query": sql_result["raw_user_query"],
        "databases_selected": [d["database"] for d in sql_result["databases"]],
        "total_queries": 0,
        "issues_found": [],
        "status": "OK"
    }
    for db in sql_result["databases"]:
        for q in db["queries"]:
            report["total_queries"] += 1
            issues = validate_sql(q["sql"], q["table"], q["columns_used"], q.get("date_column_used"))
            for iss in issues:
                report["issues_found"].append(f"[{db['database']}.{q['table']}] {iss}")
    if report["issues_found"]:
        report["status"] = "ISSUES_DETECTED"
    return report

# -------------------------------------------------------------------
# 9. Database connections & execution
# -------------------------------------------------------------------
def get_mes_connection():
    driver = os.getenv("DB_DRIVER", "ODBC Driver 18 for SQL Server")
    server = os.getenv("DB_SERVER", "localhost,1433")
    database = os.getenv("DB_NAME", "mes_new")
    trusted = os.getenv("DB_TRUSTED_CONNECTION", "yes").lower() == "yes"
    encrypt = os.getenv("DB_ENCRYPT", "no")
    trust_cert = os.getenv("DB_TRUST_SERVER_CERTIFICATE", "yes")
    conn_str = (
        f"DRIVER={{{driver}}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Trusted_Connection={'yes' if trusted else 'no'};"
        f"Encrypt={encrypt};"
        f"TrustServerCertificate={trust_cert};"
    )
    return pyodbc.connect(conn_str, timeout=15)


def get_video_analytics_connection():
    url = os.getenv("CONSTRUCTION_DB_URL")
    if url and url.startswith("postgresql"):
        url = url.replace("postgresql+psycopg2://", "postgresql://")
        return psycopg2.connect(url)
    return psycopg2.connect(
        host=os.getenv("PG_HOST", "localhost"),
        port=os.getenv("PG_PORT", "5432"),
        dbname=os.getenv("PG_DB", "construction_ai"),
        user=os.getenv("PG_USER", "postgres"),
        password=os.getenv("PG_PASSWORD", "0987654321")
    )


def execute_sql(db_name: str, sql: str) -> Tuple[List[str], List[Tuple]]:
    if db_name == "mes":
        sql = re.sub(r'"([^"]+)"', r'[\1]', sql)
        m = re.search(r"LIMIT\s+(\d+)", sql, re.IGNORECASE)
        if m:
            limit = m.group(1)
            sql = re.sub(r"LIMIT\s+\d+", "", sql, flags=re.IGNORECASE)
            sql = re.sub(r"(SELECT\s+)", rf"\1TOP {limit} ", sql, count=1, flags=re.IGNORECASE)
    try:
        if db_name == "mes":
            conn = get_mes_connection()
            cursor = conn.cursor()
            cursor.execute(sql)
            columns = [col[0] for col in cursor.description] if cursor.description else []
            rows = cursor.fetchall()
            cursor.close()
            conn.close()
            return columns, [tuple(r) for r in rows]
        else:
            conn = get_video_analytics_connection()
            cursor = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)
            cursor.execute(sql)
            columns = [desc[0] for desc in cursor.description] if cursor.description else []
            rows = cursor.fetchall()
            cursor.close()
            conn.close()
            return columns, [tuple(r) for r in rows]
    except Exception as e:
        print(f"[ERROR] Failed to execute on {db_name}: {e}")
        print(f"SQL was:\n{sql}")
        return [], []


def run_all_queries(sql_result: Dict[str, Any]) -> Dict[str, Any]:
    execution_result = {
        "raw_user_query": sql_result["raw_user_query"],
        "time_filter_applied": sql_result.get("time_filter_applied"),
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "databases": []
    }
    for db_entry in sql_result.get("databases", []):
        db_name = db_entry["database"]
        new_db = {
            "database": db_name,
            "intent": db_entry["intent"],
            "keywords": db_entry["keywords"],
            "queries": []
        }
        for q in db_entry["queries"]:
            cols, rows = execute_sql(db_name, q["sql"])
            new_db["queries"].append({
                "table": q["table"],
                "sql": q["sql"],
                "columns": cols,
                "rows": rows,
                "row_count": len(rows),
                "explanation": q["explanation"]
            })
        execution_result["databases"].append(new_db)
    return execution_result

# -------------------------------------------------------------------
# 10. Rich Metadata
# -------------------------------------------------------------------
def _detect_type(values: list) -> str:
    non_null = [v for v in values if v is not None]
    if not non_null:
        return "unknown"
    sample = non_null[0]
    if isinstance(sample, bool):
        return "boolean"
    if isinstance(sample, int) and not isinstance(sample, bool):
        return "integer"
    if isinstance(sample, float):
        return "numeric"
    try:
        float(sample)
        return "numeric"
    except (TypeError, ValueError):
        pass
    return "string"


def _build_rich_metadata(table_name: str, columns: List[str], rows: List[Tuple], max_sample: int = 5) -> Dict[str, Any]:
    if not rows or not columns:
        return {
            "table_name": table_name,
            "total_records": 0,
            "total_columns": len(columns),
            "columns": columns,
            "column_details": {},
            "data_quality": {}
        }

    total_records = len(rows)
    col_details = {}
    zero_values = {}
    potential_outliers = []

    for col_idx, col_name in enumerate(columns):
        col_values = [row[col_idx] for row in rows]
        non_null = [v for v in col_values if v is not None]
        null_count = total_records - len(non_null)
        unique_count = len(set(str(v) for v in non_null))

        dtype = _detect_type(col_values)
        detail = {
            "data_type": dtype,
            "null_count": null_count,
            "unique_count": unique_count,
            "sample_values": []
        }

        samples = []
        seen = set()
        for v in non_null:
            s = str(v)
            if s not in seen:
                samples.append(v)
                seen.add(s)
            if len(samples) >= max_sample:
                break
        detail["sample_values"] = samples

        if dtype in ("numeric", "integer"):
            nums = []
            for v in non_null:
                try:
                    nums.append(float(v))
                except (TypeError, ValueError):
                    continue
            if nums:
                detail["min"] = round(min(nums), 2)
                detail["max"] = round(max(nums), 2)
                detail["avg"] = round(sum(nums) / len(nums), 2)

                zero_cnt = sum(1 for n in nums if n == 0)
                if zero_cnt:
                    zero_values[col_name] = zero_cnt

                if len(nums) > 5:
                    avg = detail["avg"]
                    for n in nums:
                        if avg > 0 and n > avg * 10:
                            potential_outliers.append({
                                "column": col_name,
                                "value": n,
                                "issue": f"Unusually high value (>{avg * 10:.1f})"
                            })
                            break

        col_details[col_name] = detail

    return {
        "table_name": table_name,
        "total_records": total_records,
        "total_columns": len(columns),
        "columns": columns,
        "column_details": col_details,
        "data_quality": {
            "potential_outliers": potential_outliers,
            "zero_values": zero_values
        }
    }


def _truncate(val: Any, max_len: int = 40) -> str:
    s = str(val) if val is not None else ""
    return s if len(s) <= max_len else s[:max_len - 1] + "…"


def _render_metadata_block(meta: Dict[str, Any], styles) -> list:
    elements = []
    header = (
        f"<b>Table:</b> {meta['table_name']} &nbsp;&nbsp;|&nbsp;&nbsp; "
        f"<b>Records:</b> {meta['total_records']} &nbsp;&nbsp;|&nbsp;&nbsp; "
        f"<b>Columns:</b> {meta['total_columns']}"
    )
    elements.append(Paragraph(header, styles["MetadataBox"]))

    col_data = [["Column", "Type", "Min", "Max", "Avg", "Nulls", "Unique", "Sample"]]
    for col, d in meta["column_details"].items():
        col_data.append([
            Paragraph(f"<b>{col}</b>", styles["Meta"]),
            d.get("data_type", ""),
            str(d.get("min", "–")),
            str(d.get("max", "–")),
            str(d.get("avg", "–")),
            str(d.get("null_count", 0)),
            str(d.get("unique_count", 0)),
            ", ".join(str(s) for s in d.get("sample_values", [])[:3])
        ])

    t = Table(col_data, colWidths=[90, 55, 50, 50, 50, 40, 45, 150])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2874a6")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 7),
        ("ALIGN", (2, 1), (6, -1), "CENTER"),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#aab7b8")),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#eaf2f8")]),
        ("LEFTPADDING", (0, 0), (-1, -1), 3),
        ("RIGHTPADDING", (0, 0), (-1, -1), 3),
        ("TOPPADDING", (0, 0), (-1, -1), 2),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]))
    elements.append(t)
    elements.append(Spacer(1, 3 * mm))

    dq = meta.get("data_quality", {})
    notes = []
    if dq.get("potential_outliers"):
        for o in dq["potential_outliers"][:3]:
            notes.append(f"• {o['column']}: {o['value']} – {o['issue']}")
    if dq.get("zero_values"):
        zeros = ", ".join(f"{k} ({v})" for k, v in dq["zero_values"].items())
        notes.append(f"• Zero values found in: {zeros}")

    if notes:
        elements.append(Paragraph("<b>Data Quality Notes:</b>", styles["Meta"]))
        for n in notes:
            elements.append(Paragraph(n, styles["Meta"]))
        elements.append(Spacer(1, 2 * mm))

    return elements

# -------------------------------------------------------------------
# 11. Chart Generation (matplotlib → PNG → ReportLab Image)
# -------------------------------------------------------------------
def _is_numeric_col(col_values: list) -> bool:
    non_null = [v for v in col_values if v is not None]
    if not non_null:
        return False
    try:
        float(non_null[0])
        return True
    except (TypeError, ValueError):
        return False


def _choose_and_create_charts(
    table_name: str,
    columns: List[str],
    rows: List[Tuple],
    max_charts: int = 2
) -> List[io.BytesIO]:
    """
    Intelligently choose the best chart types and return list of PNG buffers.
    Supported useful types: Bar, Column, Line, Pie, Area, Histogram, Scatter, Pareto
    """
    if not rows or not columns:
        return []

    charts = []
    col_data = {col: [row[i] for row in rows] for i, col in enumerate(columns)}

    numeric_cols = [c for c in columns if _is_numeric_col(col_data[c])]
    categorical_cols = [c for c in columns if c not in numeric_cols]

    # Helper to save current figure to buffer
    def save_fig():
        buf = io.BytesIO()
        plt.tight_layout()
        plt.savefig(buf, format="png", dpi=120, bbox_inches="tight")
        buf.seek(0)
        plt.close()
        return buf

    # ---------- 1. Histogram for first numeric column ----------
    if numeric_cols and len(charts) < max_charts:
        col = numeric_cols[0]
        vals = [float(v) for v in col_data[col] if v is not None]
        if len(vals) >= 3:
            plt.figure(figsize=(7, 3.8))
            plt.hist(vals, bins=min(15, max(5, len(vals)//3)), color="#2874a6", edgecolor="white")
            plt.title(f"Histogram – {col} ({table_name})")
            plt.xlabel(col)
            plt.ylabel("Frequency")
            charts.append(save_fig())

    # ---------- 2. Modern & Clean Horizontal Bar Chart ----------
    if categorical_cols and numeric_cols and len(charts) < max_charts:
        cat_col = categorical_cols[0]
        num_col = numeric_cols[0]

        # Aggregate numerical values by categorical key
        from collections import defaultdict

        agg = defaultdict(float)
        cnt = defaultdict(int)
        for cat, num in zip(col_data[cat_col], col_data[num_col]):
            if cat is not None and num is not None:
                try:
                    agg[str(cat)] += float(num)
                    cnt[str(cat)] += 1
                except Exception:
                    pass

        if agg:
            # Take top 12 categories, sorted ascending so the largest renders at the top
            items = sorted(agg.items(), key=lambda x: x[1])[-12:]
            labels = [k[:20] for k, _ in items]
            values = [v for _, v in items]

            # Figure setup with modern styling
            fig, ax = plt.subplots(figsize=(8, 4.5))

            # Modern horizontal bars (Clean Dark Teal)
            bars = ax.barh(labels, values, color="#1D5375", height=0.65, zorder=3)

            # Value annotations on the right of each bar
            max_val = max(values) if values else 1
            for bar in bars:
                width = bar.get_width()
                ax.annotate(
                    f"{width:,.1f}",
                    xy=(width, bar.get_y() + bar.get_height() / 2),
                    xytext=(6, 0),
                    textcoords="offset points",
                    ha="left",
                    va="center",
                    fontsize=9,
                    color="#555555",
                )

            # Title and Labels formatting
            ax.set_title(
                f"{num_col.title()} by {cat_col.title()}",
                fontsize=13,
                pad=15,
                weight="bold",
                color="#1D5375",
            )
            ax.set_xlabel(num_col.title(), fontsize=10, labelpad=8, color="#333333")

            # Styling gridlines and background
            ax.set_axisbelow(True)
            ax.xaxis.grid(True, linestyle="-", alpha=0.3, color="#CCCCCC", zorder=0)
            ax.yaxis.grid(False)

            # Expand x-limit slightly to avoid text overlap with chart boundary
            ax.set_xlim(0, max_val * 1.12)

            # Spines / border cleanup
            for spine in ["top", "right", "left", "bottom"]:
                ax.spines[spine].set_color("#E0E0E0")

            plt.tight_layout()
            charts.append(save_fig())

    # ---------- 3. Clean & Modern Donut/Pie Chart ----------
    if categorical_cols and len(charts) < max_charts:
        cat_col = categorical_cols[0]
        from collections import Counter

        counts = Counter(str(v) for v in col_data[cat_col] if v is not None)

        if 2 <= len(counts) <= 8:
            # Sort values so largest slices render cleanly first
            sorted_counts = dict(
                sorted(counts.items(), key=lambda item: item[1], reverse=True)
            )
            labels = list(sorted_counts.keys())
            sizes = list(sorted_counts.values())

            # Set clean visual styling & proper aspect ratio
            fig, ax = plt.subplots(figsize=(8, 5.5), subplot_kw=dict(aspect="equal"))
            
            # Professional color palette (Seaborn dark palette / modern pastel)
            colors = plt.cm.tab10.colors

            # Create Donut Chart with percentage labels inside slices
            wedges, texts, autotexts = ax.pie(
                sizes,
                autopct="%1.1f%%",
                pctdistance=0.75,
                startangle=140,
                colors=colors[: len(labels)],
                wedgeprops=dict(width=0.4, edgecolor="white", linewidth=2),
            )

            # Style percentage text inside slices
            plt.setp(autotexts, size=9, weight="bold", color="white")

            # Move category labels outside into a clean Legend on the side
            ax.legend(
                wedges,
                labels,
                title=f"{cat_col.title()}",
                loc="center left",
                bbox_to_anchor=(1, 0, 0.5, 1),
                frameon=False,
                fontsize=10,
                title_fontsize=11,
            )

            # Title formatting
            ax.set_title(
                f"Distribution of {cat_col.title()}",
                fontsize=14,
                pad=20,
                weight="bold",
                color="#2C3E50",
            )

            plt.tight_layout()
            charts.append(save_fig())

    # ---------- 4. Line / Area (if we have a reasonable x-axis) ----------
    if len(numeric_cols) >= 1 and len(charts) < max_charts:
        # use index as x if no date
        y_col = numeric_cols[0]
        y_vals = [float(v) if v is not None else np.nan for v in col_data[y_col]]
        if len(y_vals) >= 4:
            plt.figure(figsize=(8, 3.8))
            plt.plot(range(len(y_vals)), y_vals, marker="o", linewidth=2, color="#2874a6")
            plt.fill_between(range(len(y_vals)), y_vals, alpha=0.25, color="#2874a6")
            plt.title(f"Line / Area Chart – {y_col}")
            plt.xlabel("Record Index")
            plt.ylabel(y_col)
            charts.append(save_fig())

    # ---------- 5. Scatter (two numeric columns) ----------
    if len(numeric_cols) >= 2 and len(charts) < max_charts:
        x_col, y_col = numeric_cols[0], numeric_cols[1]
        xs, ys = [], []
        for a, b in zip(col_data[x_col], col_data[y_col]):
            try:
                if a is not None and b is not None:
                    xs.append(float(a))
                    ys.append(float(b))
            except:
                pass
        if len(xs) >= 5:
            plt.figure(figsize=(6.5, 4))
            plt.scatter(xs, ys, alpha=0.7, c="#1a5276", edgecolors="white")
            plt.title(f"Scatter Plot – {y_col} vs {x_col}")
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            charts.append(save_fig())

    # ---------- 6. Pareto (sorted bar + cumulative) ----------
    if categorical_cols and numeric_cols and len(charts) < max_charts:
        cat_col = categorical_cols[0]
        num_col = numeric_cols[0]
        from collections import defaultdict
        agg = defaultdict(float)
        for cat, num in zip(col_data[cat_col], col_data[num_col]):
            if cat is not None and num is not None:
                try:
                    agg[str(cat)] += float(num)
                except:
                    pass
        if len(agg) >= 3:
            items = sorted(agg.items(), key=lambda x: -x[1])[:10]
            labels = [k[:15] for k, _ in items]
            values = [v for _, v in items]
            total = sum(values) or 1
            cum = np.cumsum(values) / total * 100

            fig, ax1 = plt.subplots(figsize=(8, 4))
            ax1.bar(labels, values, color="#2874a6")
            ax1.set_ylabel(num_col)
            ax2 = ax1.twinx()
            ax2.plot(labels, cum, color="#e74c3c", marker="D", linewidth=2)
            ax2.set_ylabel("Cumulative %")
            ax2.set_ylim(0, 105)
            plt.title(f"Pareto Chart – {num_col} by {cat_col}")
            plt.xticks(rotation=45, ha="right", fontsize=8)
            charts.append(save_fig())

    return charts[:max_charts]

# -------------------------------------------------------------------
# 12. PDF Generator (final rules)
# -------------------------------------------------------------------
def generate_pdf_report(
    execution_result: Dict[str, Any],
    output_path: str = None,
    metadata_threshold: int = 5
) -> str:
    """
    Final rules:
      - 0 rows          → completely skip the table
      - 1–5 rows        → show data table + chart(s)
      - > 5 rows        → show only rich Metadata + chart(s)
    """
    if output_path is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"query_report_{ts}.pdf"

    output_path = str(Path(output_path).resolve())

    doc = SimpleDocTemplate(
        output_path,
        pagesize=landscape(A4),
        leftMargin=12 * mm,
        rightMargin=12 * mm,
        topMargin=12 * mm,
        bottomMargin=12 * mm
    )

    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="ReportTitle", parent=styles["Heading1"],
                               fontSize=16, alignment=TA_CENTER, spaceAfter=6))
    styles.add(ParagraphStyle(name="SectionHeader", parent=styles["Heading2"],
                               fontSize=12, textColor=colors.HexColor("#1a5276"),
                               spaceBefore=10, spaceAfter=4))
    styles.add(ParagraphStyle(name="SubHeader", parent=styles["Heading3"],
                               fontSize=10, textColor=colors.HexColor("#2874a6"),
                               spaceBefore=6, spaceAfter=3))
    styles.add(ParagraphStyle(name="Meta", parent=styles["Normal"],
                               fontSize=8, textColor=colors.grey))
    styles.add(ParagraphStyle(name="MetadataBox", parent=styles["Normal"],
                               fontSize=8, leading=11,
                               backColor=colors.HexColor("#eaf2f8"),
                               borderPadding=6, spaceBefore=2, spaceAfter=4))

    story = []

    # Title
    story.append(Paragraph("Manufacturing & Video-Analytics Query Report", styles["ReportTitle"]))
    story.append(Spacer(1, 4 * mm))
    story.append(Paragraph(f"<b>User Query:</b> {execution_result['raw_user_query']}", styles["Normal"]))
    story.append(Paragraph(f"<b>Generated at:</b> {execution_result['generated_at']}", styles["Meta"]))
    if execution_result.get("time_filter_applied"):
        story.append(Paragraph(f"<b>Time filter:</b> {execution_result['time_filter_applied']}", styles["Meta"]))
    story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor("#1a5276")))
    story.append(Spacer(1, 6 * mm))

    total_tables = 0
    total_rows = 0

    for db in execution_result["databases"]:
        story.append(Paragraph(f"Database: {db['database'].upper()}", styles["SectionHeader"]))
        story.append(Paragraph(f"Intent: {db['intent']}", styles["Meta"]))
        story.append(Paragraph(f"Keywords: {', '.join(db['keywords'])}", styles["Meta"]))
        story.append(Spacer(1, 3 * mm))

        for q in db["queries"]:
            row_count = q["row_count"]

            # ---------- RULE 1: skip empty tables completely ----------
            if row_count == 0:
                continue

            total_tables += 1
            total_rows += row_count

            header = f"Table: {q['table']}  ({row_count} rows)"
            story.append(Paragraph(header, styles["SubHeader"]))

            # ---------- RULE 2 & 3 ----------
            if row_count > metadata_threshold:
                # Only Metadata
                meta = _build_rich_metadata(q["table"], q["columns"], q["rows"])
                for flowable in _render_metadata_block(meta, styles):
                    story.append(flowable)
            else:
                # Small data table
                col_names = q["columns"][:10]
                data = [[Paragraph(f"<b>{c}</b>", styles["Meta"]) for c in col_names]]
                for row in q["rows"]:
                    data.append([Paragraph(_truncate(cell), styles["Meta"]) for cell in row[:10]])

                col_widths = [doc.width / len(col_names)] * len(col_names)
                t = Table(data, colWidths=col_widths, repeatRows=1)
                t.setStyle(TableStyle([
                    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1a5276")),
                    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
                    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                    ("FONTSIZE", (0, 0), (-1, -1), 7),
                    ("ALIGN", (0, 0), (-1, -1), "LEFT"),
                    ("VALIGN", (0, 0), (-1, -1), "TOP"),
                    ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#aab7b8")),
                    ("ROWBACKGROUNDS", (0, 1), (-1, -1),
                     [colors.white, colors.HexColor("#eaf2f8")]),
                    ("LEFTPADDING", (0, 0), (-1, -1), 3),
                    ("RIGHTPADDING", (0, 0), (-1, -1), 3),
                    ("TOPPADDING", (0, 0), (-1, -1), 2),
                    ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
                ]))
                story.append(t)

            # ---------- Charts for every non-empty table ----------
            chart_buffers = _choose_and_create_charts(q["table"], q["columns"], q["rows"])
            if chart_buffers:
                story.append(Spacer(1, 3 * mm))
                story.append(Paragraph("<b>Visualizations</b>", styles["Meta"]))
                for buf in chart_buffers:
                    img = Image(buf, width=160*mm, height=85*mm)
                    story.append(img)
                    story.append(Spacer(1, 3 * mm))
            else:
                story.append(Paragraph("<i>No suitable chart could be generated for this data.</i>", styles["Meta"]))

            story.append(Spacer(1, 6 * mm))

        story.append(PageBreak())

    # Overall summary
    story.append(Paragraph("Overall Summary", styles["SectionHeader"]))
    story.append(Paragraph(
        f"Databases queried: {len(execution_result['databases'])} &nbsp;&nbsp;|&nbsp;&nbsp; "
        f"Tables shown: {total_tables} &nbsp;&nbsp;|&nbsp;&nbsp; "
        f"Total rows: {total_rows}",
        styles["Normal"]
    ))
    story.append(Paragraph(
        f"<i>Empty tables (0 rows) are hidden. "
        f"Tables with &gt; {metadata_threshold} rows show only Metadata + charts. "
        f"Tables with 1–{metadata_threshold} rows show data + charts.</i>",
        styles["Meta"]
    ))

    doc.build(story)
    print(f"[INFO] PDF report written to: {output_path}")
    return output_path

# -------------------------------------------------------------------
# 13. High-level entry point
# -------------------------------------------------------------------
def process_query_and_generate_pdf(user_query: str, pdf_path: str = None) -> str:
    print(f"\n>>> Processing: {user_query}")
    split = customize_and_split_query(user_query)
    if not split["selected_databases"]:
        print("[WARN] No databases matched the query.")
    sql_result = generate_sql_queries(split, limit_per_table=50)
    report = analyze_pipeline_result(sql_result)
    print(f"DBs selected : {report['databases_selected']}")
    print(f"Queries gen  : {report['total_queries']}")
    print(f"Status       : {report['status']}")

    execution = run_all_queries(sql_result)
    pdf_file = generate_pdf_report(execution, output_path=pdf_path)
    return pdf_file

# -------------------------------------------------------------------
# 14. Main
# -------------------------------------------------------------------
if __name__ == "__main__":
    user_q = (
        "I want the data from the mes application for machine utilization "
        "and from video analytics I want camera configuration information."
    )
    pdf_file = process_query_and_generate_pdf(user_q, pdf_path="mes_and_camera_report.pdf")
    print(f"\n✅ Done. Open the PDF: {pdf_file}")

[INFO] Loaded video_analytics: 52 tables
[INFO] Loaded mes: 108 tables

>>> Processing: I want the data from the mes application for machine utilization and from video analytics I want camera configuration information.
DBs selected : ['video_analytics', 'mes']
Queries gen  : 12
Status       : OK


10:21:27 [INFO] matplotlib.category: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
10:21:27 [INFO] matplotlib.category: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


[INFO] PDF report written to: C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\jupyter notebooks\mes_and_camera_report.pdf

✅ Done. Open the PDF: C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\jupyter notebooks\mes_and_camera_report.pdf


In [48]:
import json
import os
import re
import io
from typing import Any, Dict, List, Set, Optional, Tuple
from datetime import datetime, timedelta
from pathlib import Path
from collections import defaultdict, Counter

# -------------------------------------------------------------------
# Third-party libs
# -------------------------------------------------------------------
try:
    from dotenv import load_dotenv
except ImportError:
    raise ImportError("pip install python-dotenv")

try:
    import pyodbc
except ImportError:
    raise ImportError("pip install pyodbc")

try:
    import psycopg2
    import psycopg2.extras
except ImportError:
    raise ImportError("pip install psycopg2-binary")

try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4, landscape
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import mm
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
        PageBreak, HRFlowable, Image
    )
    from reportlab.lib.enums import TA_CENTER, TA_LEFT
except ImportError:
    raise ImportError("pip install reportlab")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np
except ImportError:
    raise ImportError("pip install matplotlib numpy")

try:
    from litellm import completion
except ImportError:
    raise ImportError("pip install litellm")

load_dotenv()

# Make sure LiteLLM sees the keys
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY", "")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

# -------------------------------------------------------------------
# 1. Schema Paths
# -------------------------------------------------------------------
SCHEMA_PATHS = {
    "video_analytics": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\video_analytics_db_info\construction_ai_schema.json",
    "mes": r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.json",
}

# -------------------------------------------------------------------
# 2. Stop words
# -------------------------------------------------------------------
STOP_WORDS = {
    "a", "an", "the", "and", "or", "of", "in", "on", "for", "to", "from", "with",
    "is", "are", "was", "were", "be", "been", "being", "have", "has", "had",
    "do", "does", "did", "will", "would", "can", "could", "should", "may", "might",
    "i", "me", "my", "we", "you", "your", "he", "she", "it", "they", "them",
    "this", "that", "these", "those", "what", "which", "who", "whom", "whose",
    "all", "any", "some", "no", "not", "only", "just", "also", "very", "too",
    "want", "need", "show", "give", "get", "list", "data", "information", "details",
    "application", "system", "report", "me", "please"
}

# -------------------------------------------------------------------
# 3. Domain Vocabulary
# -------------------------------------------------------------------
DOMAIN_VOCABULARY = {
    "video_analytics": {
        "camera": ["cameras", "camera_status_logs", "basler_devices", "basler_model_assignments", "hse_camera_rules"],
        "cameras": ["cameras", "camera_status_logs", "basler_devices"],
        "config": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments"],
        "configuration": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments"],
        "rule": ["hse_camera_rules", "hse_rule_definitions", "hse_rule_events"],
        "hse": ["hse_camera_rules", "hse_rule_definitions", "hse_rule_events"],
        "alert": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "violation": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "safety": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "anomaly": ["anomaly_flags"],
        "incident": ["incidents"],
        "zone": ["zones", "zone_risk_scores"],
        "worker": ["attendances", "employees", "employee_movements"],
        "employee": ["employees", "attendances", "employee_movements"],
        "attendance": ["attendances"],
        "model": ["ai_models", "ai_model_classes", "basler_model_assignments"],
        "basler": ["basler_devices", "basler_model_assignments"],
    },
    "mes": {
        "machine": ["Machine", "MachineMaster", "MachineFGMapping", "MachineMaterialMapping", "CapacityAnalysis"],
        "machines": ["Machine", "MachineMaster"],
        "utilization": ["CapacityAnalysis", "MachineMaster"],
        "util": ["CapacityAnalysis", "MachineMaster"],
        "capacity": ["CapacityAnalysis", "MachineMaster", "ContainerCapacity"],
        "workorder": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog", "WorkOrderResource"],
        "work_order": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog"],
        "wo": ["WorkOrder", "WorkOrderStep"],
        "production": ["WorkOrder", "ProductionPlanning", "FinishedGood", "SemiFinishedGood"],
        "shift": ["ShiftMaster", "ShiftCalendar", "OperatorMaster"],
        "operator": ["OperatorMaster", "WorkOrderResource"],
        "inventory": ["Inventory", "InventoryByLot", "FinishedGood", "SemiFinishedGood"],
        "material": ["Materials", "RawMaterial", "BOMLine", "BOMMaster"],
        "bom": ["BOMMaster", "BOMLine"],
        "mps": ["MPSHeader", "MpsMaster"],
        "mrp": ["MRP_Run", "MRP_Demand", "MRP_Result"],
        "planning": ["MPSHeader", "MpsMaster", "ProductionPlanning", "WeeklyPlanning"],
        "maintenance": ["MaintenanceWindow"],
        "schedule": ["GanttSchedule", "WorkOrder"],
        "downtime": ["MaintenanceWindow", "CapacityAnalysis"],
    }
}

# -------------------------------------------------------------------
# 4. Schema Cleaner
# -------------------------------------------------------------------
NOISE_COLUMNS = {
    "nn", "pk", "id", "date", "on", "at", "_at", "created", "updated",
    "qty", "quantity", "count", "bags", "bag"
}

def clean_column_name(raw: str) -> List[str]:
    name = raw.strip().split(":")[0].strip()
    if "/" in name:
        parts = re.split(r"[/]", name)
        cleaned = []
        for p in parts:
            p = re.sub(r"[^a-zA-Z0-9_]", "", p)
            if (p and len(p) > 2 and not p.isdigit()
                    and p.lower() not in NOISE_COLUMNS and not p.startswith("_")):
                cleaned.append(p)
        return cleaned
    name = re.sub(r"[^a-zA-Z0-9_]", "", name)
    if (name and len(name) > 2 and not name.isdigit()
            and name.lower() not in NOISE_COLUMNS
            and not name.startswith("_") and not re.match(r"^\d", name)):
        return [name]
    return []


def parse_and_clean_schema(file_path: str) -> Dict[str, List[str]]:
    table_map = {}
    if not os.path.exists(file_path):
        print(f"[WARN] Schema file not found: {file_path}")
        return table_map
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for table_name, schema_str in data.items():
        clean_cols = []
        if isinstance(schema_str, str):
            for col_def in schema_str.split(","):
                clean_cols.extend(clean_column_name(col_def))
        elif isinstance(schema_str, list):
            clean_cols = [str(c) for c in schema_str if c]
        table_map[table_name] = list(dict.fromkeys(
            c for c in clean_cols
            if c and len(c) > 2 and not c.startswith("_")
            and c.lower() not in NOISE_COLUMNS and not re.match(r"^\d", c)
        ))
    return table_map


def build_clean_registry(paths: Dict[str, str]) -> Dict[str, Dict[str, List[str]]]:
    registry = {}
    for db, path in paths.items():
        registry[db] = parse_and_clean_schema(path)
        print(f"[INFO] Loaded {db}: {len(registry[db])} tables")
    return registry


JEV_CLEAN_REGISTRY = build_clean_registry(SCHEMA_PATHS)

# -------------------------------------------------------------------
# 5. Query Splitter
# -------------------------------------------------------------------
def extract_tokens(query: str) -> Set[str]:
    tokens = set(re.findall(r"[a-zA-Z0-9_]+", query.lower()))
    return {t for t in tokens if t not in STOP_WORDS and len(t) > 1}


def find_matched_keywords(tokens: Set[str], vocab: Dict[str, List[str]]) -> Set[str]:
    matched = set()
    for t in tokens:
        if t in vocab:
            matched.add(t)
            continue
        for key in vocab:
            if (t in key or key in t) and min(len(t), len(key)) >= 4:
                matched.add(key)
    return matched


def score_table(table_name: str, matched_keywords: Set[str], vocab: Dict[str, List[str]]) -> int:
    score = 0
    table_l = table_name.lower()
    for kw in matched_keywords:
        if kw in vocab and table_name in vocab[kw]:
            score += 10
        if kw in table_l:
            score += 6
    return score


def get_relevant_columns(cols: List[str], tokens: Set[str], matched: Set[str]) -> List[str]:
    scored = []
    for col in cols:
        col_l = col.lower()
        score = 0
        if col_l in tokens:
            score += 12
        for m in matched:
            if m in col_l or col_l in m:
                score += 5
        if any(x in col_l for x in ["id", "name", "code", "status", "qty", "quantity", "util", "capacity", "is_active", "active"]):
            score += 2
        scored.append((score, col))
    scored.sort(key=lambda x: (-x[0], x[1]))
    relevant = [c for s, c in scored if s >= 2]
    if len(relevant) < 3:
        relevant = cols[:8]
    return relevant[:8]


def customize_and_split_query(user_query: str) -> Dict[str, Any]:
    tokens = extract_tokens(user_query)
    result = {
        "raw_user_query": user_query,
        "selected_databases": [],
        "customized_db_requests": []
    }
    for db_name, tables in JEV_CLEAN_REGISTRY.items():
        vocab = DOMAIN_VOCABULARY.get(db_name, {})
        matched = find_matched_keywords(tokens, vocab)
        if not matched:
            continue
        candidates = []
        for table_name in tables:
            sc = score_table(table_name, matched, vocab)
            if sc >= 6:
                candidates.append((sc, table_name))
        for kw in matched:
            for t in vocab.get(kw, []):
                real = next((r for r in tables if r.lower() == t.lower()), None)
                if real and not any(real == c[1] for c in candidates):
                    candidates.append((10, real))
        if not candidates:
            continue
        candidates.sort(key=lambda x: -x[0])
        top_tables = [t for _, t in candidates[:10]]
        mappings = []
        for table_name in top_tables:
            cols = tables[table_name]
            primary = get_relevant_columns(cols, tokens, matched)
            mappings.append({
                "responsible_table": table_name,
                "primary_columns": primary,
                "all_table_columns": cols
            })
        keywords_str = ", ".join(sorted(matched))
        intent = (f"Retrieve camera / rules / alerts related to: {keywords_str}"
                  if db_name == "video_analytics"
                  else f"Retrieve machine / production / work-order data related to: {keywords_str}")
        result["selected_databases"].append(db_name)
        result["customized_db_requests"].append({
            "database": db_name,
            "customized_sub_query": intent,
            "token_keywords": sorted(list(matched)),
            "target_mappings": mappings
        })
    return result

# -------------------------------------------------------------------
# 6. Date Detection
# -------------------------------------------------------------------
REAL_DATE_HINTS = [
    "created_at", "updated_at", "checked_at", "triggered_at", "timestamp",
    "createdon", "updatedon", "created_date", "updated_date",
    "startdate", "enddate", "plannedstart", "actualstart", "date",
    "changedat", "lastupdated"
]
BAD_DATE_COLUMNS = {
    "updatedby", "createdby", "status", "type", "name", "code", "location",
    "rtsp_template", "utilizationpercent", "utilization"
}

def detect_date_column(columns: List[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in columns}
    for p in REAL_DATE_HINTS:
        if p in cols_lower:
            return cols_lower[p]
    for c in columns:
        cl = c.lower()
        if cl in BAD_DATE_COLUMNS:
            continue
        if any(h in cl for h in ["_at", "_on", "date", "time", "timestamp"]) and len(cl) > 4:
            return c
    return None


def extract_time_filter(user_query: str) -> Optional[str]:
    q = user_query.lower()
    today = datetime.now().date()
    if "last week" in q or "past week" in q:
        start = today - timedelta(days=7)
        return f"{{date_col}} >= '{start}'"
    if "today" in q:
        return f"{{date_col}} >= '{today}'"
    if "yesterday" in q:
        y = today - timedelta(days=1)
        return f"{{date_col}} >= '{y}' AND {{date_col}} < '{today}'"
    if "last month" in q or "past month" in q:
        start = today - timedelta(days=30)
        return f"{{date_col}} >= '{start}'"
    return None

# -------------------------------------------------------------------
# 7. SQL Generator
# -------------------------------------------------------------------
def generate_sql_for_table(
    table_name: str,
    primary_columns: List[str],
    all_columns: List[str],
    time_filter: Optional[str] = None,
    limit: int = 50
) -> str:
    cols = primary_columns if primary_columns else all_columns[:8]
    if not cols:
        cols = ["*"]
    for possible_id in ["id", "Id", "ID", f"{table_name}Id"]:
        if possible_id in all_columns and possible_id not in cols:
            cols.insert(0, possible_id)
            break
    col_list = ", ".join(f'"{c}"' for c in cols)
    sql = f'SELECT {col_list}\nFROM "{table_name}"'
    date_col = detect_date_column(all_columns)
    if time_filter and date_col:
        real_filter = time_filter.replace("{date_col}", f'"{date_col}"')
        sql += f"\nWHERE {real_filter}"
    if date_col:
        sql += f'\nORDER BY "{date_col}" DESC'
    sql += f"\nLIMIT {limit};"
    return sql


def generate_sql_queries(pipeline_output: Dict[str, Any], limit_per_table: int = 50) -> Dict[str, Any]:
    time_filter = extract_time_filter(pipeline_output.get("raw_user_query", ""))
    sql_result = {
        "raw_user_query": pipeline_output.get("raw_user_query"),
        "time_filter_applied": time_filter,
        "databases": []
    }
    for req in pipeline_output.get("customized_db_requests", []):
        db_entry = {
            "database": req["database"],
            "intent": req["customized_sub_query"],
            "keywords": req["token_keywords"],
            "queries": []
        }
        for mapping in req["target_mappings"]:
            table = mapping["responsible_table"]
            primary = mapping["primary_columns"]
            all_cols = mapping["all_table_columns"]
            sql = generate_sql_for_table(table, primary, all_cols, time_filter, limit=limit_per_table)
            db_entry["queries"].append({
                "table": table,
                "sql": sql,
                "columns_used": primary if primary else all_cols[:8],
                "date_column_used": detect_date_column(all_cols),
                "explanation": f"Fetch relevant columns from {table}"
            })
        sql_result["databases"].append(db_entry)
    return sql_result

# -------------------------------------------------------------------
# 8. Validator
# -------------------------------------------------------------------
def validate_sql(sql: str, table: str, columns_used: List[str], date_col: Optional[str]) -> List[str]:
    issues = []
    if date_col and "ORDER BY" not in sql:
        issues.append(f"Has date column '{date_col}' but no ORDER BY")
    if len(columns_used) < 2:
        issues.append("Too few columns selected")
    if "*" in sql:
        issues.append("Using SELECT *")
    for bad in ["_at", "4NN", "2NN", "DateOn"]:
        if f'"{bad}"' in sql or f".{bad}" in sql:
            issues.append(f"Noise column still present: {bad}")
    return issues


def analyze_pipeline_result(sql_result: Dict[str, Any]) -> Dict[str, Any]:
    report = {
        "query": sql_result["raw_user_query"],
        "databases_selected": [d["database"] for d in sql_result["databases"]],
        "total_queries": 0,
        "issues_found": [],
        "status": "OK"
    }
    for db in sql_result["databases"]:
        for q in db["queries"]:
            report["total_queries"] += 1
            issues = validate_sql(q["sql"], q["table"], q["columns_used"], q.get("date_column_used"))
            for iss in issues:
                report["issues_found"].append(f"[{db['database']}.{q['table']}] {iss}")
    if report["issues_found"]:
        report["status"] = "ISSUES_DETECTED"
    return report

# -------------------------------------------------------------------
# 9. Database connections & execution
# -------------------------------------------------------------------
def get_mes_connection():
    driver = os.getenv("DB_DRIVER", "ODBC Driver 18 for SQL Server")
    server = os.getenv("DB_SERVER", "localhost,1433")
    database = os.getenv("DB_NAME", "mes_new")
    trusted = os.getenv("DB_TRUSTED_CONNECTION", "yes").lower() == "yes"
    encrypt = os.getenv("DB_ENCRYPT", "no")
    trust_cert = os.getenv("DB_TRUST_SERVER_CERTIFICATE", "yes")
    conn_str = (
        f"DRIVER={{{driver}}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Trusted_Connection={'yes' if trusted else 'no'};"
        f"Encrypt={encrypt};"
        f"TrustServerCertificate={trust_cert};"
    )
    return pyodbc.connect(conn_str, timeout=15)


def get_video_analytics_connection():
    url = os.getenv("CONSTRUCTION_DB_URL")
    if url and url.startswith("postgresql"):
        url = url.replace("postgresql+psycopg2://", "postgresql://")
        return psycopg2.connect(url)
    return psycopg2.connect(
        host=os.getenv("PG_HOST", "localhost"),
        port=os.getenv("PG_PORT", "5432"),
        dbname=os.getenv("PG_DB", "construction_ai"),
        user=os.getenv("PG_USER", "postgres"),
        password=os.getenv("PG_PASSWORD", "0987654321")
    )


def execute_sql(db_name: str, sql: str) -> Tuple[List[str], List[Tuple]]:
    if db_name == "mes":
        sql = re.sub(r'"([^"]+)"', r'[\1]', sql)
        m = re.search(r"LIMIT\s+(\d+)", sql, re.IGNORECASE)
        if m:
            limit = m.group(1)
            sql = re.sub(r"LIMIT\s+\d+", "", sql, flags=re.IGNORECASE)
            sql = re.sub(r"(SELECT\s+)", rf"\1TOP {limit} ", sql, count=1, flags=re.IGNORECASE)
    try:
        if db_name == "mes":
            conn = get_mes_connection()
            cursor = conn.cursor()
            cursor.execute(sql)
            columns = [col[0] for col in cursor.description] if cursor.description else []
            rows = cursor.fetchall()
            cursor.close()
            conn.close()
            return columns, [tuple(r) for r in rows]
        else:
            conn = get_video_analytics_connection()
            cursor = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)
            cursor.execute(sql)
            columns = [desc[0] for desc in cursor.description] if cursor.description else []
            rows = cursor.fetchall()
            cursor.close()
            conn.close()
            return columns, [tuple(r) for r in rows]
    except Exception as e:
        print(f"[ERROR] Failed to execute on {db_name}: {e}")
        print(f"SQL was:\n{sql}")
        return [], []


def run_all_queries(sql_result: Dict[str, Any]) -> Dict[str, Any]:
    execution_result = {
        "raw_user_query": sql_result["raw_user_query"],
        "time_filter_applied": sql_result.get("time_filter_applied"),
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "databases": []
    }
    for db_entry in sql_result.get("databases", []):
        db_name = db_entry["database"]
        new_db = {
            "database": db_name,
            "intent": db_entry["intent"],
            "keywords": db_entry["keywords"],
            "queries": []
        }
        for q in db_entry["queries"]:
            cols, rows = execute_sql(db_name, q["sql"])
            new_db["queries"].append({
                "table": q["table"],
                "sql": q["sql"],
                "columns": cols,
                "rows": rows,
                "row_count": len(rows),
                "explanation": q["explanation"]
            })
        execution_result["databases"].append(new_db)
    return execution_result

# -------------------------------------------------------------------
# 10. Rich Metadata
# -------------------------------------------------------------------
def _detect_type(values: list) -> str:
    non_null = [v for v in values if v is not None]
    if not non_null:
        return "unknown"
    sample = non_null[0]
    if isinstance(sample, bool):
        return "boolean"
    if isinstance(sample, int) and not isinstance(sample, bool):
        return "integer"
    if isinstance(sample, float):
        return "numeric"
    try:
        float(sample)
        return "numeric"
    except (TypeError, ValueError):
        pass
    return "string"


def _build_rich_metadata(table_name: str, columns: List[str], rows: List[Tuple], max_sample: int = 5) -> Dict[str, Any]:
    if not rows or not columns:
        return {
            "table_name": table_name,
            "total_records": 0,
            "total_columns": len(columns),
            "columns": columns,
            "column_details": {},
            "data_quality": {}
        }

    total_records = len(rows)
    col_details = {}
    zero_values = {}
    potential_outliers = []

    for col_idx, col_name in enumerate(columns):
        col_values = [row[col_idx] for row in rows]
        non_null = [v for v in col_values if v is not None]
        null_count = total_records - len(non_null)
        unique_count = len(set(str(v) for v in non_null))

        dtype = _detect_type(col_values)
        detail = {
            "data_type": dtype,
            "null_count": null_count,
            "unique_count": unique_count,
            "sample_values": []
        }

        samples = []
        seen = set()
        for v in non_null:
            s = str(v)
            if s not in seen:
                samples.append(v)
                seen.add(s)
            if len(samples) >= max_sample:
                break
        detail["sample_values"] = samples

        if dtype in ("numeric", "integer"):
            nums = []
            for v in non_null:
                try:
                    nums.append(float(v))
                except (TypeError, ValueError):
                    continue
            if nums:
                detail["min"] = round(min(nums), 2)
                detail["max"] = round(max(nums), 2)
                detail["avg"] = round(sum(nums) / len(nums), 2)

                zero_cnt = sum(1 for n in nums if n == 0)
                if zero_cnt:
                    zero_values[col_name] = zero_cnt

                if len(nums) > 5:
                    avg = detail["avg"]
                    for n in nums:
                        if avg > 0 and n > avg * 10:
                            potential_outliers.append({
                                "column": col_name,
                                "value": n,
                                "issue": f"Unusually high value (>{avg * 10:.1f})"
                            })
                            break

        col_details[col_name] = detail

    return {
        "table_name": table_name,
        "total_records": total_records,
        "total_columns": len(columns),
        "columns": columns,
        "column_details": col_details,
        "data_quality": {
            "potential_outliers": potential_outliers,
            "zero_values": zero_values
        }
    }


def _truncate(val: Any, max_len: int = 40) -> str:
    s = str(val) if val is not None else ""
    return s if len(s) <= max_len else s[:max_len - 1] + "…"

# -------------------------------------------------------------------
# 11. Prepare clean data for LLM
# -------------------------------------------------------------------
def prepare_llm_context(execution_result: Dict[str, Any], metadata_threshold: int = 5) -> str:
    parts = []
    parts.append(f"USER QUERY: {execution_result['raw_user_query']}")
    parts.append(f"Generated at: {execution_result['generated_at']}")
    if execution_result.get("time_filter_applied"):
        parts.append(f"Time filter: {execution_result['time_filter_applied']}")
    parts.append("")

    for db in execution_result["databases"]:
        parts.append(f"=== DATABASE: {db['database'].upper()} ===")
        parts.append(f"Intent: {db['intent']}")
        parts.append(f"Keywords: {', '.join(db['keywords'])}")
        parts.append("")

        for q in db["queries"]:
            if q["row_count"] == 0:
                continue

            parts.append(f"--- Table: {q['table']} ({q['row_count']} rows) ---")

            if q["row_count"] > metadata_threshold:
                meta = _build_rich_metadata(q["table"], q["columns"], q["rows"])
                parts.append(json.dumps(meta, indent=2, default=str))
            else:
                parts.append(f"Columns: {q['columns']}")
                for i, row in enumerate(q["rows"][:15]):
                    parts.append(f"  Row {i+1}: {row}")
                if q["row_count"] > 15:
                    parts.append(f"  ... ({q['row_count']-15} more rows)")
            parts.append("")

    return "\n".join(parts)

# -------------------------------------------------------------------
# 12. LLM Layer (LiteLLM + your new model list)
# -------------------------------------------------------------------
LLM_SYSTEM_PROMPT = """You are an expert manufacturing & video-analytics data analyst.
You receive a user query and the actual data retrieved from the databases.

Your job is to produce a clean JSON array of report sections.

STRICT RULES:
1. Output ONLY valid JSON – no markdown, no explanation, no extra text.
2. Each section must have exactly these keys:
   - "title": short meaningful title (string)
   - "summary": 3-8 sentence plain-text summary of the data in this section (string)
   - "chart": true or false
   - "chart_config": null  OR  an object with:
        {
          "chart_type": "bar" | "line" | "pie",
          "x_axis": "column name",
          "y_axis": "column name",
          "label": "human readable label",
          "title": "chart title",
          "aggregation": "sum" | "avg" | "count" | null
        }
3. Use chart=true only when a visualization truly adds value.
4. Supported chart_type values are ONLY: "bar", "line", "pie".
5. Group related tables into logical sections when it makes sense.
6. Never invent data. Base everything strictly on the provided data.
"""

def call_llm_with_fallback(user_query: str, data_context: str) -> List[Dict[str, Any]]:
    messages = [
        {"role": "system", "content": LLM_SYSTEM_PROMPT},
        {"role": "user", "content": f"USER QUERY:\n{user_query}\n\nDATA CONTEXT:\n{data_context}"}
    ]

    # ============================================================
    # ORDERED FALLBACK MODELS (exactly as you specified)
    # ============================================================
    models = [
        # Primary: Google Gemini
        "gemini/gemini-3.5-flash",
        # Secondary: Google Gemini (newer Flash)
        "gemini/gemini-3.6-flash",
        # Tertiary: Groq GPT-OSS 120B (advanced reasoning)
        "groq/openai/gpt-oss-120b",
        # Fourth: Groq GPT-OSS 20B (fast and cost-efficient)
        "groq/openai/gpt-oss-20b",
    ]

    last_error = None
    for model in models:
        try:
            print(f"[LLM] Trying model: {model}")
            response = completion(
                model=model,
                messages=messages,
                temperature=0.2,
                max_tokens=4096,
            )
            content = response.choices[0].message.content.strip()

            # Clean possible markdown fences
            if content.startswith("```"):
                content = re.sub(r"^```(?:json)?\s*", "", content)
                content = re.sub(r"\s*```$", "", content)

            parsed = json.loads(content)

            if isinstance(parsed, list):
                sections = parsed
            elif isinstance(parsed, dict) and "sections" in parsed:
                sections = parsed["sections"]
            elif isinstance(parsed, dict):
                sections = [parsed]
            else:
                raise ValueError("Unexpected JSON structure from LLM")

            for s in sections:
                if "title" not in s or "summary" not in s:
                    raise ValueError("Missing required keys in section")
                if "chart" not in s:
                    s["chart"] = False
                if s.get("chart") and not s.get("chart_config"):
                    s["chart"] = False
                    s["chart_config"] = None

            print(f"[LLM] Success with {model} – {len(sections)} sections")
            return sections

        except Exception as e:
            print(f"[LLM] {model} failed: {e}")
            last_error = e
            continue

    print(f"[LLM] All models failed. Last error: {last_error}")
    return [{
        "title": "Data Summary",
        "summary": "Unable to generate detailed AI summary. Please review the raw data tables.",
        "chart": False,
        "chart_config": None
    }]

# -------------------------------------------------------------------
# 13. Chart drawing from LLM chart_config
# -------------------------------------------------------------------
def draw_chart_from_config(
    chart_config: Dict[str, Any],
    columns: List[str],
    rows: List[Tuple]
) -> Optional[io.BytesIO]:
    if not chart_config or not rows or not columns:
        return None

    chart_type = (chart_config.get("chart_type") or "").lower()
    x_axis = chart_config.get("x_axis")
    y_axis = chart_config.get("y_axis")
    title = chart_config.get("title") or chart_config.get("label") or "Chart"
    aggregation = (chart_config.get("aggregation") or "sum").lower()

    col_idx = {c: i for i, c in enumerate(columns)}

    def get_col(name):
        if name in col_idx:
            return [row[col_idx[name]] for row in rows]
        return None

    try:
        if chart_type == "pie":
            cat_col = x_axis or (columns[0] if columns else None)
            if not cat_col:
                return None
            cats = get_col(cat_col)
            if not cats:
                return None
            counts = Counter(str(c) for c in cats if c is not None)
            if len(counts) < 2 or len(counts) > 12:
                return None
            labels = list(counts.keys())
            sizes = list(counts.values())
            plt.figure(figsize=(6.5, 4.5))
            plt.pie(sizes, labels=labels, autopct="%1.1f%%", startangle=90, colors=plt.cm.Set3.colors)
            plt.title(title)
            buf = io.BytesIO()
            plt.tight_layout()
            plt.savefig(buf, format="png", dpi=120, bbox_inches="tight")
            buf.seek(0)
            plt.close()
            return buf

        if chart_type in ("bar", "line"):
            if not x_axis or not y_axis:
                return None
            xs = get_col(x_axis)
            ys = get_col(y_axis)
            if not xs or not ys:
                return None

            agg = defaultdict(list)
            for x, y in zip(xs, ys):
                if x is not None and y is not None:
                    try:
                        agg[str(x)].append(float(y))
                    except:
                        pass
            if not agg:
                return None

            labels = list(agg.keys())[:15]
            if aggregation == "avg":
                values = [sum(agg[l]) / len(agg[l]) for l in labels]
            elif aggregation == "count":
                values = [len(agg[l]) for l in labels]
            else:
                values = [sum(agg[l]) for l in labels]

            plt.figure(figsize=(8, 4))
            if chart_type == "bar":
                plt.bar(labels, values, color="#1a5276")
            else:
                plt.plot(labels, values, marker="o", linewidth=2, color="#2874a6")
                plt.fill_between(range(len(labels)), values, alpha=0.2, color="#2874a6")
            plt.title(title)
            plt.xticks(rotation=45, ha="right", fontsize=8)
            plt.ylabel(y_axis)
            buf = io.BytesIO()
            plt.tight_layout()
            plt.savefig(buf, format="png", dpi=120, bbox_inches="tight")
            buf.seek(0)
            plt.close()
            return buf

    except Exception as e:
        print(f"[CHART] Failed to draw {chart_type}: {e}")
        plt.close()
        return None

    return None

# -------------------------------------------------------------------
# 14. PDF Generator (FIXED – BodyText renamed to ReportBody)
# -------------------------------------------------------------------
def generate_pdf_from_llm_sections(
    execution_result: Dict[str, Any],
    llm_sections: List[Dict[str, Any]],
    output_path: str = None,
    metadata_threshold: int = 5
) -> str:
    if output_path is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"query_report_{ts}.pdf"
    output_path = str(Path(output_path).resolve())

    doc = SimpleDocTemplate(
        output_path,
        pagesize=landscape(A4),
        leftMargin=12 * mm,
        rightMargin=12 * mm,
        topMargin=12 * mm,
        bottomMargin=12 * mm
    )

    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(
        name="ReportTitle",
        parent=styles["Heading1"],
        fontSize=16,
        alignment=TA_CENTER,
        spaceAfter=6
    ))
    styles.add(ParagraphStyle(
        name="SectionTitle",
        parent=styles["Heading2"],
        fontSize=12,
        textColor=colors.HexColor("#1a5276"),
        spaceBefore=10,
        spaceAfter=4
    ))
    # FIXED: renamed from BodyText → ReportBody (avoids KeyError)
    styles.add(ParagraphStyle(
        name="ReportBody",
        parent=styles["Normal"],
        fontSize=9,
        leading=12,
        spaceAfter=4
    ))
    styles.add(ParagraphStyle(
        name="Meta",
        parent=styles["Normal"],
        fontSize=8,
        textColor=colors.grey
    ))
    styles.add(ParagraphStyle(
        name="MetadataBox",
        parent=styles["Normal"],
        fontSize=8,
        leading=11,
        backColor=colors.HexColor("#eaf2f8"),
        borderPadding=6,
        spaceBefore=2,
        spaceAfter=4
    ))

    story = []

    # Title
    story.append(Paragraph("Manufacturing & Video-Analytics Intelligent Report", styles["ReportTitle"]))
    story.append(Spacer(1, 3 * mm))
    story.append(Paragraph(f"<b>User Query:</b> {execution_result['raw_user_query']}", styles["ReportBody"]))
    story.append(Paragraph(f"<b>Generated at:</b> {execution_result['generated_at']}", styles["Meta"]))
    story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor("#1a5276")))
    story.append(Spacer(1, 6 * mm))

    # Lookup for charts
    table_lookup = {}
    for db in execution_result["databases"]:
        for q in db["queries"]:
            if q["row_count"] > 0:
                table_lookup[q["table"]] = q

    # LLM sections
    for section in llm_sections:
        title = section.get("title", "Section")
        summary = section.get("summary", "")
        need_chart = section.get("chart", False)
        chart_cfg = section.get("chart_config")

        story.append(Paragraph(title, styles["SectionTitle"]))
        story.append(Paragraph(summary.replace("\n", "<br/>"), styles["ReportBody"]))
        story.append(Spacer(1, 2 * mm))

        if need_chart and chart_cfg:
            x_col = chart_cfg.get("x_axis")
            y_col = chart_cfg.get("y_axis")
            matched_q = None
            for tname, q in table_lookup.items():
                cols = q["columns"]
                if (x_col in cols or y_col in cols):
                    matched_q = q
                    break
            if matched_q is None and table_lookup:
                matched_q = next(iter(table_lookup.values()))

            if matched_q:
                buf = draw_chart_from_config(chart_cfg, matched_q["columns"], matched_q["rows"])
                if buf:
                    img = Image(buf, width=155 * mm, height=85 * mm)
                    story.append(img)
                    story.append(Spacer(1, 3 * mm))

        story.append(Spacer(1, 4 * mm))

    # Appendix – metadata
    story.append(PageBreak())
    story.append(Paragraph("Appendix – Data Metadata", styles["SectionTitle"]))
    for db in execution_result["databases"]:
        for q in db["queries"]:
            if q["row_count"] == 0:
                continue
            if q["row_count"] > metadata_threshold:
                meta = _build_rich_metadata(q["table"], q["columns"], q["rows"])
                story.append(Paragraph(f"<b>{meta['table_name']}</b> ({meta['total_records']} records)", styles["Meta"]))
                for col, d in list(meta["column_details"].items())[:6]:
                    line = f"{col}: type={d.get('data_type')}, min={d.get('min','–')}, max={d.get('max','–')}, avg={d.get('avg','–')}"
                    story.append(Paragraph(line, styles["Meta"]))
                story.append(Spacer(1, 3 * mm))

    doc.build(story)
    print(f"[INFO] PDF written to: {output_path}")
    return output_path

# -------------------------------------------------------------------
# 15. High-level pipeline
# -------------------------------------------------------------------
def process_query_and_generate_pdf(user_query: str, pdf_path: str = None) -> str:
    print(f"\n>>> Processing: {user_query}")

    split = customize_and_split_query(user_query)
    if not split["selected_databases"]:
        print("[WARN] No databases matched the query.")
    sql_result = generate_sql_queries(split, limit_per_table=50)
    report = analyze_pipeline_result(sql_result)
    print(f"DBs selected : {report['databases_selected']}")
    print(f"Queries gen  : {report['total_queries']}")

    execution = run_all_queries(sql_result)

    data_context = prepare_llm_context(execution)
    print("\n[LLM] Context length:", len(data_context), "chars")

    llm_sections = call_llm_with_fallback(user_query, data_context)
    print(f"[LLM] Received {len(llm_sections)} sections")

    pdf_file = generate_pdf_from_llm_sections(execution, llm_sections, output_path=pdf_path)
    return pdf_file

# -------------------------------------------------------------------
# 16. Main
# -------------------------------------------------------------------
if __name__ == "__main__":
    user_q = (
        "I want the data from the mes application for machine utilization "
        "and from video analytics I want camera configuration information."
    )
    pdf_file = process_query_and_generate_pdf(user_q, pdf_path="intelligent_report.pdf")
    print(f"\n✅ Done. Open the PDF: {pdf_file}")

[INFO] Loaded video_analytics: 52 tables
[INFO] Loaded mes: 108 tables

>>> Processing: I want the data from the mes application for machine utilization and from video analytics I want camera configuration information.
DBs selected : ['video_analytics', 'mes']
Queries gen  : 12


17:53:21 - LiteLLM:WARNING: vertex_and_google_ai_studio_gemini.py:1095 - DeprecationWarning: `temperature`, `top_p`, and `top_k` continue to function for Gemini 3+ (gemini-3.5-flash) but are planned for removal in a future release. Move sampling guidance into the `system` instructions instead.



[LLM] Context length: 13889 chars
[LLM] Trying model: gemini/gemini-3.5-flash
[LLM] Success with gemini/gemini-3.5-flash – 2 sections
[LLM] Received 2 sections
[INFO] PDF written to: C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\jupyter notebooks\intelligent_report.pdf

✅ Done. Open the PDF: C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\jupyter notebooks\intelligent_report.pdf


In [53]:
"""
================================================================================
 Manufacturing & Video-Analytics Intelligent Report Generator  (v2)
================================================================================

Upgrades over the original single-file agent:

  Query understanding
    - Fuzzy / typo-tolerant keyword matching (difflib) layered on top of the
      existing domain vocabulary
    - Aggregation detection ("average", "total", "trend", "compare by ...")
      -> generates GROUP BY analytic SQL instead of flat row dumps when the
         question actually asks for a summary
    - Wider time-filter vocabulary (this week / month / quarter, last N days)

  Execution
    - Generic, config-driven multi-database registry. Ships with the same
      `mes` (MSSQL) + `video_analytics` (Postgres) pair, but a third database
      can be added via an external JSON file without touching this code.
    - Parallel query execution (ThreadPoolExecutor) instead of serial
    - Per-table data-quality scoring (completeness %, outliers, uniqueness)

  Resilience
    - Retry + exponential backoff around every LLM call
    - If every model in the fallback chain fails, sections are now built
      directly from the retrieved data instead of a dead placeholder message

  PDF output
    - Cover page, real table of contents (clickable bookmarks + accurate
      page numbers via a two-pass build), running header/footer with
      "Page X of Y"
    - KPI summary cards (databases queried, tables hit, total rows, filter)
    - Actual data-preview tables per section, not just a chart
    - Expanded chart types (bar / horizontal bar / line / pie / donut /
      scatter) with a consistent color theme and value labels
    - Data-quality appendix, colour-coded by score

Entry point kept compatible with the original script:

    process_query_and_generate_pdf(user_query, pdf_path=None) -> str

================================================================================
"""

from __future__ import annotations

import argparse
import io
import json
import logging
import os
import re
import sys
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from difflib import get_close_matches
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

# --------------------------------------------------------------------------
# Third-party libs
# --------------------------------------------------------------------------
try:
    from dotenv import load_dotenv
except ImportError:
    raise ImportError("pip install python-dotenv")

try:
    import pyodbc
except ImportError:
    raise ImportError("pip install pyodbc")

try:
    import psycopg2
    import psycopg2.extras
except ImportError:
    raise ImportError("pip install psycopg2-binary")

try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4, landscape
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import mm
    from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
        PageBreak, HRFlowable, Image, NextPageTemplate,
        PageTemplate, Frame, BaseDocTemplate, KeepTogether,
    )
    from reportlab.platypus.tableofcontents import TableOfContents
    from reportlab.pdfgen import canvas as pdf_canvas
except ImportError:
    raise ImportError("pip install reportlab")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np
except ImportError:
    raise ImportError("pip install matplotlib numpy")

try:
    from litellm import completion
except ImportError:
    raise ImportError("pip install litellm")

load_dotenv()
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY", "")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

# --------------------------------------------------------------------------
# Logging
# --------------------------------------------------------------------------
logging.basicConfig(
    level=os.getenv("LOG_LEVEL", "INFO"),
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("report_agent")

# --------------------------------------------------------------------------
# Visual theme (shared by charts + PDF)
# --------------------------------------------------------------------------
THEME = {
    "primary": "#1a5276",
    "primary_light": "#2874a6",
    "accent": "#e67e22",
    "success": "#1e8449",
    "warning": "#b9770e",
    "danger": "#c0392b",
    "bg_light": "#eaf2f8",
    "bg_alt_row": "#f4f8fb",
    "grey": "#5d6d7e",
    "palette": ["#1a5276", "#2874a6", "#5499c7", "#7fb3d5",
                "#e67e22", "#d68910", "#1e8449", "#27ae60",
                "#c0392b", "#a04000"],
}

# --------------------------------------------------------------------------
# 1. Database registry (config-driven, extensible beyond mes / video_analytics)
# --------------------------------------------------------------------------
@dataclass
class DatabaseConfig:
    name: str
    db_type: str                     # "mssql" | "postgres"
    schema_path: str
    connection: Dict[str, str] = field(default_factory=dict)


def _default_registry() -> Dict[str, DatabaseConfig]:
    """
    Builds the built-in mes + video_analytics registry from environment
    variables (falling back to the original hardcoded defaults so this is
    a drop-in replacement). Extra databases can be appended by pointing
    EXTRA_DB_CONFIG_FILE at a JSON file shaped like:

    {
      "quality": {
        "db_type": "postgres",
        "schema_path": "C:/.../quality_schema.json",
        "connection": {"host": "localhost", "port": "5432", "dbname": "qa",
                        "user": "postgres", "password": "..."}
      }
    }
    """
    registry: Dict[str, DatabaseConfig] = {
        "video_analytics": DatabaseConfig(
            name="video_analytics",
            db_type="postgres",
            schema_path=os.getenv(
                "VIDEO_ANALYTICS_SCHEMA_PATH",
                r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\video_analytics_db_info\construction_ai_schema.json",
            ),
            connection={
                "url": os.getenv("CONSTRUCTION_DB_URL", ""),
                "host": os.getenv("PG_HOST", "localhost"),
                "port": os.getenv("PG_PORT", "5432"),
                "dbname": os.getenv("PG_DB", "construction_ai"),
                "user": os.getenv("PG_USER", "postgres"),
                "password": os.getenv("PG_PASSWORD", "0987654321"),
            },
        ),
        "mes": DatabaseConfig(
            name="mes",
            db_type="mssql",
            schema_path=os.getenv(
                "MES_SCHEMA_PATH",
                r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.json",
            ),
            connection={
                "driver": os.getenv("DB_DRIVER", "ODBC Driver 18 for SQL Server"),
                "server": os.getenv("DB_SERVER", "localhost,1433"),
                "database": os.getenv("DB_NAME", "mes_new"),
                "trusted": os.getenv("DB_TRUSTED_CONNECTION", "yes"),
                "encrypt": os.getenv("DB_ENCRYPT", "no"),
                "trust_cert": os.getenv("DB_TRUST_SERVER_CERTIFICATE", "yes"),
            },
        ),
    }

    extra_path = os.getenv("EXTRA_DB_CONFIG_FILE")
    if extra_path and os.path.exists(extra_path):
        try:
            with open(extra_path, "r", encoding="utf-8") as f:
                extra = json.load(f)
            for name, cfg in extra.items():
                registry[name] = DatabaseConfig(
                    name=name,
                    db_type=cfg.get("db_type", "postgres"),
                    schema_path=cfg.get("schema_path", ""),
                    connection=cfg.get("connection", {}),
                )
            log.info("Loaded %d extra database(s) from %s", len(extra), extra_path)
        except Exception as e:
            log.warning("Failed to load EXTRA_DB_CONFIG_FILE (%s): %s", extra_path, e)

    return registry


DB_REGISTRY: Dict[str, DatabaseConfig] = _default_registry()


def validate_environment() -> List[str]:
    """Sanity-checks config before doing any real work. Returns a list of
    human-readable problems (empty list == all good)."""
    problems = []
    for name, cfg in DB_REGISTRY.items():
        if not cfg.schema_path or not os.path.exists(cfg.schema_path):
            problems.append(f"[{name}] schema file not found: {cfg.schema_path}")
    if not os.getenv("GEMINI_API_KEY") and not os.getenv("GROQ_API_KEY"):
        problems.append("No GEMINI_API_KEY or GROQ_API_KEY set - LLM summaries will fall back to raw data.")
    return problems


# --------------------------------------------------------------------------
# 2. Stop words + domain vocabulary  (extensible via DOMAIN_VOCAB_FILE)
# --------------------------------------------------------------------------
STOP_WORDS = {
    "a", "an", "the", "and", "or", "of", "in", "on", "for", "to", "from", "with",
    "is", "are", "was", "were", "be", "been", "being", "have", "has", "had",
    "do", "does", "did", "will", "would", "can", "could", "should", "may", "might",
    "i", "me", "my", "we", "you", "your", "he", "she", "it", "they", "them",
    "this", "that", "these", "those", "what", "which", "who", "whom", "whose",
    "all", "any", "some", "no", "not", "only", "just", "also", "very", "too",
    "want", "need", "show", "give", "get", "list", "data", "information", "details",
    "application", "system", "report", "me", "please",
}

DOMAIN_VOCABULARY: Dict[str, Dict[str, List[str]]] = {
    "video_analytics": {
        "camera": ["cameras", "camera_status_logs", "basler_devices", "basler_model_assignments", "hse_camera_rules"],
        "cameras": ["cameras", "camera_status_logs", "basler_devices"],
        "config": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments"],
        "configuration": ["cameras", "hse_camera_rules", "basler_model_assignments", "detection_assignments"],
        "rule": ["hse_camera_rules", "hse_rule_definitions", "hse_rule_events"],
        "hse": ["hse_camera_rules", "hse_rule_definitions", "hse_rule_events"],
        "alert": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "violation": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "safety": ["alerts", "anomaly_flags", "hse_rule_events", "incidents"],
        "anomaly": ["anomaly_flags"],
        "incident": ["incidents"],
        "zone": ["zones", "zone_risk_scores"],
        "worker": ["attendances", "employees", "employee_movements"],
        "employee": ["employees", "attendances", "employee_movements"],
        "attendance": ["attendances"],
        "model": ["ai_models", "ai_model_classes", "basler_model_assignments"],
        "basler": ["basler_devices", "basler_model_assignments"],
    },
    "mes": {
        "machine": ["Machine", "MachineMaster", "MachineFGMapping", "MachineMaterialMapping", "CapacityAnalysis"],
        "machines": ["Machine", "MachineMaster"],
        "utilization": ["CapacityAnalysis", "MachineMaster"],
        "util": ["CapacityAnalysis", "MachineMaster"],
        "capacity": ["CapacityAnalysis", "MachineMaster", "ContainerCapacity"],
        "workorder": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog", "WorkOrderResource"],
        "work_order": ["WorkOrder", "WorkOrderStep", "WorkOrderBOM", "WorkOrderLog"],
        "wo": ["WorkOrder", "WorkOrderStep"],
        "production": ["WorkOrder", "ProductionPlanning", "FinishedGood", "SemiFinishedGood"],
        "shift": ["ShiftMaster", "ShiftCalendar", "OperatorMaster"],
        "operator": ["OperatorMaster", "WorkOrderResource"],
        "inventory": ["Inventory", "InventoryByLot", "FinishedGood", "SemiFinishedGood"],
        "material": ["Materials", "RawMaterial", "BOMLine", "BOMMaster"],
        "bom": ["BOMMaster", "BOMLine"],
        "mps": ["MPSHeader", "MpsMaster"],
        "mrp": ["MRP_Run", "MRP_Demand", "MRP_Result"],
        "planning": ["MPSHeader", "MpsMaster", "ProductionPlanning", "WeeklyPlanning"],
        "maintenance": ["MaintenanceWindow"],
        "schedule": ["GanttSchedule", "WorkOrder"],
        "downtime": ["MaintenanceWindow", "CapacityAnalysis"],
    },
}


def _load_extra_vocabulary() -> None:
    """Merge in extra domain vocabulary (e.g. for a 3rd database) from a
    JSON file, keyed the same way as DOMAIN_VOCABULARY."""
    vocab_path = os.getenv("DOMAIN_VOCAB_FILE")
    if not vocab_path or not os.path.exists(vocab_path):
        return
    try:
        with open(vocab_path, "r", encoding="utf-8") as f:
            extra = json.load(f)
        for db_name, kw_map in extra.items():
            DOMAIN_VOCABULARY.setdefault(db_name, {})
            for kw, tables in kw_map.items():
                DOMAIN_VOCABULARY[db_name].setdefault(kw, [])
                for t in tables:
                    if t not in DOMAIN_VOCABULARY[db_name][kw]:
                        DOMAIN_VOCABULARY[db_name][kw].append(t)
        log.info("Merged extra domain vocabulary from %s", vocab_path)
    except Exception as e:
        log.warning("Failed to load DOMAIN_VOCAB_FILE (%s): %s", vocab_path, e)


_load_extra_vocabulary()

# --------------------------------------------------------------------------
# 3. Schema cleaning  (unchanged logic, cached so repeated runs are cheap)
# --------------------------------------------------------------------------
NOISE_COLUMNS = {
    "nn", "pk", "id", "date", "on", "at", "_at", "created", "updated",
    "qty", "quantity", "count", "bags", "bag",
}


def clean_column_name(raw: str) -> List[str]:
    name = raw.strip().split(":")[0].strip()
    if "/" in name:
        parts = re.split(r"[/]", name)
        cleaned = []
        for p in parts:
            p = re.sub(r"[^a-zA-Z0-9_]", "", p)
            if (p and len(p) > 2 and not p.isdigit()
                    and p.lower() not in NOISE_COLUMNS and not p.startswith("_")):
                cleaned.append(p)
        return cleaned
    name = re.sub(r"[^a-zA-Z0-9_]", "", name)
    if (name and len(name) > 2 and not name.isdigit()
            and name.lower() not in NOISE_COLUMNS
            and not name.startswith("_") and not re.match(r"^\d", name)):
        return [name]
    return []


def parse_and_clean_schema(file_path: str) -> Dict[str, List[str]]:
    table_map: Dict[str, List[str]] = {}
    if not file_path or not os.path.exists(file_path):
        log.warning("Schema file not found: %s", file_path)
        return table_map
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for table_name, schema_str in data.items():
        clean_cols: List[str] = []
        if isinstance(schema_str, str):
            for col_def in schema_str.split(","):
                clean_cols.extend(clean_column_name(col_def))
        elif isinstance(schema_str, list):
            clean_cols = [str(c) for c in schema_str if c]
        table_map[table_name] = list(dict.fromkeys(
            c for c in clean_cols
            if c and len(c) > 2 and not c.startswith("_")
            and c.lower() not in NOISE_COLUMNS and not re.match(r"^\d", c)
        ))
    return table_map


_SCHEMA_CACHE: Dict[str, Dict[str, List[str]]] = {}


def build_clean_registry(registry: Dict[str, DatabaseConfig]) -> Dict[str, Dict[str, List[str]]]:
    out: Dict[str, Dict[str, List[str]]] = {}
    for db_name, cfg in registry.items():
        mtime = os.path.getmtime(cfg.schema_path) if os.path.exists(cfg.schema_path) else 0
        cache_key = f"{cfg.schema_path}:{mtime}"
        if cache_key in _SCHEMA_CACHE:
            out[db_name] = _SCHEMA_CACHE[cache_key]
            continue
        parsed = parse_and_clean_schema(cfg.schema_path)
        _SCHEMA_CACHE[cache_key] = parsed
        out[db_name] = parsed
        log.info("Loaded %s: %d tables", db_name, len(parsed))
    return out


CLEAN_REGISTRY = build_clean_registry(DB_REGISTRY)

# --------------------------------------------------------------------------
# 4. Query understanding: tokens, fuzzy matching, aggregation intent
# --------------------------------------------------------------------------
AGGREGATION_VERBS = {
    "average": "AVG", "avg": "AVG", "mean": "AVG",
    "total": "SUM", "sum": "SUM",
    "count": "COUNT", "number": "COUNT",
    "maximum": "MAX", "max": "MAX", "highest": "MAX",
    "minimum": "MIN", "min": "MIN", "lowest": "MIN",
}
ANALYTIC_HINTS = {"trend", "compare", "comparison", "breakdown", "by", "per", "across", "over time"}


def extract_tokens(query: str) -> Set[str]:
    tokens = set(re.findall(r"[a-zA-Z0-9_]+", query.lower()))
    return {t for t in tokens if t not in STOP_WORDS and len(t) > 1}


def find_matched_keywords(tokens: Set[str], vocab: Dict[str, List[str]]) -> Set[str]:
    """Exact + substring + fuzzy (typo-tolerant) keyword matching against a
    domain vocabulary."""
    matched: Set[str] = set()
    vocab_keys = list(vocab.keys())
    for t in tokens:
        if t in vocab:
            matched.add(t)
            continue
        substr_hit = False
        for key in vocab_keys:
            if (t in key or key in t) and min(len(t), len(key)) >= 4:
                matched.add(key)
                substr_hit = True
        if substr_hit:
            continue
        # typo tolerance: only trust a fuzzy hit for longer tokens to avoid
        # false positives on short/common words
        if len(t) >= 5:
            close = get_close_matches(t, vocab_keys, n=1, cutoff=0.82)
            if close:
                matched.add(close[0])
    return matched


def detect_aggregation(query: str) -> Optional[str]:
    """Returns a SQL aggregation function name (AVG/SUM/COUNT/MAX/MIN) if the
    question is asking for a summarised number rather than raw rows."""
    q = query.lower()
    for word, fn in AGGREGATION_VERBS.items():
        if re.search(rf"\b{re.escape(word)}\b", q):
            return fn
    if any(h in q for h in ANALYTIC_HINTS):
        return "AVG"
    return None


def score_table(table_name: str, matched_keywords: Set[str], vocab: Dict[str, List[str]]) -> int:
    score = 0
    table_l = table_name.lower()
    for kw in matched_keywords:
        if kw in vocab and table_name in vocab[kw]:
            score += 10
        if kw in table_l:
            score += 6
    return score


def get_relevant_columns(cols: List[str], tokens: Set[str], matched: Set[str]) -> List[str]:
    scored = []
    for col in cols:
        col_l = col.lower()
        s = 0
        if col_l in tokens:
            s += 12
        for m in matched:
            if m in col_l or col_l in m:
                s += 5
        if any(x in col_l for x in ["id", "name", "code", "status", "qty", "quantity",
                                     "util", "capacity", "is_active", "active"]):
            s += 2
        scored.append((s, col))
    scored.sort(key=lambda x: (-x[0], x[1]))
    relevant = [c for s, c in scored if s >= 2]
    if len(relevant) < 3:
        relevant = cols[:8]
    return relevant[:8]


def _find_numeric_column(cols: List[str]) -> Optional[str]:
    for c in cols:
        cl = c.lower()
        if any(h in cl for h in ["util", "capacity", "qty", "quantity", "amount",
                                  "percent", "score", "count", "value", "total"]):
            return c
    return None


def _find_categorical_column(cols: List[str]) -> Optional[str]:
    for c in cols:
        cl = c.lower()
        if any(h in cl for h in ["status", "type", "name", "code", "shift", "zone", "machine", "camera"]):
            return c
    return None


def customize_and_split_query(user_query: str) -> Dict[str, Any]:
    tokens = extract_tokens(user_query)
    agg_fn = detect_aggregation(user_query)
    result: Dict[str, Any] = {
        "raw_user_query": user_query,
        "aggregation": agg_fn,
        "selected_databases": [],
        "customized_db_requests": [],
    }
    for db_name, tables in CLEAN_REGISTRY.items():
        vocab = DOMAIN_VOCABULARY.get(db_name, {})
        matched = find_matched_keywords(tokens, vocab)
        if not matched:
            continue
        candidates = []
        for table_name in tables:
            sc = score_table(table_name, matched, vocab)
            if sc >= 6:
                candidates.append((sc, table_name))
        for kw in matched:
            for t in vocab.get(kw, []):
                real = next((r for r in tables if r.lower() == t.lower()), None)
                if real and not any(real == c[1] for c in candidates):
                    candidates.append((10, real))
        if not candidates:
            continue
        candidates.sort(key=lambda x: -x[0])
        top_tables = [t for _, t in candidates[:10]]
        mappings = []
        for table_name in top_tables:
            cols = tables[table_name]
            primary = get_relevant_columns(cols, tokens, matched)
            mappings.append({
                "responsible_table": table_name,
                "primary_columns": primary,
                "all_table_columns": cols,
                "numeric_column": _find_numeric_column(cols),
                "categorical_column": _find_categorical_column(cols),
            })
        keywords_str = ", ".join(sorted(matched))
        intent = (f"Retrieve camera / rules / alerts related to: {keywords_str}"
                  if db_name == "video_analytics"
                  else f"Retrieve machine / production / work-order data related to: {keywords_str}")
        result["selected_databases"].append(db_name)
        result["customized_db_requests"].append({
            "database": db_name,
            "customized_sub_query": intent,
            "token_keywords": sorted(list(matched)),
            "target_mappings": mappings,
        })
    return result

# --------------------------------------------------------------------------
# 5. Date detection + time filters
# --------------------------------------------------------------------------
REAL_DATE_HINTS = [
    "created_at", "updated_at", "checked_at", "triggered_at", "timestamp",
    "createdon", "updatedon", "created_date", "updated_date",
    "startdate", "enddate", "plannedstart", "actualstart", "date",
    "changedat", "lastupdated",
]
BAD_DATE_COLUMNS = {
    "updatedby", "createdby", "status", "type", "name", "code", "location",
    "rtsp_template", "utilizationpercent", "utilization",
}


def detect_date_column(columns: List[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in columns}
    for p in REAL_DATE_HINTS:
        if p in cols_lower:
            return cols_lower[p]
    for c in columns:
        cl = c.lower()
        if cl in BAD_DATE_COLUMNS:
            continue
        if any(h in cl for h in ["_at", "_on", "date", "time", "timestamp"]) and len(cl) > 4:
            return c
    return None


def extract_time_filter(user_query: str) -> Optional[str]:
    q = user_query.lower()
    today = datetime.now().date()

    m = re.search(r"last (\d+) days?", q)
    if m:
        start = today - timedelta(days=int(m.group(1)))
        return f"{{date_col}} >= '{start}'"

    if "last week" in q or "past week" in q:
        start = today - timedelta(days=7)
        return f"{{date_col}} >= '{start}'"
    if "this week" in q:
        start = today - timedelta(days=today.weekday())
        return f"{{date_col}} >= '{start}'"
    if "today" in q:
        return f"{{date_col}} >= '{today}'"
    if "yesterday" in q:
        y = today - timedelta(days=1)
        return f"{{date_col}} >= '{y}' AND {{date_col}} < '{today}'"
    if "last month" in q or "past month" in q:
        start = today - timedelta(days=30)
        return f"{{date_col}} >= '{start}'"
    if "this month" in q:
        start = today.replace(day=1)
        return f"{{date_col}} >= '{start}'"
    if "last quarter" in q or "past quarter" in q:
        start = today - timedelta(days=90)
        return f"{{date_col}} >= '{start}'"
    return None


# --------------------------------------------------------------------------
# 6. SQL generation  (now aggregation-aware)
# --------------------------------------------------------------------------
def _apply_time_filter(sql_where: List[str], time_filter: Optional[str], date_col: Optional[str]) -> None:
    if time_filter and date_col:
        sql_where.append(time_filter.replace("{date_col}", f'"{date_col}"'))


def generate_sql_for_table(
    table_name: str,
    primary_columns: List[str],
    all_columns: List[str],
    time_filter: Optional[str] = None,
    limit: int = 50,
    aggregation: Optional[str] = None,
    numeric_column: Optional[str] = None,
    categorical_column: Optional[str] = None,
) -> Tuple[str, bool]:
    """Returns (sql, is_aggregate). Builds an analytic GROUP BY query when the
    user asked for an average/total/trend AND the table actually has a
    numeric + categorical column pair; otherwise falls back to the original
    flat row-select behaviour."""
    date_col = detect_date_column(all_columns)
    where_clauses: List[str] = []
    _apply_time_filter(where_clauses, time_filter, date_col)
    where_sql = f"\nWHERE {' AND '.join(where_clauses)}" if where_clauses else ""

    if aggregation and numeric_column and categorical_column:
        sql = (
            f'SELECT "{categorical_column}", '
            f'{aggregation}("{numeric_column}") AS "{aggregation.lower()}_{numeric_column}", '
            f'COUNT(*) AS "record_count"\n'
            f'FROM "{table_name}"'
            f'{where_sql}\n'
            f'GROUP BY "{categorical_column}"\n'
            f'ORDER BY 2 DESC\n'
            f'LIMIT {min(limit, 25)};'
        )
        return sql, True

    cols = primary_columns if primary_columns else all_columns[:8]
    if not cols:
        cols = ["*"]
    for possible_id in ["id", "Id", "ID", f"{table_name}Id"]:
        if possible_id in all_columns and possible_id not in cols:
            cols.insert(0, possible_id)
            break
    col_list = ", ".join(f'"{c}"' for c in cols)
    sql = f'SELECT {col_list}\nFROM "{table_name}"{where_sql}'
    if date_col:
        sql += f'\nORDER BY "{date_col}" DESC'
    sql += f"\nLIMIT {limit};"
    return sql, False


def generate_sql_queries(pipeline_output: Dict[str, Any], limit_per_table: int = 50) -> Dict[str, Any]:
    time_filter = extract_time_filter(pipeline_output.get("raw_user_query", ""))
    aggregation = pipeline_output.get("aggregation")
    sql_result: Dict[str, Any] = {
        "raw_user_query": pipeline_output.get("raw_user_query"),
        "time_filter_applied": time_filter,
        "aggregation_applied": aggregation,
        "databases": [],
    }
    for req in pipeline_output.get("customized_db_requests", []):
        db_entry = {
            "database": req["database"],
            "intent": req["customized_sub_query"],
            "keywords": req["token_keywords"],
            "queries": [],
        }
        for mapping in req["target_mappings"]:
            table = mapping["responsible_table"]
            primary = mapping["primary_columns"]
            all_cols = mapping["all_table_columns"]
            sql, is_agg = generate_sql_for_table(
                table, primary, all_cols, time_filter, limit=limit_per_table,
                aggregation=aggregation,
                numeric_column=mapping.get("numeric_column"),
                categorical_column=mapping.get("categorical_column"),
            )
            db_entry["queries"].append({
                "table": table,
                "sql": sql,
                "is_aggregate": is_agg,
                "columns_used": primary if primary else all_cols[:8],
                "date_column_used": detect_date_column(all_cols),
                "explanation": (
                    f"Aggregate ({aggregation}) view of {table}" if is_agg
                    else f"Fetch relevant columns from {table}"
                ),
            })
        sql_result["databases"].append(db_entry)
    return sql_result


# --------------------------------------------------------------------------
# 7. Validator
# --------------------------------------------------------------------------
def validate_sql(sql: str, columns_used: List[str], date_col: Optional[str], is_aggregate: bool) -> List[str]:
    issues = []
    if not is_aggregate and date_col and "ORDER BY" not in sql:
        issues.append(f"Has date column '{date_col}' but no ORDER BY")
    if not is_aggregate and len(columns_used) < 2:
        issues.append("Too few columns selected")
    if "SELECT *" in sql:
        issues.append("Using SELECT *")
    for bad in ["_at", "4NN", "2NN", "DateOn"]:
        if f'"{bad}"' in sql or f".{bad}" in sql:
            issues.append(f"Noise column still present: {bad}")
    return issues


def analyze_pipeline_result(sql_result: Dict[str, Any]) -> Dict[str, Any]:
    report = {
        "query": sql_result["raw_user_query"],
        "databases_selected": [d["database"] for d in sql_result["databases"]],
        "total_queries": 0,
        "issues_found": [],
        "status": "OK",
    }
    for db in sql_result["databases"]:
        for q in db["queries"]:
            report["total_queries"] += 1
            issues = validate_sql(q["sql"], q["columns_used"], q.get("date_column_used"), q.get("is_aggregate", False))
            for iss in issues:
                report["issues_found"].append(f"[{db['database']}.{q['table']}] {iss}")
    if report["issues_found"]:
        report["status"] = "ISSUES_DETECTED"
    return report

# --------------------------------------------------------------------------
# 8. Database connections & execution (generic across db_type, parallelized)
# --------------------------------------------------------------------------
def get_connection(cfg: DatabaseConfig):
    if cfg.db_type == "mssql":
        c = cfg.connection
        driver = c.get("driver", "ODBC Driver 18 for SQL Server")
        server = c.get("server", "localhost,1433")
        database = c.get("database", "")
        trusted = str(c.get("trusted", "yes")).lower() == "yes"
        encrypt = c.get("encrypt", "no")
        trust_cert = c.get("trust_cert", "yes")
        conn_str = (
            f"DRIVER={{{driver}}};SERVER={server};DATABASE={database};"
            f"Trusted_Connection={'yes' if trusted else 'no'};"
            f"Encrypt={encrypt};TrustServerCertificate={trust_cert};"
        )
        return pyodbc.connect(conn_str, timeout=15)

    if cfg.db_type == "postgres":
        c = cfg.connection
        url = c.get("url")
        if url and url.startswith("postgresql"):
            url = url.replace("postgresql+psycopg2://", "postgresql://")
            return psycopg2.connect(url)
        return psycopg2.connect(
            host=c.get("host", "localhost"),
            port=c.get("port", "5432"),
            dbname=c.get("dbname", "postgres"),
            user=c.get("user", "postgres"),
            password=c.get("password", ""),
        )

    raise ValueError(f"Unsupported db_type: {cfg.db_type}")


def execute_sql(db_name: str, sql: str) -> Tuple[List[str], List[Tuple], Optional[str]]:
    """Returns (columns, rows, error_message). error_message is None on success."""
    cfg = DB_REGISTRY.get(db_name)
    if cfg is None:
        return [], [], f"Unknown database: {db_name}"

    run_sql = sql
    if cfg.db_type == "mssql":
        run_sql = re.sub(r'"([^"]+)"', r'[\1]', run_sql)
        m = re.search(r"LIMIT\s+(\d+)", run_sql, re.IGNORECASE)
        if m:
            limit = m.group(1)
            run_sql = re.sub(r"LIMIT\s+\d+\s*;?", "", run_sql, flags=re.IGNORECASE)
            run_sql = re.sub(r"(SELECT\s+)", rf"\1TOP {limit} ", run_sql, count=1, flags=re.IGNORECASE)

    try:
        conn = get_connection(cfg)
        try:
            if cfg.db_type == "mssql":
                cursor = conn.cursor()
                cursor.execute(run_sql)
                columns = [col[0] for col in cursor.description] if cursor.description else []
                rows = [tuple(r) for r in cursor.fetchall()]
                cursor.close()
            else:
                cursor = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)
                cursor.execute(run_sql)
                columns = [desc[0] for desc in cursor.description] if cursor.description else []
                rows = [tuple(r) for r in cursor.fetchall()]
                cursor.close()
            return columns, rows, None
        finally:
            conn.close()
    except Exception as e:
        log.error("Query failed on %s: %s\nSQL was:\n%s", db_name, e, run_sql)
        return [], [], str(e)


def run_all_queries(sql_result: Dict[str, Any], max_workers: int = 8) -> Dict[str, Any]:
    """Executes every generated query concurrently (one thread per query;
    each thread opens its own DB connection, which is safe)."""
    execution_result: Dict[str, Any] = {
        "raw_user_query": sql_result["raw_user_query"],
        "time_filter_applied": sql_result.get("time_filter_applied"),
        "aggregation_applied": sql_result.get("aggregation_applied"),
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "databases": [],
    }

    # Flatten all (db, query) pairs so they can run in one shared pool.
    jobs = []
    for db_entry in sql_result.get("databases", []):
        for q in db_entry["queries"]:
            jobs.append((db_entry["database"], q))

    results_by_id: Dict[int, Dict[str, Any]] = {}
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        future_map = {
            pool.submit(execute_sql, db_name, q["sql"]): (id(q), db_name, q)
            for db_name, q in jobs
        }
        for fut in as_completed(future_map):
            job_id, db_name, q = future_map[fut]
            cols, rows, err = fut.result()
            results_by_id[job_id] = {
                "table": q["table"],
                "sql": q["sql"],
                "is_aggregate": q.get("is_aggregate", False),
                "columns": cols,
                "rows": rows,
                "row_count": len(rows),
                "error": err,
                "explanation": q["explanation"],
            }

    for db_entry in sql_result.get("databases", []):
        new_db = {
            "database": db_entry["database"],
            "intent": db_entry["intent"],
            "keywords": db_entry["keywords"],
            "queries": [results_by_id[id(q)] for q in db_entry["queries"]],
        }
        execution_result["databases"].append(new_db)

    return execution_result

# --------------------------------------------------------------------------
# 9. Rich metadata + data-quality scoring
# --------------------------------------------------------------------------
def _detect_type(values: list) -> str:
    non_null = [v for v in values if v is not None]
    if not non_null:
        return "unknown"
    sample = non_null[0]
    if isinstance(sample, bool):
        return "boolean"
    if isinstance(sample, int) and not isinstance(sample, bool):
        return "integer"
    if isinstance(sample, float):
        return "numeric"
    try:
        float(sample)
        return "numeric"
    except (TypeError, ValueError):
        pass
    return "string"


def _build_rich_metadata(table_name: str, columns: List[str], rows: List[Tuple], max_sample: int = 5) -> Dict[str, Any]:
    if not rows or not columns:
        return {
            "table_name": table_name, "total_records": 0, "total_columns": len(columns),
            "columns": columns, "column_details": {},
            "data_quality": {"potential_outliers": [], "zero_values": {}, "completeness_pct": 0, "score": "N/A"},
        }

    total_records = len(rows)
    col_details: Dict[str, Any] = {}
    zero_values: Dict[str, int] = {}
    potential_outliers: List[Dict[str, Any]] = []
    total_cells = total_records * len(columns)
    null_cells = 0

    for col_idx, col_name in enumerate(columns):
        col_values = [row[col_idx] for row in rows]
        non_null = [v for v in col_values if v is not None]
        null_count = total_records - len(non_null)
        null_cells += null_count
        unique_count = len(set(str(v) for v in non_null))
        dtype = _detect_type(col_values)

        detail: Dict[str, Any] = {
            "data_type": dtype, "null_count": null_count, "unique_count": unique_count,
            "sample_values": [],
        }
        samples, seen = [], set()
        for v in non_null:
            s = str(v)
            if s not in seen:
                samples.append(v)
                seen.add(s)
            if len(samples) >= max_sample:
                break
        detail["sample_values"] = samples

        if dtype in ("numeric", "integer"):
            nums = []
            for v in non_null:
                try:
                    nums.append(float(v))
                except (TypeError, ValueError):
                    continue
            if nums:
                detail["min"] = round(min(nums), 2)
                detail["max"] = round(max(nums), 2)
                detail["avg"] = round(sum(nums) / len(nums), 2)
                zero_cnt = sum(1 for n in nums if n == 0)
                if zero_cnt:
                    zero_values[col_name] = zero_cnt
                if len(nums) > 5:
                    avg = detail["avg"]
                    for n in nums:
                        if avg > 0 and n > avg * 10:
                            potential_outliers.append({
                                "column": col_name, "value": n,
                                "issue": f"Unusually high value (>{avg * 10:.1f})",
                            })
                            break
        col_details[col_name] = detail

    completeness_pct = round(100 * (1 - null_cells / total_cells), 1) if total_cells else 0
    if completeness_pct >= 95 and len(potential_outliers) == 0:
        score = "Good"
    elif completeness_pct >= 80:
        score = "Fair"
    else:
        score = "Needs review"

    return {
        "table_name": table_name, "total_records": total_records, "total_columns": len(columns),
        "columns": columns, "column_details": col_details,
        "data_quality": {
            "potential_outliers": potential_outliers,
            "zero_values": zero_values,
            "completeness_pct": completeness_pct,
            "score": score,
        },
    }


# --------------------------------------------------------------------------
# 10. LLM context preparation
# --------------------------------------------------------------------------
def prepare_llm_context(execution_result: Dict[str, Any], metadata_threshold: int = 5) -> str:
    parts = [f"USER QUERY: {execution_result['raw_user_query']}",
             f"Generated at: {execution_result['generated_at']}"]
    if execution_result.get("time_filter_applied"):
        parts.append(f"Time filter: {execution_result['time_filter_applied']}")
    if execution_result.get("aggregation_applied"):
        parts.append(f"Aggregation requested: {execution_result['aggregation_applied']}")
    parts.append("")

    for db in execution_result["databases"]:
        parts.append(f"=== DATABASE: {db['database'].upper()} ===")
        parts.append(f"Intent: {db['intent']}")
        parts.append(f"Keywords: {', '.join(db['keywords'])}")
        parts.append("")
        for q in db["queries"]:
            if q.get("error"):
                parts.append(f"--- Table: {q['table']} (QUERY FAILED: {q['error']}) ---\n")
                continue
            if q["row_count"] == 0:
                continue
            parts.append(f"--- Table: {q['table']} ({q['row_count']} rows"
                          f"{', aggregated' if q.get('is_aggregate') else ''}) ---")
            if q["row_count"] > metadata_threshold:
                meta = _build_rich_metadata(q["table"], q["columns"], q["rows"])
                parts.append(json.dumps(meta, indent=2, default=str))
            else:
                parts.append(f"Columns: {q['columns']}")
                for i, row in enumerate(q["rows"][:15]):
                    parts.append(f"  Row {i+1}: {row}")
                if q["row_count"] > 15:
                    parts.append(f"  ... ({q['row_count']-15} more rows)")
            parts.append("")
    return "\n".join(parts)


# --------------------------------------------------------------------------
# 11. LLM layer: fallback chain + retry/backoff + data-driven safety net
# --------------------------------------------------------------------------
LLM_SYSTEM_PROMPT = """You are an expert manufacturing & video-analytics data analyst.
You receive a user query and the actual data retrieved from the databases.

Your job is to produce a clean JSON array of report sections.

STRICT RULES:
1. Output ONLY valid JSON - no markdown, no explanation, no extra text.
2. Each section must have exactly these keys:
   - "title": short meaningful title (string)
   - "summary": 3-8 sentence plain-text summary of the data in this section (string)
   - "chart": true or false
   - "chart_config": null OR an object with:
        {
          "chart_type": "bar" | "hbar" | "line" | "pie" | "donut" | "scatter",
          "x_axis": "column name",
          "y_axis": "column name",
          "label": "human readable label",
          "title": "chart title",
          "aggregation": "sum" | "avg" | "count" | null
        }
   - "table_name": the source table this section is primarily about (string, best guess)
3. Use chart=true only when a visualization truly adds value.
4. Supported chart_type values are ONLY: "bar", "hbar", "line", "pie", "donut", "scatter".
5. Group related tables into logical sections when it makes sense.
6. Never invent data. Base everything strictly on the provided data.
7. If a table's query failed or returned no rows, mention that briefly rather than skipping it silently.
"""

LLM_MODELS = [m.strip() for m in os.getenv(
    "LLM_FALLBACK_MODELS",
    "gemini/gemini-3.5-flash,gemini/gemini-3.6-flash,groq/openai/gpt-oss-120b,groq/openai/gpt-oss-20b"
).split(",") if m.strip()]

MAX_RETRIES_PER_MODEL = int(os.getenv("LLM_MAX_RETRIES", "2"))
RETRY_BACKOFF_SECONDS = float(os.getenv("LLM_RETRY_BACKOFF", "1.5"))


def _clean_json_content(content: str) -> str:
    content = content.strip()
    if content.startswith("```"):
        content = re.sub(r"^```(?:json)?\s*", "", content)
        content = re.sub(r"\s*```$", "", content)
    return content


def _normalize_sections(parsed: Any) -> List[Dict[str, Any]]:
    if isinstance(parsed, list):
        sections = parsed
    elif isinstance(parsed, dict) and "sections" in parsed:
        sections = parsed["sections"]
    elif isinstance(parsed, dict):
        sections = [parsed]
    else:
        raise ValueError("Unexpected JSON structure from LLM")
    for s in sections:
        if "title" not in s or "summary" not in s:
            raise ValueError("Missing required keys in section")
        s.setdefault("chart", False)
        s.setdefault("table_name", None)
        if s.get("chart") and not s.get("chart_config"):
            s["chart"] = False
            s["chart_config"] = None
    return sections


def call_llm_with_fallback(
    user_query: str,
    data_context: str,
    execution_result: Optional[Dict[str, Any]] = None,
) -> List[Dict[str, Any]]:
    messages = [
        {"role": "system", "content": LLM_SYSTEM_PROMPT},
        {"role": "user", "content": f"USER QUERY:\n{user_query}\n\nDATA CONTEXT:\n{data_context}"},
    ]

    last_error = None
    for model in LLM_MODELS:
        for attempt in range(1, MAX_RETRIES_PER_MODEL + 1):
            try:
                log.info("Trying LLM model %s (attempt %d/%d)", model, attempt, MAX_RETRIES_PER_MODEL)
                response = completion(model=model, messages=messages, temperature=0.2, max_tokens=4096)
                content = _clean_json_content(response.choices[0].message.content)
                sections = _normalize_sections(json.loads(content))
                log.info("LLM success with %s - %d sections", model, len(sections))
                return sections
            except Exception as e:
                last_error = e
                log.warning("%s attempt %d failed: %s", model, attempt, e)
                if attempt < MAX_RETRIES_PER_MODEL:
                    time.sleep(RETRY_BACKOFF_SECONDS * attempt)
        # move to next model in the chain

    log.error("All LLM models failed (last error: %s). Falling back to data-driven sections.", last_error)
    return build_fallback_sections_from_data(user_query, execution_result)


def build_fallback_sections_from_data(user_query: str, execution_result: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
    """Safety net: if every LLM call fails, still produce a useful report by
    summarising each table directly from its metadata instead of showing a
    dead placeholder message. `execution_result` is injected by the caller
    when available (see process_query_and_generate_pdf)."""
    sections: List[Dict[str, Any]] = []
    if not execution_result:
        return [{
            "title": "Data Summary", "table_name": None,
            "summary": "AI summarisation was unavailable for this run. See the data tables and appendix below for the raw results.",
            "chart": False, "chart_config": None,
        }]

    for db in execution_result.get("databases", []):
        for q in db["queries"]:
            if q.get("error") or q["row_count"] == 0:
                continue
            meta = _build_rich_metadata(q["table"], q["columns"], q["rows"])
            dq = meta["data_quality"]
            summary = (
                f"{meta['total_records']} record(s) retrieved from {q['table']} "
                f"across {meta['total_columns']} columns. Data completeness is "
                f"{dq['completeness_pct']}% ({dq['score']}). "
            )
            if dq["potential_outliers"]:
                summary += f"{len(dq['potential_outliers'])} potential outlier(s) flagged. "
            summary += "AI narrative summarisation was unavailable for this run; figures above are computed directly from the data."

            chart_cfg = None
            use_chart = False
            if q.get("is_aggregate") and len(q["columns"]) >= 2:
                use_chart = True
                chart_cfg = {
                    "chart_type": "bar", "x_axis": q["columns"][0], "y_axis": q["columns"][1],
                    "label": q["columns"][1], "title": f"{q['table']}: {q['columns'][1]} by {q['columns'][0]}",
                    "aggregation": None,
                }
            sections.append({
                "title": f"{q['table']} overview",
                "table_name": q["table"],
                "summary": summary,
                "chart": use_chart,
                "chart_config": chart_cfg,
            })

    if not sections:
        sections.append({
            "title": "No Data Found", "table_name": None,
            "summary": "No matching data was returned for this query. Try rephrasing it or widening the time filter.",
            "chart": False, "chart_config": None,
        })
    return sections

# --------------------------------------------------------------------------
# 12. Chart drawing  (bar / hbar / line / pie / donut / scatter, themed)
# --------------------------------------------------------------------------
plt.rcParams.update({
    "axes.edgecolor": "#cfd8dc",
    "axes.grid": True,
    "grid.color": "#e5e8e8",
    "grid.linewidth": 0.6,
    "font.size": 9,
})


def _truncate_label(v: Any, max_len: int = 16) -> str:
    s = str(v)
    return s if len(s) <= max_len else s[: max_len - 1] + "…"


def draw_chart_from_config(chart_config: Dict[str, Any], columns: List[str], rows: List[Tuple]) -> Optional[io.BytesIO]:
    if not chart_config or not rows or not columns:
        return None

    chart_type = (chart_config.get("chart_type") or "").lower()
    x_axis = chart_config.get("x_axis")
    y_axis = chart_config.get("y_axis")
    title = chart_config.get("title") or chart_config.get("label") or "Chart"
    aggregation = (chart_config.get("aggregation") or "sum").lower()
    col_idx = {c: i for i, c in enumerate(columns)}
    palette = THEME["palette"]

    def get_col(name):
        return [row[col_idx[name]] for row in rows] if name in col_idx else None

    try:
        if chart_type in ("pie", "donut"):
            cat_col = x_axis or (columns[0] if columns else None)
            if not cat_col:
                return None
            cats = get_col(cat_col)
            if not cats:
                return None
            counts = Counter(str(c) for c in cats if c is not None)
            if len(counts) < 2 or len(counts) > 12:
                return None
            labels = [_truncate_label(k) for k in counts.keys()]
            sizes = list(counts.values())
            fig, ax = plt.subplots(figsize=(6.5, 4.5))
            wedge_kw = {"width": 0.4} if chart_type == "donut" else {}
            ax.pie(sizes, labels=labels, autopct="%1.1f%%", startangle=90,
                   colors=palette, wedgeprops=wedge_kw, pctdistance=0.8 if chart_type == "donut" else 0.6)
            ax.set_title(title, fontsize=11, color=THEME["primary"], weight="bold")
            buf = io.BytesIO()
            fig.tight_layout()
            fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
            buf.seek(0)
            plt.close(fig)
            return buf

        if chart_type == "scatter":
            if not x_axis or not y_axis:
                return None
            xs, ys = get_col(x_axis), get_col(y_axis)
            if not xs or not ys:
                return None
            pairs = [(float(x), float(y)) for x, y in zip(xs, ys)
                     if x is not None and y is not None
                     and _is_number(x) and _is_number(y)]
            if len(pairs) < 2:
                return None
            xv, yv = zip(*pairs)
            fig, ax = plt.subplots(figsize=(8, 4.2))
            ax.scatter(xv, yv, color=THEME["primary_light"], alpha=0.75, edgecolors="white", s=45)
            ax.set_title(title, fontsize=11, color=THEME["primary"], weight="bold")
            ax.set_xlabel(x_axis)
            ax.set_ylabel(y_axis)
            buf = io.BytesIO()
            fig.tight_layout()
            fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
            buf.seek(0)
            plt.close(fig)
            return buf

        if chart_type in ("bar", "hbar", "line"):
            if not x_axis or not y_axis:
                return None
            xs, ys = get_col(x_axis), get_col(y_axis)
            if not xs or not ys:
                return None

            agg: Dict[str, List[float]] = defaultdict(list)
            for x, y in zip(xs, ys):
                if x is not None and y is not None and _is_number(y):
                    agg[str(x)].append(float(y))
            if not agg:
                return None

            items = list(agg.items())
            if aggregation == "avg":
                values = [sum(v) / len(v) for _, v in items]
            elif aggregation == "count":
                values = [len(v) for _, v in items]
            else:
                values = [sum(v) for _, v in items]
            labels = [_truncate_label(k) for k, _ in items]

            order = sorted(range(len(values)), key=lambda i: -values[i])[:15]
            labels = [labels[i] for i in order]
            values = [values[i] for i in order]

            if chart_type == "hbar":
                fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(labels) + 1)))
                bars = ax.barh(labels[::-1], values[::-1], color=THEME["primary"])
                ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=8, color=THEME["grey"])
                ax.set_xlabel(y_axis)
            elif chart_type == "bar":
                fig, ax = plt.subplots(figsize=(8, 4.2))
                bars = ax.bar(labels, values, color=THEME["primary"])
                ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=8, color=THEME["grey"])
                ax.set_ylabel(y_axis)
                plt.setp(ax.get_xticklabels(), rotation=40, ha="right", fontsize=8)
            else:  # line
                fig, ax = plt.subplots(figsize=(8, 4.2))
                ax.plot(labels, values, marker="o", linewidth=2, color=THEME["primary_light"])
                ax.fill_between(range(len(labels)), values, alpha=0.15, color=THEME["primary_light"])
                ax.set_ylabel(y_axis)
                plt.setp(ax.get_xticklabels(), rotation=40, ha="right", fontsize=8)

            ax.set_title(title, fontsize=11, color=THEME["primary"], weight="bold")
            buf = io.BytesIO()
            fig.tight_layout()
            fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
            buf.seek(0)
            plt.close(fig)
            return buf

    except Exception as e:
        log.warning("Chart draw failed (%s): %s", chart_type, e)
        plt.close("all")
        return None

    return None


def _is_number(v: Any) -> bool:
    try:
        float(v)
        return True
    except (TypeError, ValueError):
        return False


def auto_chart_config(section: Dict[str, Any], table_lookup: Dict[str, Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    """If the LLM asked for a chart but gave a config that doesn't line up
    with any retrieved table, try to build a sane one automatically from the
    matched table's columns so the section still gets a visual."""
    q = table_lookup.get(section.get("table_name") or "")
    if not q:
        return None
    cols = q["columns"]
    if q.get("is_aggregate") and len(cols) >= 2:
        return {"chart_type": "bar", "x_axis": cols[0], "y_axis": cols[1],
                "label": cols[1], "title": section.get("title", q["table"]), "aggregation": None}
    numeric = _find_numeric_column(cols)
    categorical = _find_categorical_column(cols)
    if numeric and categorical:
        return {"chart_type": "bar", "x_axis": categorical, "y_axis": numeric,
                "label": numeric, "title": section.get("title", q["table"]), "aggregation": "avg"}
    return None

# --------------------------------------------------------------------------
# 13. PDF building blocks
# --------------------------------------------------------------------------
_STYLES = getSampleStyleSheet()
_STYLES.add(ParagraphStyle(name="CoverTitle", parent=_STYLES["Title"], fontSize=24,
                            textColor=colors.HexColor(THEME["primary"]), alignment=TA_CENTER, spaceAfter=10))
_STYLES.add(ParagraphStyle(name="CoverSub", parent=_STYLES["Normal"], fontSize=12,
                            textColor=colors.HexColor(THEME["grey"]), alignment=TA_CENTER, spaceAfter=4))
_STYLES.add(ParagraphStyle(name="CoverMeta", parent=_STYLES["Normal"], fontSize=9,
                            textColor=colors.HexColor(THEME["grey"]), alignment=TA_CENTER))
_STYLES.add(ParagraphStyle(name="TOCTitle", parent=_STYLES["Heading1"], fontSize=15,
                            textColor=colors.HexColor(THEME["primary"]), spaceAfter=10))
_STYLES.add(ParagraphStyle(name="SectionTitle", parent=_STYLES["Heading2"], fontSize=13,
                            textColor=colors.HexColor(THEME["primary"]), spaceBefore=12, spaceAfter=5))
_STYLES.add(ParagraphStyle(name="SubHeading", parent=_STYLES["Heading3"], fontSize=10,
                            textColor=colors.HexColor(THEME["grey"]), spaceBefore=6, spaceAfter=3))
_STYLES.add(ParagraphStyle(name="ReportBody", parent=_STYLES["Normal"], fontSize=9.5, leading=13, spaceAfter=4))
_STYLES.add(ParagraphStyle(name="Meta", parent=_STYLES["Normal"], fontSize=7.5, textColor=colors.grey))
_STYLES.add(ParagraphStyle(name="KPILabel", parent=_STYLES["Normal"], fontSize=8,
                            textColor=colors.white, alignment=TA_CENTER))
_STYLES.add(ParagraphStyle(name="KPIValue", parent=_STYLES["Normal"], fontSize=16,
                            textColor=colors.white, alignment=TA_CENTER, leading=18))
_STYLES.add(ParagraphStyle(name="TableCell", parent=_STYLES["Normal"], fontSize=7.5, leading=9))
_STYLES.add(ParagraphStyle(name="TableHeader", parent=_STYLES["Normal"], fontSize=7.5,
                            textColor=colors.white, leading=9))


class NumberedCanvas(pdf_canvas.Canvas):
    """Draws 'Page X of Y' footers by buffering pages, then stamping the
    total page count once the document is complete."""

    def __init__(self, *args, **kwargs):
        pdf_canvas.Canvas.__init__(self, *args, **kwargs)
        self._saved_page_states = []

    def showPage(self):
        self._saved_page_states.append(dict(self.__dict__))
        self._startPage()

    def save(self):
        total_pages = len(self._saved_page_states)
        for state in self._saved_page_states:
            self.__dict__.update(state)
            self._draw_footer(total_pages)
            pdf_canvas.Canvas.showPage(self)
        pdf_canvas.Canvas.save(self)

    def _draw_footer(self, total_pages: int):
        page_num = self._pageNumber
        if page_num == 1:
            return  # no footer on the cover page
        self.setFont("Helvetica", 7.5)
        self.setFillColor(colors.HexColor(THEME["grey"]))
        w, h = landscape(A4)
        self.drawString(12 * mm, 8 * mm, "Manufacturing & Video-Analytics Intelligent Report")
        self.drawRightString(w - 12 * mm, 8 * mm, f"Page {page_num} of {total_pages}")
        self.setStrokeColor(colors.HexColor("#d5d8dc"))
        self.line(12 * mm, 11 * mm, w - 12 * mm, 11 * mm)


class ReportDocTemplate(BaseDocTemplate):
    """Adds real TOC entries + PDF outline bookmarks whenever a SectionTitle
    or CoverTitle paragraph is rendered."""

    def afterFlowable(self, flowable):
        if isinstance(flowable, Paragraph):
            style_name = flowable.style.name
            if style_name == "SectionTitle":
                text = flowable.getPlainText()
                key = f"sec-{abs(hash(text))}-{self.page}"
                self.canv.bookmarkPage(key)
                self.canv.addOutlineEntry(text, key, level=1, closed=False)
                self.notify("TOCEntry", (1, text, self.page, key))
            elif style_name == "TOCTitle":
                text = flowable.getPlainText()
                key = f"toc-{self.page}"
                self.canv.bookmarkPage(key)
                self.canv.addOutlineEntry(text, key, level=0, closed=False)


def _make_frame_template(page_size):
    frame = Frame(12 * mm, 15 * mm, page_size[0] - 24 * mm, page_size[1] - 27 * mm, id="normal")
    return PageTemplate(id="normal", frames=[frame])


def kpi_card(label: str, value: str, color_hex: str) -> Table:
    t = Table([[Paragraph(value, _STYLES["KPIValue"])],
               [Paragraph(label, _STYLES["KPILabel"])]], colWidths=[42 * mm])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor(color_hex)),
        ("TOPPADDING", (0, 0), (-1, 0), 10), ("BOTTOMPADDING", (0, 0), (-1, 0), 2),
        ("TOPPADDING", (0, 1), (-1, 1), 0), ("BOTTOMPADDING", (0, 1), (-1, 1), 10),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("ROUNDEDCORNERS", [6, 6, 6, 6]),
    ]))
    return t


def build_kpi_row(execution_result: Dict[str, Any]) -> Table:
    total_tables = sum(len(db["queries"]) for db in execution_result["databases"])
    total_rows = sum(q["row_count"] for db in execution_result["databases"] for q in db["queries"])
    failed = sum(1 for db in execution_result["databases"] for q in db["queries"] if q.get("error"))
    cards = [
        kpi_card("DATABASES QUERIED", str(len(execution_result["databases"])), THEME["primary"]),
        kpi_card("TABLES MATCHED", str(total_tables), THEME["primary_light"]),
        kpi_card("TOTAL RECORDS", f"{total_rows:,}", THEME["accent"]),
        kpi_card("TIME FILTER", "Applied" if execution_result.get("time_filter_applied") else "None",
                  THEME["success"] if execution_result.get("time_filter_applied") else THEME["grey"]),
        kpi_card("FAILED QUERIES", str(failed), THEME["danger"] if failed else THEME["success"]),
    ]
    row = Table([cards], colWidths=[46 * mm] * len(cards))
    row.setStyle(TableStyle([("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                              ("LEFTPADDING", (0, 0), (-1, -1), 3), ("RIGHTPADDING", (0, 0), (-1, -1), 3)]))
    return row


def make_data_preview_table(columns: List[str], rows: List[Tuple], max_rows: int = 10, max_cols: int = 8) -> Table:
    cols = columns[:max_cols]
    col_idx = [columns.index(c) for c in cols]
    header = [Paragraph(f"<b>{c}</b>", _STYLES["TableHeader"]) for c in cols]
    data = [header]
    for row in rows[:max_rows]:
        cells = []
        for i in col_idx:
            val = row[i] if i < len(row) else ""
            cells.append(Paragraph(_truncate_label(val, 28), _STYLES["TableCell"]))
        data.append(cells)

    n_cols = len(cols)
    avail_width = landscape(A4)[0] - 24 * mm
    col_width = avail_width / n_cols
    t = Table(data, colWidths=[col_width] * n_cols, repeatRows=1)
    style = [
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(THEME["primary"])),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#d5d8dc")),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("TOPPADDING", (0, 0), (-1, -1), 3), ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
    ]
    for r in range(1, len(data)):
        if r % 2 == 0:
            style.append(("BACKGROUND", (0, r), (-1, r), colors.HexColor(THEME["bg_alt_row"])))
    t.setStyle(TableStyle(style))
    return t


def make_quality_table(metas: List[Dict[str, Any]]) -> Table:
    header = ["Table", "Records", "Completeness", "Outliers", "Score"]
    data = [[Paragraph(f"<b>{h}</b>", _STYLES["TableHeader"]) for h in header]]
    score_color = {"Good": THEME["success"], "Fair": THEME["warning"], "Needs review": THEME["danger"], "N/A": THEME["grey"]}
    for m in metas:
        dq = m["data_quality"]
        data.append([
            Paragraph(m["table_name"], _STYLES["TableCell"]),
            Paragraph(str(m["total_records"]), _STYLES["TableCell"]),
            Paragraph(f"{dq['completeness_pct']}%", _STYLES["TableCell"]),
            Paragraph(str(len(dq["potential_outliers"])), _STYLES["TableCell"]),
            Paragraph(f'<font color="{score_color.get(dq["score"], THEME["grey"])}"><b>{dq["score"]}</b></font>', _STYLES["TableCell"]),
        ])
    avail_width = landscape(A4)[0] - 24 * mm
    widths = [avail_width * w for w in (0.32, 0.15, 0.2, 0.15, 0.18)]
    t = Table(data, colWidths=widths, repeatRows=1)
    style = [
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(THEME["primary"])),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#d5d8dc")),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("TOPPADDING", (0, 0), (-1, -1), 4), ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
    ]
    for r in range(1, len(data)):
        if r % 2 == 0:
            style.append(("BACKGROUND", (0, r), (-1, r), colors.HexColor(THEME["bg_alt_row"])))
    t.setStyle(TableStyle(style))
    return t

# --------------------------------------------------------------------------
# 14. Full PDF generation
# --------------------------------------------------------------------------
def generate_pdf_from_llm_sections(
    execution_result: Dict[str, Any],
    llm_sections: List[Dict[str, Any]],
    output_path: str = None,
    metadata_threshold: int = 5,
) -> str:
    if output_path is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"query_report_{ts}.pdf"
    output_path = str(Path(output_path).resolve())

    page_size = landscape(A4)
    doc = ReportDocTemplate(
        output_path, pagesize=page_size,
        leftMargin=12 * mm, rightMargin=12 * mm, topMargin=12 * mm, bottomMargin=15 * mm,
    )
    doc.addPageTemplates([_make_frame_template(page_size)])

    table_lookup: Dict[str, Dict[str, Any]] = {}
    all_metas: List[Dict[str, Any]] = []
    for db in execution_result["databases"]:
        for q in db["queries"]:
            if q["row_count"] > 0:
                table_lookup[q["table"]] = q
                all_metas.append(_build_rich_metadata(q["table"], q["columns"], q["rows"]))

    story: List[Any] = []

    # ---- Cover page --------------------------------------------------
    story.append(Spacer(1, 45 * mm))
    story.append(Paragraph("Manufacturing & Video-Analytics", _STYLES["CoverTitle"]))
    story.append(Paragraph("Intelligent Report", _STYLES["CoverTitle"]))
    story.append(Spacer(1, 8 * mm))
    story.append(HRFlowable(width="40%", thickness=1.4, color=colors.HexColor(THEME["accent"]), hAlign="CENTER"))
    story.append(Spacer(1, 8 * mm))
    story.append(Paragraph(f"&ldquo;{execution_result['raw_user_query']}&rdquo;", _STYLES["CoverSub"]))
    story.append(Spacer(1, 14 * mm))
    story.append(Paragraph(f"Generated: {execution_result['generated_at']}", _STYLES["CoverMeta"]))
    if execution_result.get("time_filter_applied"):
        story.append(Paragraph("Time filter applied", _STYLES["CoverMeta"]))
    story.append(PageBreak())

    # ---- Table of contents --------------------------------------------
    story.append(Paragraph("Contents", _STYLES["TOCTitle"]))
    toc = TableOfContents()
    toc.levelStyles = [
        ParagraphStyle(name="TOCLevel1", fontSize=10.5, leading=16,
                        textColor=colors.HexColor(THEME["primary"])),
    ]
    story.append(toc)
    story.append(PageBreak())

    # ---- KPI summary ----------------------------------------------------
    story.append(Paragraph("Executive Summary", _STYLES["SectionTitle"]))
    story.append(build_kpi_row(execution_result))
    story.append(Spacer(1, 4 * mm))
    kw_all = sorted({kw for db in execution_result["databases"] for kw in db.get("keywords", [])})
    if kw_all:
        story.append(Paragraph(f"<b>Matched keywords:</b> {', '.join(kw_all)}", _STYLES["ReportBody"]))
    story.append(Spacer(1, 6 * mm))

    # ---- LLM sections -----------------------------------------------
    for section in llm_sections:
        block: List[Any] = []
        title = section.get("title", "Section")
        summary = section.get("summary", "")
        need_chart = section.get("chart", False)
        chart_cfg = section.get("chart_config")
        table_name = section.get("table_name")

        block.append(Paragraph(title, _STYLES["SectionTitle"]))
        block.append(Paragraph(summary.replace("\n", "<br/>"), _STYLES["ReportBody"]))
        block.append(Spacer(1, 2 * mm))

        matched_q = table_lookup.get(table_name) if table_name else None
        if matched_q is None and chart_cfg:
            for q in table_lookup.values():
                if chart_cfg.get("x_axis") in q["columns"] or chart_cfg.get("y_axis") in q["columns"]:
                    matched_q = q
                    break

        if need_chart:
            cfg = chart_cfg
            if matched_q and (not cfg or (cfg.get("x_axis") not in matched_q["columns"]
                                           and cfg.get("y_axis") not in matched_q["columns"])):
                cfg = auto_chart_config(section, table_lookup) or cfg
            if matched_q and cfg:
                buf = draw_chart_from_config(cfg, matched_q["columns"], matched_q["rows"])
                if buf:
                    block.append(Image(buf, width=150 * mm, height=78 * mm))
                    block.append(Spacer(1, 3 * mm))

        if matched_q and matched_q["row_count"] > 0:
            block.append(Paragraph(
                f"Data preview &mdash; {matched_q['table']} "
                f"(showing {min(10, matched_q['row_count'])} of {matched_q['row_count']} rows)",
                _STYLES["SubHeading"]))
            block.append(make_data_preview_table(matched_q["columns"], matched_q["rows"]))
        elif matched_q and matched_q.get("error"):
            block.append(Paragraph(f"<font color='{THEME['danger']}'>Query failed: {matched_q['error']}</font>",
                                    _STYLES["Meta"]))

        block.append(Spacer(1, 6 * mm))
        story.append(KeepTogether(block))

    # ---- Appendix: data quality -----------------------------------------
    story.append(PageBreak())
    story.append(Paragraph("Appendix &mdash; Data Quality", _STYLES["SectionTitle"]))
    if all_metas:
        story.append(make_quality_table(all_metas))
    story.append(Spacer(1, 6 * mm))

    failed_queries = [(db["database"], q) for db in execution_result["databases"]
                       for q in db["queries"] if q.get("error")]
    if failed_queries:
        story.append(Paragraph("Failed Queries", _STYLES["SubHeading"]))
        for db_name, q in failed_queries:
            story.append(Paragraph(f"[{db_name}.{q['table']}] {q['error']}", _STYLES["Meta"]))

    doc.multiBuild(story, canvasmaker=NumberedCanvas)
    log.info("PDF written to: %s", output_path)
    return output_path


# --------------------------------------------------------------------------
# 15. High-level pipeline
# --------------------------------------------------------------------------
def process_query_and_generate_pdf(user_query: str, pdf_path: str = None, limit_per_table: int = 50) -> str:
    t0 = time.time()
    log.info("Processing: %s", user_query)

    problems = validate_environment()
    for p in problems:
        log.warning("Environment check: %s", p)

    split = customize_and_split_query(user_query)
    if not split["selected_databases"]:
        log.warning("No databases matched the query - check vocabulary / phrasing.")

    sql_result = generate_sql_queries(split, limit_per_table=limit_per_table)
    report = analyze_pipeline_result(sql_result)
    log.info("DBs selected: %s | Queries generated: %d | Status: %s",
              report["databases_selected"], report["total_queries"], report["status"])
    for issue in report["issues_found"]:
        log.debug("SQL validator: %s", issue)

    execution = run_all_queries(sql_result)
    total_rows = sum(q["row_count"] for db in execution["databases"] for q in db["queries"])
    log.info("Execution complete: %d row(s) across %d table(s)", total_rows,
              sum(len(db["queries"]) for db in execution["databases"]))

    data_context = prepare_llm_context(execution)
    log.info("LLM context length: %d chars", len(data_context))

    llm_sections = call_llm_with_fallback(user_query, data_context, execution_result=execution)
    log.info("Received %d report section(s)", len(llm_sections))

    pdf_file = generate_pdf_from_llm_sections(execution, llm_sections, output_path=pdf_path)
    log.info("Done in %.1fs -> %s", time.time() - t0, pdf_file)
    return pdf_file


# --------------------------------------------------------------------------
# 16. CLI
# --------------------------------------------------------------------------
def _build_arg_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="Generate an intelligent PDF report from MES + video-analytics data.")
    p.add_argument("query", nargs="?", help="Natural-language query. If omitted, you'll be prompted.")
    p.add_argument("-o", "--output", default=None, help="Output PDF path.")
    p.add_argument("--limit", type=int, default=50, help="Row limit per table (default 50).")
    p.add_argument("--check-env", action="store_true", help="Only validate environment/config and exit.")
    return p


def main(argv=None) -> int:
    args = _build_arg_parser().parse_args(argv)

    if args.check_env:
        problems = validate_environment()

        if problems:
            print("Environment problems found:")

            for p in problems:
                print(f"  - {p}")

            return 1

        print("Environment OK.")
        return 0

    query = args.query or input("Enter your report query: ").strip()

    if not query:
        log.error("No query provided.")
        return 1

    try:
        pdf_file = process_query_and_generate_pdf(
            query,
            pdf_path=args.output,
            limit_per_table=args.limit,
        )

        print(f"\nDone. Report saved to: {pdf_file}")

        return 0

    except Exception as e:
        log.exception("Report generation failed: %s", e)
        return 1
        
if __name__ == "__main__":
    # Avoid parsing Jupyter kernel arguments
    main()

09:59:47 [INFO] report_agent: Loaded video_analytics: 52 tables
09:59:47 [INFO] report_agent: Loaded mes: 108 tables
usage: ipykernel_launcher.py [-h] [-o OUTPUT] [--limit LIMIT] [--check-env]
                             [query]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\soroj\AppData\Roaming\jupyter\runtime\kernel-v376a28f68b604fe37fae39ae00739d643b1ad4e23.json


SystemExit: 2

In [54]:
pdf_path = process_query_and_generate_pdf(
    user_query="Show machine production and safety violations",
    pdf_path="report.pdf",
    limit_per_table=50,
)

print(f"Report generated: {pdf_path}")

09:59:53 [INFO] report_agent: Processing: Show machine production and safety violations
09:59:53 [INFO] report_agent: DBs selected: ['video_analytics', 'mes'] | Queries generated: 14 | Status: OK
09:59:55 [INFO] report_agent: Execution complete: 240 row(s) across 14 table(s)
09:59:55 [INFO] report_agent: LLM context length: 20770 chars
09:59:55 [INFO] report_agent: Trying LLM model gemini/gemini-3.5-flash (attempt 1/2)
09:59:55 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= gemini-3.5-flash; provider = gemini
09:59:55 [INFO] LiteLLM: 
LiteLLM completion() model= gemini-3.5-flash; provider = gemini
09:59:55 - LiteLLM:INFO: vertex_and_google_ai_studio_gemini.py:1089 - Warning: Setting temperature < 1.0 for Gemini 3 models (gemini-3.5-flash) can cause infinite loops, degraded reasoning performance, and failure on complex tasks. Strongly recommended to use temperature = 1.0 (default).
09:59:55 [INFO] LiteLLM: Warning: Setting temperature < 1.0 for Gemini 3 models (gemini-3.5-


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



09:59:58 [INFO] report_agent: Trying LLM model gemini/gemini-3.5-flash (attempt 2/2)
09:59:58 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= gemini-3.5-flash; provider = gemini
09:59:58 [INFO] LiteLLM: 
LiteLLM completion() model= gemini-3.5-flash; provider = gemini
09:59:58 - LiteLLM:INFO: vertex_and_google_ai_studio_gemini.py:1089 - Warning: Setting temperature < 1.0 for Gemini 3 models (gemini-3.5-flash) can cause infinite loops, degraded reasoning performance, and failure on complex tasks. Strongly recommended to use temperature = 1.0 (default).
09:59:58 [INFO] LiteLLM: Warning: Setting temperature < 1.0 for Gemini 3 models (gemini-3.5-flash) can cause infinite loops, degraded reasoning performance, and failure on complex tasks. Strongly recommended to use temperature = 1.0 (default).
09:59:58 - LiteLLM:WARNING: vertex_and_google_ai_studio_gemini.py:1095 - DeprecationWarning: `temperature`, `top_p`, and `top_k` continue to function for Gemini 3+ (gemini-3.5-flash) but

Report generated: C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\jupyter notebooks\report.pdf
